In [ ]:

# ===================================================================
# NETGUARD — RESEARCH OUTPUTS PIPELINE
# CELL 1: RESEARCH SOURCE DISCOVERY AND MANIFEST
#
# Existing Google Drive structure:
#
# network-attack-forecasting/
# └── Research_Outputs/
#     ├── 01_Figures/
#     │   ├── Forecasting/
#     │   ├── Current_Detection/
#     │   └── Model_Comparisons/
#     ├── 02_Tables/
#     ├── 03_Metrics/
#     ├── 04_Reports/
#     └── 05_Manifests/
#
# SAFETY:
# - No model training
# - No official TEST data access
# - No existing artifacts moved, modified or overwritten
# - Only NEW manifest files written to 05_Manifests/
# ===================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone

import os
import csv
import json
import zipfile
from collections import Counter

# -------------------------------------------------------------------
# 1. MOUNT DRIVE
# -------------------------------------------------------------------

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUT = PROJECT / "Research_Outputs"

FIGURES = OUTPUT / "01_Figures"
TABLES = OUTPUT / "02_Tables"
METRICS = OUTPUT / "03_Metrics"
REPORTS = OUTPUT / "04_Reports"
MANIFESTS = OUTPUT / "05_Manifests"

print("=" * 90)
print("NETGUARD — RESEARCH OUTPUTS SOURCE AUDIT")
print("=" * 90)

# -------------------------------------------------------------------
# 2. VERIFY THE EXACT EXISTING OUTPUT STRUCTURE
# -------------------------------------------------------------------

EXPECTED_DIRS = {
    "Project": PROJECT,
    "Research Outputs": OUTPUT,
    "Figures": FIGURES,
    "Forecasting Figures": FIGURES / "Forecasting",
    "Current Detection Figures": FIGURES / "Current_Detection",
    "Model Comparison Figures": FIGURES / "Model_Comparisons",
    "Tables": TABLES,
    "Metrics": METRICS,
    "Reports": REPORTS,
    "Manifests": MANIFESTS
}

print("\nA. OUTPUT DIRECTORY VERIFICATION")
print("-" * 90)

for name, path in EXPECTED_DIRS.items():

    exists = path.is_dir()

    print(
        f"{'PASS' if exists else 'MISSING':<8}"
        f"{name:<32}"
        f"{path}"
    )

    if not exists:
        raise RuntimeError(
            f"Required directory missing: {path}\n"
            "The audit has stopped without creating folders."
        )

# -------------------------------------------------------------------
# 3. IDENTIFY OUR SOURCE LOCATIONS
# -------------------------------------------------------------------

SOURCE_AREAS = {
    "Forecasting":
        PROJECT / "Synthetic_NAF_v2",

    "Graph_Forecasting":
        PROJECT / "Synthetic_NAF_v2_GraphCompanion",

    "Current_Detection":
        PROJECT / "Current_Attack_Detection",

    "Forecast_Outputs":
        PROJECT / "ForecastPipeline_Outputs",

    "Dataset_Audit_Reports":
        PROJECT / "Dataset_Audit_v2" / "audit_outputs"
}

print("\nB. SOURCE DIRECTORY VERIFICATION")
print("-" * 90)

for name, path in SOURCE_AREAS.items():

    print(
        f"{name:<28}"
        f"{'FOUND' if path.is_dir() else 'MISSING'}"
    )

# -------------------------------------------------------------------
# 4. FILE SELECTION RULES
#
# Research outputs ONLY.
# Do not inspect raw traffic, official TEST data
# or saved model weights.
# -------------------------------------------------------------------

RELEVANT_EXTENSIONS = {
    ".csv",
    ".json",
    ".parquet",
    ".npz",
    ".txt",
    ".md"
}

RELEVANT_TERMS = (
    "validation",
    "metric",
    "comparison",
    "calibration",
    "bootstrap",
    "confidence",
    "confusion",
    "threshold",
    "alert",
    "prediction",
    "precision",
    "recall",
    "f1",
    "ap_",
    "horizon",
    "eta",
    "family",
    "per_class",
    "source_specific",
    "paired",
    "report",
    "summary",
    "manifest",
    "release",
    "audit",
    "artifact_hashes",
    "model_schema"
)

CHECKPOINT_EXTENSIONS = {
    ".ubj", ".joblib", ".pkl", ".pickle",
    ".pt", ".pth", ".h5", ".keras",
    ".ckpt", ".cbm", ".onnx",
    ".safetensors"
}

def is_excluded(path):

    relative = path.relative_to(PROJECT)
    parts = [part.lower() for part in relative.parts]

    # Do not examine raw source datasets.
    if "raw" in parts:
        return True

    # Do not examine test-specific files or directories.
    if any("test" in part for part in parts):
        return True

    # Do not examine training-data directories.
    if any(part in {
        "train",
        "training",
        "train_only_v1",
        "generated"
    } for part in parts):
        return True

    return False

def file_category(path):

    suffix = path.suffix.lower()

    if suffix in CHECKPOINT_EXTENSIONS:
        return "MODEL_CHECKPOINT"

    if suffix == ".csv":
        return "CSV_RESULTS"

    if suffix == ".json":
        return "JSON_RESULTS"

    if suffix == ".parquet":
        return "PARQUET_RESULTS"

    if suffix == ".npz":
        return "NPZ_RESULTS"

    if suffix in {".txt", ".md"}:
        return "DOCUMENTATION"

    return "OTHER"

def research_use(path):

    name = path.name.lower()

    if "prediction" in name or "prob" in name:
        return "TIME_SERIES_OR_PR_CURVES"

    if "confusion" in name or "per_class" in name:
        return "CONFUSION_OR_CLASS_METRICS"

    if "calibration" in name:
        return "CALIBRATION_FIGURES"

    if "bootstrap" in name or "confidence" in name:
        return "UNCERTAINTY_AND_CI"

    if "threshold" in name or "alert" in name:
        return "THRESHOLD_AND_ALERT_ANALYSIS"

    if "eta" in name:
        return "ETA_EVALUATION"

    if "comparison" in name or "metric" in name:
        return "MODEL_COMPARISON_TABLES"

    if "report" in name or "summary" in name:
        return "RESEARCH_REPORT"

    return "PROVENANCE_OR_SUPPORT"

# -------------------------------------------------------------------
# 5. METADATA-ONLY CONTENT INSPECTION
#
# CSV: header only
# JSON: top-level keys only, if <= 1 MB
# NPZ: archive member names only
# Parquet: schema only if pyarrow exists
#
# No complete prediction arrays are loaded.
# -------------------------------------------------------------------

def inspect_fields(path):

    suffix = path.suffix.lower()

    try:

        if suffix == ".csv":

            for encoding in ("utf-8-sig", "cp1252"):

                try:
                    with path.open(
                        "r",
                        encoding=encoding,
                        newline=""
                    ) as handle:

                        reader = csv.reader(handle)
                        return next(reader, [])

                except UnicodeDecodeError:
                    continue

            return ["HEADER_ENCODING_ERROR"]

        if suffix == ".json":

            if path.stat().st_size > 1_000_000:
                return ["JSON_TOO_LARGE_FOR_KEY_INSPECTION"]

            with path.open(
                "r",
                encoding="utf-8"
            ) as handle:

                data = json.load(handle)

            if isinstance(data, dict):
                return list(data.keys())[:60]

            if isinstance(data, list):
                return [f"JSON_LIST_{len(data)}_ITEMS"]

            return [type(data).__name__]

        if suffix == ".npz":

            # Inspect archive names; never deserialize arrays.
            with zipfile.ZipFile(path) as archive:
                return archive.namelist()[:60]

        if suffix == ".parquet":

            try:
                import pyarrow.parquet as pq

                schema = pq.read_schema(str(path))
                return schema.names[:100]

            except ImportError:
                return ["PYARROW_NOT_INSTALLED"]

        if suffix in {".md", ".txt"}:
            return ["DOCUMENTATION_NOT_OPENED"]

        return []

    except Exception as error:

        return [
            "INSPECTION_ERROR",
            type(error).__name__
        ]

# -------------------------------------------------------------------
# 6. DISCOVER SAVED RESEARCH RESULTS
# -------------------------------------------------------------------

records = []
scan_errors = []
checkpoint_count = 0

print("\nC. DISCOVERING RESEARCH RESULT FILES")
print("-" * 90)

for area, root in SOURCE_AREAS.items():

    if not root.is_dir():
        continue

    discovered = 0

    def record_walk_error(error):
        scan_errors.append(str(error))

    for current_dir, directories, filenames in os.walk(
        root,
        onerror=record_walk_error,
        followlinks=False
    ):

        current = Path(current_dir)

        # Do not descend into excluded directories.
        directories[:] = sorted(
            [
                d for d in directories
                if not is_excluded(current / d)
            ]
        )

        for filename in sorted(filenames):

            path = current / filename

            if is_excluded(path):
                continue

            if path.suffix.lower() in CHECKPOINT_EXTENSIONS:
                checkpoint_count += 1
                continue

            if path.suffix.lower() not in RELEVANT_EXTENSIONS:
                continue

            if not any(
                term in filename.lower()
                for term in RELEVANT_TERMS
            ):
                continue

            try:

                stat = path.stat()

                record = {
                    "source_area": area,
                    "relative_path": str(
                        path.relative_to(PROJECT)
                    ),
                    "filename": filename,
                    "category": file_category(path),
                    "suggested_use": research_use(path),
                    "size_bytes": int(stat.st_size),
                    "size_mb": round(
                        stat.st_size / 1048576, 4
                    ),
                    "fields": inspect_fields(path)
                }

                records.append(record)
                discovered += 1

            except Exception as error:

                scan_errors.append(
                    f"{path}: {type(error).__name__}: {error}"
                )

    print(
        f"{area:<30}"
        f"{discovered:>5} candidate result files"
    )

# -------------------------------------------------------------------
# 7. PRIORITY RESEARCH SOURCES
# -------------------------------------------------------------------

PRIORITY_NAMES = (
    "FOUR_MODEL_VALIDATION_COMPARISON.csv",
    "SOURCE_SPECIFIC_METRICS.csv",
    "SOURCE_FAMILY_PER_CLASS.csv",
    "COMPETING_RISK_RECOMPUTED_METRICS.csv",
    "FIXED_THRESHOLD_ALERT_AUDIT.csv",
    "EXPLOIT_APP_PAIRED_SOURCE_CI.csv",
    "CALIBRATION_10_BINS.csv",
    "FAMILY_CONFUSION_MATRIX.csv",
    "VALIDATION_PREDICTIONS.csv",
    "VALIDATION_PREDICTIONS.parquet",
    "GAT_VAL_PREDICTIONS.npz",
    "GCN_VAL_PREDICTIONS.npz",
    "validation_metrics.json",
    "evaluation.json"
)

print("\nD. HIGH-PRIORITY RESEARCH SOURCES")
print("-" * 90)

for priority_name in PRIORITY_NAMES:

    matches = [
        r for r in records
        if r["filename"].lower()
        == priority_name.lower()
    ]

    if not matches:
        print(f"NOT FOUND | {priority_name}")
        continue

    print(
        f"\nFOUND ({len(matches)}) | {priority_name}"
    )

    for match in matches[:5]:
        print(
            "  ",
            match["relative_path"]
        )

# -------------------------------------------------------------------
# 8. EXPORT SOURCE MANIFESTS
#
# Write ONLY to the existing 05_Manifests folder.
# Unique names prevent overwriting.
# -------------------------------------------------------------------

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

CSV_PATH = (
    MANIFESTS /
    f"research_source_inventory_{STAMP}.csv"
)

JSON_PATH = (
    MANIFESTS /
    f"research_source_manifest_{STAMP}.json"
)

SUMMARY_PATH = (
    MANIFESTS /
    f"research_source_summary_{STAMP}.txt"
)

csv_fields = [
    "source_area",
    "relative_path",
    "filename",
    "category",
    "suggested_use",
    "size_bytes",
    "size_mb",
    "fields"
]

with CSV_PATH.open(
    "x",
    newline="",
    encoding="utf-8"
) as handle:

    writer = csv.DictWriter(
        handle,
        fieldnames=csv_fields
    )

    writer.writeheader()

    for record in records:

        writer.writerow({
            **record,
            "fields": json.dumps(
                record["fields"],
                ensure_ascii=False
            )
        })

manifest = {
    "project": "NetGuard",
    "purpose": "Research-output source discovery",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_root": str(PROJECT),
    "output_root": str(OUTPUT),
    "candidate_count": len(records),
    "skipped_model_checkpoint_count": checkpoint_count,
    "scan_errors": scan_errors,
    "official_test_accessed": False,
    "models_trained": 0,
    "records": records
}

with JSON_PATH.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False
    )

area_counts = Counter(
    record["source_area"]
    for record in records
)

use_counts = Counter(
    record["suggested_use"]
    for record in records
)

summary_lines = [
    "NETGUARD — RESEARCH SOURCE SUMMARY",
    "=" * 80,
    f"Candidates found: {len(records)}",
    f"Checkpoint files skipped: {checkpoint_count}",
    f"Scan errors: {len(scan_errors)}",
    "",
    "SOURCE AREAS",
    "-" * 80
]

for area, count in sorted(area_counts.items()):
    summary_lines.append(f"{area}: {count}")

summary_lines.extend([
    "",
    "RESEARCH OUTPUT CATEGORIES",
    "-" * 80
])

for category, count in use_counts.most_common():
    summary_lines.append(
        f"{category}: {count}"
    )

summary_lines.extend([
    "",
    "NOTE",
    "-" * 80,
    "Candidate files are discovered, not yet scientifically",
    "validated or approved for cross-model comparisons.",
    "Prediction splits, targets, class definitions and",
    "units must be checked before plotting.",
    "",
    "No official TEST files were opened.",
    "No checkpoints were opened or modified.",
    "All existing research outputs were preserved."
])

with SUMMARY_PATH.open(
    "x",
    encoding="utf-8"
) as handle:

    handle.write(
        "\n".join(summary_lines)
    )

# -------------------------------------------------------------------
# 9. FINAL SUMMARY
# -------------------------------------------------------------------

print("\n" + "=" * 90)
print("E. DISCOVERY SUMMARY")
print("=" * 90)

print("Research result candidates:", len(records))
print("Model checkpoints skipped:", checkpoint_count)
print("Scan errors:", len(scan_errors))

print("\nSuggested output groups:")

for category, count in use_counts.most_common():
    print(f"  {category:<36}: {count}")

print("\nManifest files created:")

print("  ", CSV_PATH.name)
print("  ", JSON_PATH.name)
print("  ", SUMMARY_PATH.name)

print("\nSaved to:")
print(MANIFESTS)

print("\n" + "=" * 90)
print("CELL 1 COMPLETE")
print("=" * 90)

print("Existing folders reused: YES")
print("Model training performed: NO")
print("Official TEST accessed: NO")
print("Existing model files changed: NO")
print("Existing results overwritten: NO")
print("New source manifest created: YES")

print("=" * 90)


In [ ]:

# ================================================================
# NETGUARD RESEARCH OUTPUTS
# CELL 2 — FORECASTING RESULTS VALIDATION
#
# Uses existing Research_Outputs structure.
# Reads only saved validation reports.
#
# No training.
# No official TEST data access.
# No source files modified.
# Only a NEW audit report is created.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 88)
print("CELL 2 — FORECASTING RESULT VALIDATION")
print("=" * 88)

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUT = PROJECT / "Research_Outputs"
MANIFESTS = OUTPUT / "05_Manifests"

AUDIT = (
    PROJECT
    / "Synthetic_NAF_v2"
    / "models"
    / "FINAL_VALIDATION_AUDIT"
)

if not AUDIT.is_dir():
    raise FileNotFoundError(
        f"Forecasting audit directory missing: {AUDIT}"
    )

if not MANIFESTS.is_dir():
    raise FileNotFoundError(
        f"Manifest directory missing: {MANIFESTS}"
    )

# ------------------------------------------------------------
# 1. Explicitly choose official validation REPORTS.
#    Do not traverse raw, training, or TEST directories.
# ------------------------------------------------------------

SOURCES = {
    "model_comparison":
        "FOUR_MODEL_VALIDATION_COMPARISON.csv",

    "competing_risk":
        "COMPETING_RISK_RECOMPUTED_METRICS.csv",

    "source_specific":
        "SOURCE_SPECIFIC_METRICS.csv",

    "source_family":
        "SOURCE_FAMILY_PER_CLASS.csv",

    "bootstrap":
        "PAIRED_COMPETING_RISK_BOOTSTRAP.csv",

    "threshold_alerts":
        "FIXED_THRESHOLD_ALERT_AUDIT.csv",

    "exploitation_ci":
        "EXPLOIT_APP_PAIRED_SOURCE_CI.csv"
}

DATA = {}
source_audit = {}
warnings = []

print("\nA. LOADING SAVED REPORTS")
print("-" * 88)

for key, filename in SOURCES.items():

    path = AUDIT / filename

    if not path.is_file():

        warnings.append(
            f"Missing optional report: {filename}"
        )

        print("MISSING:", filename)
        continue

    if "TEST" in filename.upper():
        raise RuntimeError(
            "TEST files are not allowed."
        )

    frame = pd.read_csv(path)

    DATA[key] = frame

    source_audit[key] = {
        "path": str(path.relative_to(PROJECT)),
        "rows": int(len(frame)),
        "columns": frame.columns.tolist(),
        "size_bytes": int(path.stat().st_size)
    }

    print(f"\n{filename}")
    print("  Shape:", frame.shape)
    print("  Columns:", frame.columns.tolist())

# ------------------------------------------------------------
# 2. Compare the four forecasting models.
# ------------------------------------------------------------

if "model_comparison" not in DATA:
    raise RuntimeError(
        "FOUR_MODEL_VALIDATION_COMPARISON.csv missing."
    )

comparison = DATA["model_comparison"].copy()

required_columns = [
    "Model",
    "H5_AP",
    "H10_AP",
    "H15_AP",
    "Evidence"
]

missing = [
    c for c in required_columns
    if c not in comparison.columns
]

if missing:
    raise RuntimeError(
        f"Comparison report missing columns: {missing}"
    )

for col in ["H5_AP", "H10_AP", "H15_AP"]:

    comparison[col] = pd.to_numeric(
        comparison[col],
        errors="coerce"
    )

    valid = comparison[col].dropna()

    if not valid.between(0, 1).all():
        raise ValueError(
            f"Invalid AP values in {col}"
        )

if comparison["Model"].duplicated().any():
    raise ValueError(
        "Duplicate model names in benchmark table."
    )

print("\n" + "=" * 88)
print("B. FOUR-MODEL HORIZON BENCHMARK")
print("=" * 88)

print(
    comparison[
        ["Model", "H5_AP", "H10_AP", "H15_AP"]
    ].to_string(index=False)
)

print("\nMISSING METRICS")

missing_metrics = []

for _, row in comparison.iterrows():

    for metric in [
        "H5_AP", "H10_AP", "H15_AP"
    ]:

        if pd.isna(row[metric]):

            missing_metrics.append({
                "model": str(row["Model"]),
                "metric": metric
            })

            print(
                f"  {row['Model']}: {metric}"
            )

if not missing_metrics:
    print("  None")

print("\nEVIDENCE TYPES")

for _, row in comparison.iterrows():

    print(
        f"  {row['Model']}: {row['Evidence']}"
    )

# ------------------------------------------------------------
# 3. Check recomputed competing-risk metrics.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("C. RECOMPUTED METRIC CONSISTENCY")
print("=" * 88)

consistency_results = []

if "competing_risk" in DATA:

    recomputed = DATA["competing_risk"]

    for _, row in recomputed.iterrows():

        name = str(row["Model"])

        matching = comparison.loc[
            comparison["Model"] == name
        ]

        if matching.empty:
            warnings.append(
                f"{name} absent from comparison table"
            )
            continue

        main = matching.iloc[0]

        for metric in [
            "H5_AP", "H10_AP", "H15_AP"
        ]:

            a = pd.to_numeric(
                main[metric],
                errors="coerce"
            )

            b = pd.to_numeric(
                row[metric],
                errors="coerce"
            )

            equal = bool(
                np.isclose(
                    a, b,
                    atol=1e-8,
                    rtol=1e-7
                )
            )

            consistency_results.append({
                "Model": name,
                "Metric": metric,
                "Match": equal
            })

            print(
                f"{name:<25} "
                f"{metric:<8} "
                f"{'PASS' if equal else 'MISMATCH'}"
            )

            if not equal:
                warnings.append(
                    f"Metric mismatch: {name} {metric}"
                )

# ------------------------------------------------------------
# 4. Audit source-specific comparisons.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("D. SOURCE-SPECIFIC EVALUATION")
print("=" * 88)

if "source_specific" in DATA:

    sources = DATA["source_specific"].copy()

    display_cols = [
        c for c in [
            "Model",
            "Source",
            "Event_Windows",
            "H5_AP",
            "H10_AP",
            "H15_AP",
            "Family_Macro_F1",
            "ETA_MAE"
        ]
        if c in sources.columns
    ]

    print(
        sources[display_cols].round(4)
        .to_string(index=False)
    )

    print("\nSource names:")
    print(
        sorted(
            sources["Source"]
            .dropna()
            .unique()
            .tolist()
        )
    )

# ------------------------------------------------------------
# 5. Family metrics: distinguish absent classes
#    from genuine zero scores.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("E. ATTACK-FAMILY SUPPORT CHECK")
print("=" * 88)

if "source_family" in DATA:

    family = DATA["source_family"].copy()

    family["Support"] = pd.to_numeric(
        family["Support"],
        errors="coerce"
    )

    absent_classes = family.loc[
        family["Support"] == 0,
        ["Model", "Source", "Family"]
    ]

    print(
        "Zero-support class rows:",
        len(absent_classes)
    )

    if len(absent_classes):

        print(
            absent_classes.to_string(
                index=False
            )
        )

    print(
        "\nIMPORTANT: Zero-support classes "
        "must be marked N/A in relevant plots, "
        "not interpreted as measured zero recall."
    )

# ------------------------------------------------------------
# 6. Inspect paired bootstrap evidence.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("F. BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 88)

if "bootstrap" in DATA:

    ci = DATA["bootstrap"].copy()

    print(
        ci.round(5).to_string(
            index=False
        )
    )

    if {
        "CI_Lower_95",
        "CI_Upper_95"
    }.issubset(ci.columns):

        lower = pd.to_numeric(
            ci["CI_Lower_95"],
            errors="coerce"
        )

        upper = pd.to_numeric(
            ci["CI_Upper_95"],
            errors="coerce"
        )

        if not (lower <= upper).all():

            warnings.append(
                "Some bootstrap CI bounds are invalid."
            )

    print(
        "\nPaired bootstrap comparisons apply "
        "only to the models/metrics explicitly "
        "included in this source report."
    )

# ------------------------------------------------------------
# 7. Verify fixed-threshold records.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("G. FIXED-THRESHOLD ALERT METRICS")
print("=" * 88)

if "threshold_alerts" in DATA:

    alerts = DATA["threshold_alerts"]

    print(
        "Threshold evaluation rows:",
        len(alerts)
    )

    print(
        "Models:",
        sorted(
            alerts["Model"].unique().tolist()
        )
    )

    print(
        "Horizons:",
        sorted(
            alerts["Horizon"].unique().tolist()
        )
    )

    print(
        "Thresholds:",
        sorted(
            alerts["Threshold"].unique().tolist()
        )
    )

    # Validate the confusion matrix totals.
    for col in ["TP", "FP", "TN", "FN"]:

        if (alerts[col] < 0).any():
            warnings.append(
                f"Negative counts detected in {col}"
            )

    print(
        "\nAlert metrics are available for "
        "threshold-vs-recall and false-alarm plots."
    )

# ------------------------------------------------------------
# 8. Save validation manifest.
#    Unique filename, no overwriting.
# ------------------------------------------------------------

stamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

REPORT_PATH = (
    MANIFESTS /
    f"forecast_source_validation_{stamp}.json"
)

report = {
    "project": "NetGuard",
    "stage": "Research Outputs Cell 2",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sources": source_audit,

    "four_model_comparison": (
        comparison
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),

    "missing_metrics": missing_metrics,

    "consistency_results": (
        consistency_results
    ),

    "warnings": warnings,

    "scientific_notes": [
        "AP values are not accuracy percentages.",
        "Do not substitute missing H15 metrics.",
        "Distinguish previously reported and recomputed results.",
        "Do not treat zero-support family classes as measured failures.",
        "Do not pool different prediction tasks into one ranking.",
        "Bootstrap CIs apply only to documented paired comparisons.",
        "No official TEST dataset was accessed."
    ]
}

with REPORT_PATH.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        report,
        handle,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 9. Final summary.
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 2 COMPLETE")
print("=" * 88)

print("Reports loaded:", len(DATA))
print("Missing metric cells:", len(missing_metrics))
print("Warnings:", len(warnings))

for warning in warnings:
    print("WARNING:", warning)

print("\nManifest saved:")
print(REPORT_PATH)

print("\nNo model training: YES")
print("Official TEST accessed: NO")
print("Original reports modified: NO")
print("Original checkpoints modified: NO")
print("=" * 88)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 4 — PREDICTION SCHEMA & TEMPORAL ALIGNMENT AUDIT
#
# Goal:
#   Determine which saved models can appear together in
#   research-paper-quality multi-panel forecasting figures.
#
# READS:
#   Saved VALIDATION prediction files only.
#
# WRITES:
#   One NEW JSON audit in Research_Outputs/05_Manifests.
#
# SAFETY:
#   No model training
#   No official TEST access
#   No existing file modifications
#   No row-order alignment assumptions
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import json
import re

import numpy as np
import pandas as pd

print("=" * 90)
print("CELL 4 — TEMPORAL PREDICTION & ALIGNMENT AUDIT")
print("=" * 90)

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUT = PROJECT / "Research_Outputs"
MANIFESTS = OUTPUT / "05_Manifests"

FORECAST_MODELS = (
    PROJECT / "Synthetic_NAF_v2" / "models"
)

GRAPH_MODELS = (
    PROJECT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
)

if not MANIFESTS.is_dir():
    raise RuntimeError("Research_Outputs/05_Manifests missing.")

# ---------------------------------------------------------------
# A. Explicitly declared validation prediction sources.
#    Never search raw, TRAIN, or official TEST directories.
# ---------------------------------------------------------------

SOURCES4 = {
    "Stacked Horizon LSTM": (
        FORECAST_MODELS
        / "LSTM_V2_Horizon"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    ),

    "Single-Layer LSTM": (
        FORECAST_MODELS
        / "LSTM_V2_SingleLayer"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    ),

    "Competing-Risk LSTM": (
        FORECAST_MODELS
        / "LSTM_V2_CompetingRisk"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    ),

    "Competing-Risk TCN": (
        FORECAST_MODELS
        / "TCN_V2_CompetingRisk"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    ),

    "XGBoost V2": (
        FORECAST_MODELS
        / "XGBoost_V2_FLAT140"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.parquet"
    ),

    "GAT-LSTM": (
        GRAPH_MODELS
        / "GAT_LSTM"
        / "validation_v1"
        / "GAT_VAL_PREDICTIONS.npz"
    ),

    "GCN-LSTM": (
        GRAPH_MODELS
        / "GCN_LSTM"
        / "validation_v1"
        / "GCN_VAL_PREDICTIONS.npz"
    )
}

# ---------------------------------------------------------------
# B. Read validation tables.
#    These are saved prediction outputs, not training datasets.
# ---------------------------------------------------------------

TABLES4 = {}
AUDIT4 = {}
ERRORS4 = []

def temporal_candidates(columns):

    aliases = {
        "source": [
            "source", "dataset", "dataset_source",
            "source_dataset", "dataset_name"
        ],
        "episode": [
            "episode_id", "episode", "episode_idx",
            "episode_index", "scenario_id",
            "sequence_id", "seq_id"
        ],
        "time": [
            "time_idx", "time_index", "step",
            "step_idx", "window_idx", "window_index",
            "minute", "time_min", "time_minute",
            "decision_step", "decision_time",
            "timestamp", "window_end", "time", "t"
        ],
        "record_id": [
            "row_id", "record_id", "window_id",
            "sample_id", "validation_id"
        ]
    }

    lookup = {
        str(c).lower(): str(c)
        for c in columns
    }

    result = {}

    for role, names in aliases.items():
        result[role] = next(
            (lookup[n] for n in names if n in lookup),
            None
        )

    return result

def find_prediction_columns(columns):

    columns = list(columns)

    risk = []
    ground_truth = []
    other = []

    for name in columns:

        low = str(name).lower()

        is_horizon = bool(
            re.search(r"(?:^|_)h(?:5|10|15)(?:_|$)", low)
        )

        is_probability = any(
            word in low
            for word in [
                "prob", "risk", "score",
                "pred", "forecast"
            ]
        )

        is_truth = any(
            word in low
            for word in [
                "true", "actual", "target",
                "label", "ground_truth", "y_"
            ]
        )

        if is_truth:
            ground_truth.append(name)

        elif is_horizon and is_probability:
            risk.append(name)

        elif is_probability:
            other.append(name)

    return {
        "horizon_risk_candidates": risk,
        "ground_truth_candidates": ground_truth,
        "other_prediction_candidates": other
    }

print("\nA. MODEL PREDICTION FILE DISCOVERY")
print("-" * 90)

for model_name, path in SOURCES4.items():

    if not path.is_file():

        print(f"\nMISSING: {model_name}")
        print(" ", path)

        AUDIT4[model_name] = {
            "status": "MISSING",
            "path": str(path)
        }

        continue

    relative = path.relative_to(PROJECT)

    if any(
        "test" in component.lower()
        for component in relative.parts
    ):
        raise RuntimeError(
            f"Blocked TEST-like path: {relative}"
        )

    print(f"\nMODEL: {model_name}")
    print("-" * 72)
    print("File:", relative)
    print(
        "Size:",
        round(path.stat().st_size / 1048576, 3),
        "MB"
    )

    try:

        # ------------------------------------------------------
        # NPZ — inspect saved array structure without pickle.
        # ------------------------------------------------------

        if path.suffix.lower() == ".npz":

            with np.load(
                str(path),
                allow_pickle=False
            ) as archive:

                arrays = {}

                for key in archive.files:

                    try:
                        arr = archive[key]

                        arrays[key] = {
                            "shape": list(arr.shape),
                            "dtype": str(arr.dtype)
                        }

                        print(
                            f"  {key:<35} "
                            f"shape={arr.shape} "
                            f"dtype={arr.dtype}"
                        )

                    except Exception as error:

                        arrays[key] = {
                            "error": str(error)
                        }

                        print(
                            f"  {key}: "
                            f"COULD NOT INSPECT — {error}"
                        )

            AUDIT4[model_name] = {
                "status": "NPZ_INSPECTED",
                "path": str(relative),
                "arrays": arrays,
                "alignment": "NOT_YET_VERIFIED"
            }

            continue

        # ------------------------------------------------------
        # CSV / Parquet — inspect complete saved validation table.
        # ------------------------------------------------------

        if path.suffix.lower() == ".csv":

            frame = pd.read_csv(
                path,
                low_memory=False
            )

        elif path.suffix.lower() == ".parquet":

            frame = pd.read_parquet(path)

        else:

            print("Unsupported file format; skipping.")
            continue

        if frame.empty:
            print("WARNING: prediction table is empty.")
            continue

        TABLES4[model_name] = frame

        keys = temporal_candidates(
            frame.columns
        )

        predictions = find_prediction_columns(
            frame.columns
        )

        print("Shape:", frame.shape)

        print("\nAll columns:")

        for i, col in enumerate(frame.columns, 1):
            print(f"  {i:02d}. {col}")

        print("\nPossible alignment keys:")

        for role, col in keys.items():
            print(
                f"  {role:<12}: "
                f"{col if col else 'NOT FOUND'}"
            )

        print("\nHorizon-risk candidates:")
        print(
            predictions[
                "horizon_risk_candidates"
            ] or "None identified"
        )

        print("\nGround-truth candidates:")
        print(
            predictions[
                "ground_truth_candidates"
            ] or "None identified"
        )

        # Show limited sample rows, not the entire dataset.
        preview_columns = list(dict.fromkeys(
            [
                c for c in keys.values()
                if c is not None
            ]
            + predictions[
                "ground_truth_candidates"
            ][:4]
            + predictions[
                "horizon_risk_candidates"
            ][:6]
        ))

        if not preview_columns:
            preview_columns = list(
                frame.columns[:8]
            )

        print("\nFirst three records:")

        print(
            frame[preview_columns]
            .head(3)
            .to_string(index=False)
        )

        AUDIT4[model_name] = {
            "status": "TABLE_LOADED",
            "path": str(relative),
            "rows": int(len(frame)),
            "columns": frame.columns.tolist(),
            "alignment_key_candidates": keys,
            "prediction_columns": predictions,
            "alignment": "NOT_YET_VERIFIED"
        }

    except Exception as error:

        print(
            "ERROR:",
            type(error).__name__,
            str(error)
        )

        ERRORS4.append({
            "model": model_name,
            "error": str(error)
        })

        AUDIT4[model_name] = {
            "status": "ERROR",
            "path": str(relative),
            "error": str(error)
        }

# ---------------------------------------------------------------
# C. Pairwise temporal alignment checks.
#
# Only attempt a join when both tables have:
#   - an explicit source identifier
#   - an explicit episode identifier
#   - an explicit time/step identifier
#
# Never join based solely on row number or sorting.
# ---------------------------------------------------------------

print("\n" + "=" * 90)
print("B. PAIRWISE TEMPORAL ALIGNMENT")
print("=" * 90)

ALIGNMENT4 = []

for model_a, model_b in combinations(
    TABLES4.keys(),
    2
):

    frame_a = TABLES4[model_a]
    frame_b = TABLES4[model_b]

    keys_a = AUDIT4[model_a][
        "alignment_key_candidates"
    ]

    keys_b = AUDIT4[model_b][
        "alignment_key_candidates"
    ]

    roles = ["source", "episode", "time"]

    if not all(
        keys_a.get(role) is not None
        and keys_b.get(role) is not None
        for role in roles
    ):

        status = "NOT VERIFIED — MISSING EXPLICIT KEYS"

        print(
            f"{model_a:<25} vs {model_b:<25}"
            f" {status}"
        )

        ALIGNMENT4.append({
            "model_a": model_a,
            "model_b": model_b,
            "status": status
        })

        continue

    try:

        left = frame_a[
            [keys_a[role] for role in roles]
        ].copy()

        right = frame_b[
            [keys_b[role] for role in roles]
        ].copy()

        left.columns = roles
        right.columns = roles

        # Join keys must be present, not null.
        left = left.dropna()
        right = right.dropna()

        # Strings prevent mismatched categorical encodings
        # but do NOT establish common time units.
        for role in roles:

            left[role] = (
                left[role].astype(str).str.strip()
            )

            right[role] = (
                right[role].astype(str).str.strip()
            )

        left_unique = left.drop_duplicates()
        right_unique = right.drop_duplicates()

        overlaps = left_unique.merge(
            right_unique,
            on=roles,
            how="inner"
        )

        overlap_count = int(len(overlaps))

        left_fraction = (
            overlap_count / len(left_unique)
            if len(left_unique)
            else 0
        )

        right_fraction = (
            overlap_count / len(right_unique)
            if len(right_unique)
            else 0
        )

        has_duplicates = (
            len(left) != len(left_unique)
            or len(right) != len(right_unique)
        )

        if overlap_count == 0:
            status = "NO MATCHING COMPOSITE KEYS"

        elif has_duplicates:
            status = "REVIEW — DUPLICATE KEYS"

        else:
            status = (
                "CANDIDATE ALIGNMENT — "
                "SEMANTICS STILL REQUIRE VERIFICATION"
            )

        print(
            f"\n{model_a} vs {model_b}"
        )

        print("  Status:", status)
        print("  Shared unique keys:", overlap_count)
        print("  Coverage A:", f"{left_fraction:.2%}")
        print("  Coverage B:", f"{right_fraction:.2%}")
        print("  Duplicate keys:", has_duplicates)

        ALIGNMENT4.append({
            "model_a": model_a,
            "model_b": model_b,
            "status": status,
            "shared_unique_keys": overlap_count,
            "coverage_a": left_fraction,
            "coverage_b": right_fraction,
            "duplicate_keys": bool(has_duplicates),
            "join_roles": roles
        })

    except Exception as error:

        print(
            f"{model_a} vs {model_b}: "
            f"ERROR {error}"
        )

        ALIGNMENT4.append({
            "model_a": model_a,
            "model_b": model_b,
            "status": "ERROR",
            "error": str(error)
        })

# ---------------------------------------------------------------
# D. Save new audit manifest only.
# ---------------------------------------------------------------

print("\n" + "=" * 90)
print("C. RESEARCH OUTPUT READINESS")
print("=" * 90)

print("\nPrediction tables loaded:", len(TABLES4))

print(
    "NPZ prediction archives inspected:",
    sum(
        d["status"] == "NPZ_INSPECTED"
        for d in AUDIT4.values()
    )
)

print("Read errors:", len(ERRORS4))

for error in ERRORS4:
    print("ERROR:", error)

stamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

manifest_path4 = (
    MANIFESTS
    / f"temporal_alignment_audit_{stamp}.json"
)

manifest4 = {
    "project": "NetGuard",
    "stage": "Research Outputs — Cell 4",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sources": AUDIT4,
    "pairwise_alignment": ALIGNMENT4,
    "errors": ERRORS4,
    "scientific_constraints": [
        "No alignment by row order.",
        "Matching key strings alone do not prove identical time units.",
        "Model horizon definitions must be checked.",
        "Ground-truth values must match on aligned records.",
        "Separate datasets and prediction tasks cannot be merged without validation.",
        "No official TEST files opened."
    ]
}

with manifest_path4.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest4,
        handle,
        indent=2,
        default=str
    )

print("\nManifest saved:")
print(manifest_path4)

print("\n" + "=" * 90)
print("CELL 4 COMPLETE")
print("=" * 90)

print("New models trained: 0")
print("Official TEST accessed: NO")
print("Original artifacts modified: NO")
print("Existing figures overwritten: NO")
print("New audit manifest created: YES")
print("=" * 90)


In [ ]:

# ================================================================
# NETGUARD RESEARCH OUTPUTS
# CELL 5 — SEVEN-MODEL TEMPORAL FORECASTING ATLAS
#
# Output:
#   F01_Seven_Model_Temporal_Atlas
#   4 rows x 7 models, using real H10 predictions
#
# Scientific checks:
#   - Exact Episode_ID + Minute alignment
#   - No duplicate observation keys
#   - Matching H5/H10/H15 ground truth
#   - Matching sources where available
#   - No alignment by row order
#   - No selection based on model performance
#
# Saves only NEW research figure/table/manifest files.
# No training or official TEST access.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

print("=" * 95)
print("CELL 5 — SEVEN-MODEL TEMPORAL FORECASTING ATLAS")
print("=" * 95)

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

FORECAST = PROJECT / "Synthetic_NAF_v2" / "models"

GRAPH = (
    PROJECT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
)

OUTPUT = PROJECT / "Research_Outputs"

FIGURES = OUTPUT / "01_Figures" / "Forecasting"
TABLES = OUTPUT / "02_Tables"
MANIFESTS = OUTPUT / "05_Manifests"

for folder in [FIGURES, TABLES, MANIFESTS]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

# ---------------------------------------------------------------
# 1. Explicit model sources.
# ---------------------------------------------------------------

SOURCES5 = {
    "Single LSTM": (
        FORECAST / "LSTM_V2_SingleLayer"
        / "official_validation" / "VALIDATION_PREDICTIONS.csv"
    ),
    "Stacked LSTM": (
        FORECAST / "LSTM_V2_Horizon"
        / "official_validation" / "VALIDATION_PREDICTIONS.csv"
    ),
    "XGBoost V2": (
        FORECAST / "XGBoost_V2_FLAT140"
        / "official_validation" / "VALIDATION_PREDICTIONS.parquet"
    ),
    "CR-LSTM": (
        FORECAST / "LSTM_V2_CompetingRisk"
        / "official_validation" / "VALIDATION_PREDICTIONS.csv"
    ),
    "CR-TCN": (
        FORECAST / "TCN_V2_CompetingRisk"
        / "official_validation" / "VALIDATION_PREDICTIONS.csv"
    ),
    "GAT-LSTM": (
        GRAPH / "GAT_LSTM"
        / "validation_v1" / "GAT_VAL_PREDICTIONS.npz"
    ),
    "GCN-LSTM": (
        GRAPH / "GCN_LSTM"
        / "validation_v1" / "GCN_VAL_PREDICTIONS.npz"
    )
}

for name, path in SOURCES5.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{name} prediction file missing: {path}"
        )

# ---------------------------------------------------------------
# 2. Load the stacked LSTM as the alignment reference.
#
# It has explicit source, episode, minute and
# all three future-event targets.
# ---------------------------------------------------------------

reference5 = pd.read_csv(
    SOURCES5["Stacked LSTM"]
)

required_ref5 = [
    "Episode_ID", "Minute", "Source",
    "Y_H5", "Y_H10", "Y_H15",
    "P_H5", "P_H10", "P_H15"
]

if not set(required_ref5).issubset(reference5.columns):
    raise RuntimeError("Reference schema mismatch.")

def normalize_keys5(df):
    df = df.copy()

    df["Episode_ID"] = (
        df["Episode_ID"].astype(str).str.strip()
    )

    df["Minute"] = pd.to_numeric(
        df["Minute"], errors="raise"
    )

    if df[["Episode_ID", "Minute"]].isna().any().any():
        raise RuntimeError("Null alignment keys found.")

    if df.duplicated(["Episode_ID", "Minute"]).any():
        raise RuntimeError(
            "Duplicate Episode_ID + Minute keys found."
        )

    return df

reference5 = normalize_keys5(reference5)

if reference5.groupby("Episode_ID")["Source"].nunique().max() != 1:
    raise RuntimeError(
        "An Episode_ID maps to multiple sources."
    )

reference5 = reference5.set_index(
    ["Episode_ID", "Minute"],
    verify_integrity=True
).sort_index()

REF_INDEX5 = reference5.index

TRUTH5 = {}

for h in [5, 10, 15]:

    values = pd.to_numeric(
        reference5[f"Y_H{h}"],
        errors="raise"
    ).to_numpy()

    if not np.isin(values, [0, 1]).all():
        raise RuntimeError(
            f"Reference H{h} target is not binary."
        )

    TRUTH5[h] = values.astype(np.int8)

print("\nREFERENCE VALIDATION")
print("-" * 95)

print("Records:", len(reference5))
print(
    "Episodes:",
    reference5.index.get_level_values(0).nunique()
)
print(
    "Sources:",
    reference5["Source"].value_counts().to_dict()
)

# ---------------------------------------------------------------
# 3. Load and align tabular model predictions.
# ---------------------------------------------------------------

TABULAR5 = {
    "Single LSTM": {
        "columns": {
            5: "P_H5",
            10: "P_H10",
            15: "P_H15"
        },
        "truth": {
            5: "Y_H5",
            10: "Y_H10",
            15: "Y_H15"
        }
    },
    "Stacked LSTM": {
        "columns": {
            5: "P_H5",
            10: "P_H10",
            15: "P_H15"
        },
        "truth": {
            5: "Y_H5",
            10: "Y_H10",
            15: "Y_H15"
        }
    },
    "XGBoost V2": {
        "columns": {
            5: "H5_Probability",
            10: "H10_Probability"
        },
        "truth": {
            5: "H5_Actual",
            10: "H10_Actual"
        }
    },
    "CR-LSTM": {
        "columns": {
            5: "P_H5",
            10: "P_H10",
            15: "P_H15"
        },
        "truth": {
            5: "True_H5",
            10: "True_H10",
            15: "True_H15"
        }
    },
    "CR-TCN": {
        "columns": {
            5: "P_H5",
            10: "P_H10",
            15: "P_H15"
        },
        "truth": {
            5: "True_H5",
            10: "True_H10",
            15: "True_H15"
        }
    }
}

MODEL_PROB5 = {}
VALIDATION_CHECKS5 = []

def verify_probabilities5(name, array):
    array = np.asarray(array, dtype=float)

    if len(array) != len(REF_INDEX5):
        raise RuntimeError(f"{name}: row count mismatch.")

    if not np.isfinite(array).all():
        raise RuntimeError(f"{name}: nonfinite probabilities.")

    if np.any(array < -1e-7) or np.any(array > 1+1e-7):
        raise RuntimeError(f"{name}: probability outside [0,1].")

    return np.clip(array, 0, 1)

print("\nA. VALIDATING TABULAR PREDICTIONS")
print("-" * 95)

for name, config in TABULAR5.items():

    if name == "Stacked LSTM":
        frame = reference5.copy()

    else:
        path = SOURCES5[name]

        if path.suffix == ".parquet":
            frame = pd.read_parquet(path)
        else:
            frame = pd.read_csv(path)

        frame = normalize_keys5(frame)

        frame = frame.set_index(
            ["Episode_ID", "Minute"],
            verify_integrity=True
        ).sort_index()

    if len(frame) != len(REF_INDEX5):
        raise RuntimeError(f"{name}: row count differs.")

    missing = REF_INDEX5.difference(frame.index)
    extra = frame.index.difference(REF_INDEX5)

    if len(missing) or len(extra):
        raise RuntimeError(
            f"{name}: key mismatch. "
            f"Missing={len(missing)}, extra={len(extra)}"
        )

    frame = frame.reindex(REF_INDEX5)

    if "Source" in frame.columns:

        same_source = np.array_equal(
            frame["Source"].astype(str).to_numpy(),
            reference5["Source"].astype(str).to_numpy()
        )

        if not same_source:
            raise RuntimeError(f"{name}: Source mismatch.")

    # Check labels before using model probabilities.
    for h, column in config["truth"].items():

        actual = pd.to_numeric(
            frame[column], errors="raise"
        ).to_numpy()

        if not np.array_equal(actual, TRUTH5[h]):
            mismatch = int(
                np.sum(actual != TRUTH5[h])
            )
            raise RuntimeError(
                f"{name} H{h}: {mismatch} label mismatches."
            )

    MODEL_PROB5[name] = {}

    for h, column in config["columns"].items():

        MODEL_PROB5[name][h] = verify_probabilities5(
            f"{name} H{h}",
            pd.to_numeric(
                frame[column], errors="raise"
            ).to_numpy()
        )

    VALIDATION_CHECKS5.append({
        "model": name,
        "records": len(frame),
        "keys_match": True,
        "labels_match": True,
        "horizons": list(config["columns"].keys())
    })

    print(
        f"PASS | {name:<18} "
        f"{len(frame):>6} matched records"
    )

# ---------------------------------------------------------------
# 4. Align graph model NPZ files by keys.
#
# Never assume NPZ row order matches CSV row order.
# ---------------------------------------------------------------

GRAPH_CONFIG5 = {
    "GAT-LSTM": "gat_probabilities",
    "GCN-LSTM": "gcn_probabilities"
}

print("\nB. VALIDATING GRAPH-MODEL PREDICTIONS")
print("-" * 95)

for name, probability_key in GRAPH_CONFIG5.items():

    with np.load(
        SOURCES5[name],
        allow_pickle=False
    ) as archive:

        episode_ids = archive["episode_id"]
        minutes = archive["minute"]
        labels = archive["y_horizons"]
        probabilities = archive[probability_key]

    if labels.shape != (len(REF_INDEX5), 3):
        raise RuntimeError(
            f"{name}: unexpected label shape {labels.shape}"
        )

    if probabilities.shape != (len(REF_INDEX5), 3):
        raise RuntimeError(
            f"{name}: unexpected prediction shape."
        )

    graph_keys = pd.DataFrame({
        "Episode_ID": episode_ids.astype(str),
        "Minute": minutes
    })

    graph_keys = normalize_keys5(graph_keys)

    graph_keys = pd.MultiIndex.from_frame(
        graph_keys[["Episode_ID", "Minute"]]
    )

    if len(graph_keys.difference(REF_INDEX5)):
        raise RuntimeError(
            f"{name}: extra graph prediction keys."
        )

    if len(REF_INDEX5.difference(graph_keys)):
        raise RuntimeError(
            f"{name}: missing graph prediction keys."
        )

    if len(graph_keys) != len(REF_INDEX5):
        raise RuntimeError(
            f"{name}: inconsistent prediction count."
        )

    graph_frame = pd.DataFrame(
        {
            "Y_H5": labels[:, 0],
            "Y_H10": labels[:, 1],
            "Y_H15": labels[:, 2],
            "P_H5": probabilities[:, 0],
            "P_H10": probabilities[:, 1],
            "P_H15": probabilities[:, 2]
        },
        index=graph_keys
    )

    graph_frame = graph_frame.reindex(REF_INDEX5)

    MODEL_PROB5[name] = {}

    for h in [5, 10, 15]:

        graph_truth = graph_frame[
            f"Y_H{h}"
        ].to_numpy()

        if not np.array_equal(graph_truth, TRUTH5[h]):
            raise RuntimeError(
                f"{name}: H{h} ground-truth mismatch."
            )

        MODEL_PROB5[name][h] = verify_probabilities5(
            f"{name} H{h}",
            graph_frame[f"P_H{h}"].to_numpy()
        )

    VALIDATION_CHECKS5.append({
        "model": name,
        "records": len(graph_frame),
        "keys_match": True,
        "labels_match": True,
        "horizons": [5, 10, 15]
    })

    print(
        f"PASS | {name:<18} "
        f"{len(graph_frame):>6} matched records"
    )

print("\nALL SEVEN MODELS VERIFIED.")

# ---------------------------------------------------------------
# 5. Select four representative episodes.
#
# Predefined selection:
#   CIC: first H10-positive and first H10-negative episode
#   UNSW: first H10-positive and first H10-negative episode
#
# At least 25 recorded decision windows.
# Alphabetical episode order.
#
# This avoids selecting episodes based on model performance.
# ---------------------------------------------------------------

episode_table5 = reference5.reset_index()[
    ["Episode_ID", "Minute", "Source", "Y_H10"]
].copy()

episode_stats5 = (
    episode_table5
    .groupby(["Source", "Episode_ID"], as_index=False)
    .agg(
        n_windows=("Minute", "size"),
        h10_positive=("Y_H10", "sum")
    )
)

episode_stats5 = episode_stats5.loc[
    episode_stats5["n_windows"] >= 25
].sort_values(["Source", "Episode_ID"])

selected5 = []

for source in ["CIC_IDS2017", "UNSW_NB15"]:

    subset = episode_stats5.loc[
        episode_stats5["Source"] == source
    ]

    positives = subset.loc[
        subset["h10_positive"] > 0
    ]

    negatives = subset.loc[
        subset["h10_positive"] == 0
    ]

    if positives.empty or negatives.empty:
        raise RuntimeError(
            f"{source}: cannot find both episode types. "
            "Review selection criteria before plotting."
        )

    selected5.append(positives.iloc[0])
    selected5.append(negatives.iloc[0])

selected5 = pd.DataFrame(
    selected5
).reset_index(drop=True)

print("\nC. DETERMINISTIC EPISODE SELECTION")
print("-" * 95)

print(
    selected5.to_string(index=False)
)

# ---------------------------------------------------------------
# 6. Publication-oriented plotting design.
# ---------------------------------------------------------------

MODEL_ORDER5 = [
    "Single LSTM",
    "Stacked LSTM",
    "XGBoost V2",
    "CR-LSTM",
    "CR-TCN",
    "GCN-LSTM",
    "GAT-LSTM"
]

MODEL_COLORS5 = {
    "Single LSTM": "#306C96",
    "Stacked LSTM": "#264C84",
    "XGBoost V2": "#BA6937",
    "CR-LSTM": "#17857B",
    "CR-TCN": "#8B5D9F",
    "GCN-LSTM": "#708332",
    "GAT-LSTM": "#C15B79"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig, axes = plt.subplots(
    nrows=4,
    ncols=7,
    figsize=(23, 12),
    squeeze=False,
    sharey=True
)

fig.suptitle(
    "Temporal comparison of future-attack risk across seven models",
    x=0.075,
    y=0.985,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

def positive_spans5(minutes, truth):
    """Consecutive integer-minute H10-positive intervals."""

    points = np.asarray(minutes)[
        np.asarray(truth) == 1
    ]

    if len(points) == 0:
        return []

    points = np.sort(np.unique(points))

    boundaries = np.flatnonzero(
        np.diff(points) > 1
    )

    starts = np.r_[
        0, boundaries + 1
    ]

    ends = np.r_[
        boundaries, len(points) - 1
    ]

    return [
        (float(points[a]) - 0.45,
         float(points[b]) + 0.45)
        for a, b in zip(starts, ends)
    ]

for row_idx, selection in selected5.iterrows():

    episode_id = str(selection["Episode_ID"])

    # Obtain exactly this episode's aligned reference rows.
    mask = (
        REF_INDEX5.get_level_values("Episode_ID")
        == episode_id
    )

    minutes = REF_INDEX5.get_level_values(
        "Minute"
    ).to_numpy()[mask]

    y_h10 = TRUTH5[10][mask]

    order = np.argsort(minutes)

    minutes = minutes[order]
    y_h10 = y_h10[order]

    shade_ranges = positive_spans5(
        minutes, y_h10
    )

    source_label = (
        "CIC-derived"
        if selection["Source"] == "CIC_IDS2017"
        else "UNSW-derived"
    )

    episode_type = (
        "H10-positive"
        if int(selection["h10_positive"]) > 0
        else "No H10-positive windows"
    )

    for col_idx, model in enumerate(MODEL_ORDER5):

        ax = axes[row_idx, col_idx]

        probabilities = (
            MODEL_PROB5[model][10][mask][order]
        )

        # Highlight ground-truth positive decision windows.
        for start, end in shade_ranges:

            ax.axvspan(
                start, end,
                color="#E4BD72",
                alpha=0.24,
                linewidth=0,
                zorder=0
            )

        ax.plot(
            minutes,
            probabilities,
            color=MODEL_COLORS5[model],
            lw=1.65,
            alpha=0.95,
            solid_joinstyle="round",
            zorder=3
        )

        ax.fill_between(
            minutes,
            probabilities,
            0,
            color=MODEL_COLORS5[model],
            alpha=0.055,
            zorder=1
        )

        ax.set_ylim(-0.025, 1.025)

        ax.set_xlim(
            float(minutes.min()),
            float(minutes.max())
        )

        ax.set_facecolor("#FAFAFB")

        ax.grid(
            axis="y",
            color="#B8BDC4",
            linestyle=":",
            alpha=0.45,
            linewidth=0.65
        )

        ax.set_axisbelow(True)

        ax.set_yticks(
            [0, 0.25, 0.5, 0.75, 1.0]
        )

        ax.xaxis.set_major_locator(
            MaxNLocator(
                nbins=4,
                integer=True
            )
        )

        ax.tick_params(
            axis="both",
            labelsize=8,
            length=2.5,
            width=0.6,
            pad=3
        )

        if col_idx == 0:

            ax.set_ylabel(
                "Attack probability",
                fontsize=9,
                labelpad=8
            )

        if row_idx == 0:

            ax.set_title(
                model,
                fontsize=12,
                fontweight="bold",
                color=MODEL_COLORS5[model],
                pad=11
            )

        if row_idx == 3:

            ax.set_xlabel(
                "Decision minute",
                fontsize=9,
                labelpad=7
            )

    # Row descriptions are outside the leftmost subplot.
    axes[row_idx, 0].annotate(
        f"{source_label}\n{episode_type}\n"
        f"{episode_id}",
        xy=(0, 0.5),
        xycoords="axes fraction",
        xytext=(-0.41, 0.5),
        textcoords="axes fraction",
        ha="right",
        va="center",
        fontsize=8,
        color="#303942",
        annotation_clip=False
    )

fig.legend(
    handles=[
        Patch(
            facecolor="#E4BD72",
            alpha=0.38,
            edgecolor="none",
            label="Ground truth: attack within next 10 min"
        )
    ],
    loc="lower center",
    bbox_to_anchor=(0.52, 0.038),
    frameon=False,
    fontsize=11
)

fig.text(
    0.075, 0.013,
    "Synthetic NAF v2 development validation  |  "
    "H10 = probability of an attack within the next 10 minutes  |  "
    "Episode selection uses true labels, not predicted performance.",
    fontsize=9.2,
    color="#56616C"
)

fig.subplots_adjust(
    left=0.17,
    right=0.985,
    top=0.91,
    bottom=0.12,
    hspace=0.35,
    wspace=0.20
)

# ---------------------------------------------------------------
# 7. Save vector and raster outputs.
# ---------------------------------------------------------------

RUN5 = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

NAME5 = (
    f"F01_Seven_Model_Temporal_Atlas_{RUN5}"
)

saved5 = []

for ext in ["pdf", "svg", "png"]:

    path = FIGURES / f"{NAME5}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig.savefig(
        path,
        dpi=450 if ext == "png" else 150,
        facecolor="white"
    )

    saved5.append(path)

plt.show()
plt.close(fig)

# ---------------------------------------------------------------
# 8. Save selected episode metadata.
# ---------------------------------------------------------------

TABLE5 = TABLES / (
    f"T01_Temporal_Atlas_Episode_Selection_{RUN5}.csv"
)

selected5.to_csv(
    TABLE5,
    index=False,
    mode="x"
)

# ---------------------------------------------------------------
# 9. Create a reproducibility manifest.
# ---------------------------------------------------------------

def hash_file5(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest5 = {
    "figure": NAME5,
    "dataset": "Synthetic_NAF_v2",
    "split": "development_validation",
    "horizon": "H10",
    "validation_records": len(REF_INDEX5),
    "alignment_keys": ["Episode_ID", "Minute"],
    "alignment_verified": True,
    "ground_truth_verified": True,
    "episode_selection_rule": (
        "For CIC_IDS2017 and UNSW_NB15, choose the "
        "lexicographically first episode with at least "
        "25 windows and a positive H10 target, plus "
        "the first episode with no positive H10 target."
    ),
    "episodes": selected5.to_dict("records"),
    "models": MODEL_ORDER5,
    "source_files": {
        name: {
            "path": str(path.relative_to(PROJECT)),
            "sha256": hash_file5(path)
        }
        for name, path in SOURCES5.items()
    },
    "generated_files": [
        {
            "path": str(path.relative_to(PROJECT)),
            "sha256": hash_file5(path)
        }
        for path in saved5 + [TABLE5]
    ],
    "limitations": [
        "These are synthetic forecasting validation windows.",
        "H10 target-positive regions are not exact attack-onset timestamps.",
        "Graph model source membership is inherited only after verifying matched observation keys and labels.",
        "The displayed episodes are illustrative, not representative accuracy estimates.",
        "No official TEST data was used."
    ]
}

MANIFEST5 = MANIFESTS / (
    f"temporal_atlas_manifest_{RUN5}.json"
)

with MANIFEST5.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest5,
        handle,
        indent=2,
        default=str
    )

# ---------------------------------------------------------------
# 10. Final verification.
# ---------------------------------------------------------------

print("\n" + "=" * 95)
print("CELL 5 COMPLETE")
print("=" * 95)

print("Validated models:", len(MODEL_ORDER5))
print("Validated windows:", len(REF_INDEX5))
print("Selected episodes:", len(selected5))
print("Subplots generated:", len(selected5) * len(MODEL_ORDER5))

print("\nSaved files:")
for file in saved5 + [TABLE5, MANIFEST5]:
    assert file.is_file()
    print(" -", file.relative_to(OUTPUT))

print("\nNew models trained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Source predictions modified: NO")
print("New figure generated: YES")
print("=" * 95)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 6: CROSS-FAMILY MULTI-HORIZON FORECASTING ATLAS
#
# Five attack families × eight columns:
#   Ground truth + seven forecasting models
#
# Each model panel:
#   H5, H10 and H15 predicted risk where available
#   H10-positive ground-truth regions
#   Illustrative 0.5 decision threshold
#
# Uses ALREADY VERIFIED aligned predictions from Cell 5.
#
# NO retraining.
# NO official TEST access.
# NO existing artifact overwrites.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 95)
print("CELL 6 — CROSS-FAMILY MULTI-HORIZON ATLAS")
print("=" * 95)

# ---------------------------------------------------------------
# 1. Verify Cell 5's validated predictions are in memory.
# ---------------------------------------------------------------

required6 = [
    "reference5",
    "REF_INDEX5",
    "TRUTH5",
    "MODEL_PROB5",
    "SOURCES5"
]

for variable in required6:
    if variable not in globals():
        raise RuntimeError(
            f"Missing {variable}. Run the successful "
            "Cell 5 validation first."
        )

PROJECT6 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

FIGURES6 = (
    PROJECT6
    / "Research_Outputs"
    / "01_Figures"
    / "Forecasting"
)

TABLES6 = (
    PROJECT6
    / "Research_Outputs"
    / "02_Tables"
)

MANIFESTS6 = (
    PROJECT6
    / "Research_Outputs"
    / "05_Manifests"
)

for directory in [FIGURES6, TABLES6, MANIFESTS6]:
    if not directory.is_dir():
        raise FileNotFoundError(directory)

# ---------------------------------------------------------------
# 2. Select attack-family episodes using LABELS ONLY.
#
# No prediction scores are used for episode selection.
#
# Alternate preferred sources where supported to
# illustrate both source datasets.
# ---------------------------------------------------------------

reference_table6 = reference5.reset_index().copy()

needed6 = {
    "Episode_ID",
    "Minute",
    "Source",
    "Family_Metadata_Audit_Only",
    "Y_H5",
    "Y_H10",
    "Y_H15"
}

if not needed6.issubset(reference_table6.columns):
    raise RuntimeError(
        "Expected reference columns are missing."
    )

reference_table6["Family"] = (
    reference_table6["Family_Metadata_Audit_Only"]
    .astype(str)
    .str.strip()
    .str.upper()
)

episode_stats6 = (
    reference_table6
    .groupby("Episode_ID", as_index=False)
    .agg(
        Source=("Source", "first"),
        Family=("Family", "first"),
        Windows=("Minute", "size"),
        H5_Positive=("Y_H5", "sum"),
        H10_Positive=("Y_H10", "sum"),
        H15_Positive=("Y_H15", "sum")
    )
)

# Ensure source and family metadata are constant per episode.
for col in ["Source", "Family"]:

    diversity = (
        reference_table6
        .groupby("Episode_ID")[col]
        .nunique()
    )

    if diversity.max() > 1:
        raise RuntimeError(
            f"Inconsistent {col} metadata within episodes."
        )

family_preferences6 = [
    ("BRUTE_AUTH", "CIC_IDS2017"),
    ("DOS_DDOS", "UNSW_NB15"),
    ("EXPLOIT_APP", "CIC_IDS2017"),
    ("FUZZING", "UNSW_NB15"),
    ("RECON_SCAN", "CIC_IDS2017")
]

selected_rows6 = []

for family, preferred_source in family_preferences6:

    candidates = episode_stats6.loc[
        (episode_stats6["Family"] == family)
        & (episode_stats6["H10_Positive"] > 0)
        & (episode_stats6["Windows"] >= 30)
    ].copy()

    if candidates.empty:
        raise RuntimeError(
            f"No eligible episode for {family}. "
            "Review available family labels."
        )

    # Prefer requested source if eligible; otherwise use
    # the available source, preserving deterministic order.
    preferred = candidates.loc[
        candidates["Source"] == preferred_source
    ]

    if not preferred.empty:
        candidates = preferred

    chosen = (
        candidates
        .sort_values("Episode_ID")
        .iloc[0]
        .to_dict()
    )

    selected_rows6.append(chosen)

selected6 = pd.DataFrame(selected_rows6)

print("\nA. SELECTED ATTACK-FAMILY EPISODES")
print("-" * 95)

print(
    selected6[
        [
            "Family",
            "Source",
            "Episode_ID",
            "Windows",
            "H5_Positive",
            "H10_Positive",
            "H15_Positive"
        ]
    ].to_string(index=False)
)

# ---------------------------------------------------------------
# 3. Configure an academic figure design system.
# ---------------------------------------------------------------

MODELS6 = [
    "Single LSTM",
    "Stacked LSTM",
    "XGBoost V2",
    "CR-LSTM",
    "CR-TCN",
    "GCN-LSTM",
    "GAT-LSTM"
]

HORIZONS6 = [5, 10, 15]

HORIZON_COLORS6 = {
    5: "#267A83",
    10: "#244C91",
    15: "#B76042"
}

HORIZON_STYLES6 = {
    5: (0, (1.5, 1.5)),
    10: "-",
    15: (0, (5, 2))
}

HORIZON_WIDTHS6 = {
    5: 1.50,
    10: 2.05,
    15: 1.65
}

FAMILY_LABELS6 = {
    "BRUTE_AUTH": "Brute-force / authentication",
    "DOS_DDOS": "DoS / DDoS",
    "EXPLOIT_APP": "Application exploitation",
    "FUZZING": "Fuzzing",
    "RECON_SCAN": "Reconnaissance / scanning"
}

SOURCE_LABELS6 = {
    "CIC_IDS2017": "CIC-derived",
    "UNSW_NB15": "UNSW-derived"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

# ---------------------------------------------------------------
# 4. Build the five-row, eight-column panel atlas.
# ---------------------------------------------------------------

n_rows6 = len(selected6)
n_cols6 = len(MODELS6) + 1

fig6, axes6 = plt.subplots(
    nrows=n_rows6,
    ncols=n_cols6,
    figsize=(27, 16),
    sharey=True,
    squeeze=False
)

fig6.patch.set_facecolor("white")

fig6.suptitle(
    "Multi-horizon attack-risk trajectories across "
    "network attack families",
    x=0.12,
    y=0.988,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig6.text(
    0.12,
    0.954,
    "Aligned synthetic validation episodes  |  "
    "Five attack families  |  "
    "Seven forecasting architectures",
    fontsize=11,
    color="#576574"
)

def positive_spans6(minutes, binary_labels):
    """
    Return contiguous integer-minute intervals where
    the specified target is positive.
    """
    minutes = np.asarray(minutes)
    labels = np.asarray(binary_labels)

    positive = np.sort(
        np.unique(minutes[labels == 1])
    )

    if len(positive) == 0:
        return []

    breaks = np.flatnonzero(
        np.diff(positive) > 1
    )

    starts = np.r_[0, breaks + 1]
    ends = np.r_[breaks, len(positive) - 1]

    return [
        (
            float(positive[a]) - 0.45,
            float(positive[b]) + 0.45
        )
        for a, b in zip(starts, ends)
    ]

def format_panel6(ax, minutes, ground_truth_h10):

    ax.set_facecolor("#FBFCFD")

    for start, end in positive_spans6(
        minutes,
        ground_truth_h10
    ):
        ax.axvspan(
            start,
            end,
            color="#E4B75C",
            alpha=0.23,
            linewidth=0,
            zorder=0
        )

    ax.axhline(
        0.5,
        color="#7B8490",
        linestyle=(0, (2, 3)),
        linewidth=0.75,
        alpha=0.60,
        zorder=1
    )

    ax.set_ylim(-0.035, 1.035)

    ax.set_xlim(
        float(np.min(minutes)),
        float(np.max(minutes))
    )

    ax.set_yticks(
        [0, 0.25, 0.5, 0.75, 1.0]
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=4,
            integer=True
        )
    )

    ax.grid(
        axis="y",
        linestyle=":",
        color="#AEB8C4",
        alpha=0.35,
        linewidth=0.6
    )

    ax.tick_params(
        axis="both",
        labelsize=8,
        length=2.4,
        pad=3
    )

# ---------------------------------------------------------------
# 5. Draw ground truth and model probabilities.
# ---------------------------------------------------------------

for row_number, selection in selected6.iterrows():

    episode_id6 = str(selection["Episode_ID"])
    family6 = str(selection["Family"])
    source6 = str(selection["Source"])

    mask6 = (
        REF_INDEX5.get_level_values("Episode_ID")
        == episode_id6
    )

    minutes6 = (
        REF_INDEX5
        .get_level_values("Minute")
        .to_numpy()[mask6]
    )

    sorting6 = np.argsort(minutes6)
    minutes6 = minutes6[sorting6]

    truth_row6 = {
        h: TRUTH5[h][mask6][sorting6]
        for h in HORIZONS6
    }

    if len(minutes6) < 30:
        raise RuntimeError(
            f"{episode_id6}: insufficient windows."
        )

    # -----------------------------------------------------------
    # Column 1: actual horizon target sequences.
    # -----------------------------------------------------------

    ax_truth6 = axes6[row_number, 0]

    format_panel6(
        ax_truth6,
        minutes6,
        truth_row6[10]
    )

    for h in HORIZONS6:

        ax_truth6.step(
            minutes6,
            truth_row6[h],
            where="post",
            color=HORIZON_COLORS6[h],
            linestyle=HORIZON_STYLES6[h],
            linewidth=HORIZON_WIDTHS6[h],
            alpha=0.95,
            zorder=4
        )

    ax_truth6.set_ylabel(
        "Target / risk",
        fontsize=9,
        labelpad=8
    )

    family_display6 = FAMILY_LABELS6.get(
        family6, family6
    )

    source_display6 = SOURCE_LABELS6.get(
        source6, source6
    )

    ax_truth6.annotate(
        f"{family_display6}\n"
        f"{source_display6}\n"
        f"n = {len(minutes6)}",
        xy=(0, 0.5),
        xycoords="axes fraction",
        xytext=(-0.32, 0.5),
        textcoords="axes fraction",
        ha="right",
        va="center",
        fontsize=9,
        fontweight="medium",
        color="#344054",
        annotation_clip=False
    )

    # -----------------------------------------------------------
    # Remaining columns: all available model horizons.
    # -----------------------------------------------------------

    for column_number, model_name6 in enumerate(
        MODELS6,
        start=1
    ):

        ax6 = axes6[
            row_number,
            column_number
        ]

        format_panel6(
            ax6,
            minutes6,
            truth_row6[10]
        )

        available_horizons6 = MODEL_PROB5[
            model_name6
        ]

        for h in HORIZONS6:

            if h not in available_horizons6:
                continue

            probabilities6 = (
                available_horizons6[h]
                [mask6]
                [sorting6]
            )

            if not np.isfinite(probabilities6).all():
                raise RuntimeError(
                    f"Invalid probability in {model_name6}"
                )

            ax6.plot(
                minutes6,
                probabilities6,
                color=HORIZON_COLORS6[h],
                linestyle=HORIZON_STYLES6[h],
                linewidth=HORIZON_WIDTHS6[h],
                alpha=0.93,
                zorder=3 + (h == 10)
            )

        if 15 not in available_horizons6:

            ax6.text(
                0.04,
                0.05,
                "H15 N/A",
                transform=ax6.transAxes,
                fontsize=8,
                color="#9C5F3F",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "#E7DFD9",
                    "boxstyle": "round,pad=0.3"
                }
            )

    # Column headings.
    if row_number == 0:

        headings6 = [
            "Ground truth",
            *MODELS6
        ]

        for c, heading in enumerate(headings6):

            axes6[row_number, c].set_title(
                heading,
                fontsize=11,
                fontweight="bold",
                pad=13
            )

    # Only final row needs time-axis labels.
    if row_number == n_rows6 - 1:

        for ax6 in axes6[row_number]:

            ax6.set_xlabel(
                "Decision minute",
                fontsize=9,
                labelpad=7
            )

# ---------------------------------------------------------------
# 6. Shared legend and scientific caption.
# ---------------------------------------------------------------

legend_handles6 = []

for h in HORIZONS6:

    legend_handles6.append(
        Line2D(
            [0], [0],
            color=HORIZON_COLORS6[h],
            linestyle=HORIZON_STYLES6[h],
            linewidth=2.1,
            label=f"H{h}: event within {h} min"
        )
    )

legend_handles6.extend([
    Patch(
        facecolor="#E4B75C",
        alpha=0.30,
        label="H10 ground-truth positive"
    ),
    Line2D(
        [0], [0],
        color="#7B8490",
        linestyle=(0, (2, 3)),
        linewidth=1,
        label="Illustrative threshold = 0.5"
    )
])

fig6.legend(
    handles=legend_handles6,
    loc="upper center",
    bbox_to_anchor=(0.55, 0.933),
    ncol=5,
    frameon=False,
    fontsize=10,
    handlelength=3.2,
    columnspacing=2.5
)

fig6.text(
    0.12,
    0.028,
    "Synthetic NAF v2 development validation. "
    "Rows are selected using ground-truth family metadata, "
    "not model performance. Gold shading marks positive "
    "H10 targets; it does not identify exact attack onset. "
    "XGBoost V2 has no H15 prediction.",
    fontsize=9.2,
    color="#56616B"
)

fig6.subplots_adjust(
    left=0.145,
    right=0.988,
    top=0.855,
    bottom=0.09,
    hspace=0.34,
    wspace=0.16
)

# ---------------------------------------------------------------
# 7. Save into the EXISTING Research_Outputs folders.
# ---------------------------------------------------------------

run6 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem6 = (
    f"F02_Cross_Family_Multi_Horizon_Atlas_{run6}"
)

saved_figures6 = []

for ext in ["pdf", "svg", "png"]:

    out_path6 = FIGURES6 / f"{stem6}.{ext}"

    if out_path6.exists():
        raise FileExistsError(out_path6)

    fig6.savefig(
        out_path6,
        dpi=360 if ext == "png" else 150,
        bbox_inches="tight",
        facecolor="white"
    )

    saved_figures6.append(out_path6)

plt.show()
plt.close(fig6)

# ---------------------------------------------------------------
# 8. Export exact selected episodes as a research table.
# ---------------------------------------------------------------

table_path6 = TABLES6 / (
    f"T02_Attack_Family_Episode_Selection_{run6}.csv"
)

selected6.to_csv(
    table_path6,
    index=False,
    mode="x"
)

# ---------------------------------------------------------------
# 9. Save provenance of model predictions and outputs.
# ---------------------------------------------------------------

def sha256_6(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest6 = {
    "figure": stem6,
    "experiment": "Synthetic_NAF_v2",
    "split": "development_validation",
    "models": MODELS6,
    "horizons": HORIZONS6,
    "selection": (
        "One lexicographically first eligible episode per "
        "attack family (minimum 30 windows, at least one "
        "H10-positive target). Preferred source used when "
        "available. No prediction scores used for selection."
    ),
    "episodes": selected6.to_dict("records"),
    "alignment": {
        "keys": ["Episode_ID", "Minute"],
        "validated_records": len(REF_INDEX5),
        "matching_labels": True,
        "verified_by": "Cell 5"
    },
    "source_files": {
        name: {
            "path": str(path.relative_to(PROJECT6)),
            "sha256": sha256_6(path)
        }
        for name, path in SOURCES5.items()
    },
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT6)),
            "sha256": sha256_6(p)
        }
        for p in saved_figures6 + [table_path6]
    ],
    "limitations": [
        "Validation uses synthetic forecasting windows.",
        "Family metadata is used only for figure selection.",
        "H10 ground-truth spans are not exact attack onsets.",
        "The 0.5 line is illustrative, not a tuned threshold.",
        "Episode examples do not establish aggregate performance.",
        "XGBoost V2 lacks H15 predictions.",
        "No official TEST data accessed."
    ]
}

manifest_path6 = MANIFESTS6 / (
    f"cross_family_atlas_manifest_{run6}.json"
)

with manifest_path6.open(
    "x", encoding="utf-8"
) as file:

    json.dump(
        manifest6,
        file,
        indent=2,
        default=str
    )

# ---------------------------------------------------------------
# 10. Final verification.
# ---------------------------------------------------------------

print("\n" + "=" * 95)
print("CELL 6 COMPLETE")
print("=" * 95)

print("Attack families:", len(selected6))
print("Forecasting models:", len(MODELS6))
print("Subplots:", n_rows6 * n_cols6)
print("Horizons:", HORIZONS6)

print("\nSaved outputs:")

for path in saved_figures6 + [
    table_path6,
    manifest_path6
]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(" -", path.relative_to(PROJECT6))

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Existing model checkpoints modified: NO")
print("Existing research outputs overwritten: NO")

print("=" * 95)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 7 — PAIRED STATISTICAL COMPARISON ATLAS
#
# Figure F03:
#   A. Future-event AP differences (H5/H10/H15)
#   B. Attack-family accuracy and macro F1
#   C. Joint negative log-likelihood
#   D. Conditional ETA MAE
#   E. Source-specific EXPLOIT_APP recall
#
# Source: saved paired bootstrap validation reports.
#
# Exports:
#   01_Figures/Model_Comparisons/ (PDF, SVG, PNG)
#   02_Tables/ (CSV)
#   05_Manifests/ (JSON)
#
# NO retraining
# NO official TEST access
# NO existing source or checkpoint modifications
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

print("=" * 90)
print("CELL 7 — PAIRED STATISTICAL COMPARISON ATLAS")
print("=" * 90)

# ---------------------------------------------------------------
# 1. Existing Google Drive directories
# ---------------------------------------------------------------

PROJECT7 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

AUDIT7 = (
    PROJECT7
    / "Synthetic_NAF_v2"
    / "models"
    / "FINAL_VALIDATION_AUDIT"
)

OUTPUT7 = PROJECT7 / "Research_Outputs"

FIGURES7 = (
    OUTPUT7 / "01_Figures" / "Model_Comparisons"
)

TABLES7 = OUTPUT7 / "02_Tables"
MANIFESTS7 = OUTPUT7 / "05_Manifests"

for directory in [
    AUDIT7, FIGURES7, TABLES7, MANIFESTS7
]:
    if not directory.is_dir():
        raise FileNotFoundError(directory)

MAIN_CI7 = (
    AUDIT7 / "PAIRED_COMPETING_RISK_BOOTSTRAP.csv"
)

SOURCE_CI7 = (
    AUDIT7 / "EXPLOIT_APP_PAIRED_SOURCE_CI.csv"
)

for path in [MAIN_CI7, SOURCE_CI7]:
    if not path.is_file():
        raise FileNotFoundError(path)

# ---------------------------------------------------------------
# 2. Read actual saved confidence intervals
# ---------------------------------------------------------------

main7 = pd.read_csv(MAIN_CI7)
source7 = pd.read_csv(SOURCE_CI7)

required_main7 = {
    "Metric", "TCN", "LSTM", "TCN_Minus_LSTM",
    "CI_Lower_95", "CI_Upper_95",
    "Bootstrap_Replicates"
}

required_source7 = {
    "Source", "TCN_Recall", "LSTM_Recall",
    "TCN_Minus_LSTM", "CI_Lower_95",
    "CI_Upper_95", "True_Exploit_Episodes"
}

if not required_main7.issubset(main7.columns):
    raise RuntimeError("Main bootstrap schema mismatch.")

if not required_source7.issubset(source7.columns):
    raise RuntimeError("Source bootstrap schema mismatch.")

if main7["Metric"].duplicated().any():
    raise RuntimeError("Duplicate metric in bootstrap report.")

if source7["Source"].duplicated().any():
    raise RuntimeError("Duplicate source in bootstrap report.")

numeric_main7 = [
    "TCN", "LSTM", "TCN_Minus_LSTM",
    "CI_Lower_95", "CI_Upper_95",
    "Bootstrap_Replicates"
]

numeric_source7 = [
    "TCN_Recall", "LSTM_Recall",
    "TCN_Minus_LSTM", "CI_Lower_95",
    "CI_Upper_95", "True_Exploit_Episodes"
]

for col in numeric_main7:
    main7[col] = pd.to_numeric(
        main7[col], errors="raise"
    )

for col in numeric_source7:
    source7[col] = pd.to_numeric(
        source7[col], errors="raise"
    )

# Validate reported differences and confidence intervals.

for df, left, right in [
    (main7, "TCN", "LSTM"),
    (source7, "TCN_Recall", "LSTM_Recall")
]:

    observed_delta = df[left] - df[right]

    np.testing.assert_allclose(
        observed_delta.to_numpy(),
        df["TCN_Minus_LSTM"].to_numpy(),
        rtol=1e-7,
        atol=1e-9
    )

    if not (
        df["CI_Lower_95"] <= df["CI_Upper_95"]
    ).all():
        raise RuntimeError("Invalid confidence interval.")

    if not np.isfinite(
        df[
            ["TCN_Minus_LSTM",
             "CI_Lower_95",
             "CI_Upper_95"]
        ].to_numpy()
    ).all():
        raise RuntimeError("Nonfinite bootstrap result.")

EXPECTED_METRICS7 = [
    "H5_AP",
    "H10_AP",
    "H15_AP",
    "Family_Accuracy",
    "Family_Macro_F1",
    "Joint_NLL",
    "ETA_MAE"
]

if set(main7["Metric"]) != set(EXPECTED_METRICS7):
    raise RuntimeError(
        "Unexpected metric set. Review saved report."
    )

if set(source7["Source"]) != {
    "CIC_IDS2017", "UNSW_NB15"
}:
    raise RuntimeError("Unexpected source labels.")

bootstrap_n7 = sorted(
    main7["Bootstrap_Replicates"].unique()
)

print("\nA. SOURCE VALIDATION")
print("-" * 90)
print("Main metric comparisons:", len(main7))
print("Source-specific comparisons:", len(source7))
print("Bootstrap replicates:", bootstrap_n7)
print("Consistency checks: PASS")

# ---------------------------------------------------------------
# 3. Scientific plotting style
# ---------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

COLORS7 = {
    "overlap": "#53738F",
    "excludes_zero": "#A24B50",
    "reference": "#6D7683",
    "grid": "#D7DDE4"
}

def forest_panel7(
    ax,
    data,
    labels,
    title,
    metric_is_lower_better=False,
    xlabel="TCN − LSTM"
):
    """
    Forest plot with asymmetric bootstrap intervals.

    Interpretation:
      Positive = larger TCN metric
      Negative = smaller TCN metric

    For errors/losses, lower values are preferable.
    """

    data = data.reset_index(drop=True).copy()

    if len(data) != len(labels):
        raise RuntimeError(
            "Forest plot label count mismatch."
        )

    n = len(data)

    deltas = data["TCN_Minus_LSTM"].to_numpy()
    lows = data["CI_Lower_95"].to_numpy()
    highs = data["CI_Upper_95"].to_numpy()

    includes_zero = (
        (lows <= 0) & (highs >= 0)
    )

    y = np.arange(n)[::-1]

    ax.axvline(
        0,
        color=COLORS7["reference"],
        lw=1.2,
        ls=(0, (4, 3)),
        zorder=1
    )

    for i in range(n):

        color = (
            COLORS7["overlap"]
            if includes_zero[i]
            else COLORS7["excludes_zero"]
        )

        ax.plot(
            [lows[i], highs[i]],
            [y[i], y[i]],
            color=color,
            lw=2.8,
            solid_capstyle="round",
            zorder=3
        )

        ax.plot(
            [lows[i], lows[i]],
            [y[i]-0.09, y[i]+0.09],
            color=color,
            lw=1.3
        )

        ax.plot(
            [highs[i], highs[i]],
            [y[i]-0.09, y[i]+0.09],
            color=color,
            lw=1.3
        )

        ax.scatter(
            deltas[i],
            y[i],
            s=70,
            color=color,
            edgecolors="white",
            linewidth=1.1,
            zorder=4
        )

    max_abs = max(
        np.max(np.abs(lows)),
        np.max(np.abs(highs)),
        np.max(np.abs(deltas))
    )

    # Symmetric axes make the zero reference unambiguous.
    limit = max(float(max_abs) * 1.22, 0.003)

    ax.set_xlim(-limit, limit)
    ax.set_ylim(-0.6, n-0.4)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)

    ax.set_title(
        title,
        loc="left",
        fontweight="bold",
        pad=15
    )

    ax.set_xlabel(xlabel, labelpad=10)

    ax.grid(
        axis="x",
        linestyle=":",
        color=COLORS7["grid"],
        linewidth=0.8
    )

    ax.set_axisbelow(True)

    ax.tick_params(
        axis="y", length=0, pad=9
    )

    ax.tick_params(
        axis="x", length=3, pad=6
    )

    ax.set_facecolor("#FAFBFC")

    preference = (
        "Negative values favor TCN"
        if metric_is_lower_better
        else "Positive values favor TCN"
    )

    ax.text(
        0.01,
        0.02,
        preference,
        transform=ax.transAxes,
        fontsize=8,
        color="#66717E",
        va="bottom"
    )

    return includes_zero

def select_main7(names):
    lookup = main7.set_index("Metric")
    return lookup.loc[names].reset_index()

# ---------------------------------------------------------------
# 4. Assemble the five-panel figure
# ---------------------------------------------------------------

fig7 = plt.figure(
    figsize=(16, 13),
    facecolor="white"
)

gs7 = GridSpec(
    3, 2,
    figure=fig7,
    height_ratios=[1.18, 1, 1],
    hspace=0.58,
    wspace=0.40,
    left=0.15,
    right=0.95,
    top=0.85,
    bottom=0.12
)

ax_a7 = fig7.add_subplot(gs7[0, :])
ax_b7 = fig7.add_subplot(gs7[1, 0])
ax_c7 = fig7.add_subplot(gs7[1, 1])
ax_d7 = fig7.add_subplot(gs7[2, 0])
ax_e7 = fig7.add_subplot(gs7[2, 1])

fig7.suptitle(
    "Paired validation comparison of competing-risk "
    "forecasting models",
    x=0.12,
    y=0.975,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig7.text(
    0.12,
    0.937,
    "Observed differences and 95% bootstrap intervals  |  "
    "Difference = TCN minus LSTM",
    fontsize=11,
    color="#586575"
)

forest_panel7(
    ax_a7,
    select_main7(["H5_AP", "H10_AP", "H15_AP"]),
    ["H5 average precision",
     "H10 average precision",
     "H15 average precision"],
    "A   Future-attack discrimination",
    xlabel="Difference in average precision"
)

forest_panel7(
    ax_b7,
    select_main7([
        "Family_Accuracy",
        "Family_Macro_F1"
    ]),
    ["Family accuracy",
     "Family macro F1"],
    "B   Future attack-family classification",
    xlabel="Difference in classification score"
)

forest_panel7(
    ax_c7,
    select_main7(["Joint_NLL"]),
    ["Joint NLL"],
    "C   Joint prediction loss",
    metric_is_lower_better=True,
    xlabel="Difference in negative log-likelihood"
)

forest_panel7(
    ax_d7,
    select_main7(["ETA_MAE"]),
    ["ETA MAE"],
    "D   Conditional time-to-attack estimation",
    metric_is_lower_better=True,
    xlabel="Difference in ETA MAE (minutes)"
)

source_plot7 = (
    source7
    .set_index("Source")
    .loc[["CIC_IDS2017", "UNSW_NB15"]]
    .reset_index()
)

forest_panel7(
    ax_e7,
    source_plot7,
    ["CIC-IDS2017 derived",
     "UNSW-NB15 derived"],
    "E   Exploit/App attack-family recall",
    xlabel="Difference in Exploit/App recall"
)

legend7 = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="-",
        color=COLORS7["overlap"],
        markersize=7,
        linewidth=2.3,
        label="95% interval includes zero"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="-",
        color=COLORS7["excludes_zero"],
        markersize=7,
        linewidth=2.3,
        label="95% interval excludes zero"
    ),
    Line2D(
        [0], [0],
        linestyle=(0, (4, 3)),
        color=COLORS7["reference"],
        linewidth=1.2,
        label="Zero difference"
    )
]

fig7.legend(
    handles=legend7,
    loc="upper left",
    bbox_to_anchor=(0.12, 0.915),
    ncol=3,
    frameon=False,
    fontsize=10,
    handlelength=2.6
)

fig7.text(
    0.12,
    0.055,
    "Synthetic NAF v2 validation. Main comparisons use "
    "500 paired-bootstrap replicates. The Exploit/App "
    "panel uses source-specific paired intervals. "
    "Interval exclusion of zero is descriptive and "
    "depends on the reported bootstrap procedure.",
    fontsize=9,
    color="#5B6775",
    wrap=True
)

fig7.text(
    0.12,
    0.032,
    "Conditional family and ETA scores are evaluated "
    "given a true future event; these do not measure "
    "end-to-end attack detection accuracy.",
    fontsize=9,
    color="#5B6775"
)

# ---------------------------------------------------------------
# 5. Save PDF, SVG and high-resolution PNG
# ---------------------------------------------------------------

stamp7 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem7 = (
    f"F03_Paired_Statistical_Comparison_{stamp7}"
)

figure_files7 = []

for extension in ["pdf", "svg", "png"]:

    output_path = (
        FIGURES7 / f"{stem7}.{extension}"
    )

    if output_path.exists():
        raise FileExistsError(output_path)

    fig7.savefig(
        output_path,
        dpi=500 if extension == "png" else 200,
        bbox_inches="tight",
        facecolor="white"
    )

    figure_files7.append(output_path)

plt.show()
plt.close(fig7)

# ---------------------------------------------------------------
# 6. Research table
# ---------------------------------------------------------------

main_table7 = main7[
    [
        "Metric", "LSTM", "TCN", "TCN_Minus_LSTM",
        "CI_Lower_95", "CI_Upper_95",
        "Bootstrap_Replicates"
    ]
].copy()

main_table7.insert(
    0, "Analysis", "Overall paired validation"
)

source_table7 = source7[
    [
        "Source",
        "LSTM_Recall",
        "TCN_Recall",
        "TCN_Minus_LSTM",
        "CI_Lower_95",
        "CI_Upper_95"
    ]
].copy()

source_table7 = source_table7.rename(
    columns={
        "LSTM_Recall": "LSTM",
        "TCN_Recall": "TCN"
    }
)

source_table7.insert(
    0, "Analysis", "Source-specific Exploit/App"
)

source_table7["Metric"] = (
    "EXPLOIT_APP_Recall_"
    + source_table7["Source"]
)

source_table7["Bootstrap_Replicates"] = np.nan

combined_table7 = pd.concat(
    [
        main_table7,
        source_table7[
            main_table7.columns
        ]
    ],
    ignore_index=True
)

combined_table7["CI_Excludes_Zero"] = (
    (combined_table7["CI_Lower_95"] > 0)
    | (combined_table7["CI_Upper_95"] < 0)
)

table_path7 = TABLES7 / (
    f"T03_Paired_Statistical_Comparison_{stamp7}.csv"
)

combined_table7.to_csv(
    table_path7,
    index=False,
    mode="x",
    float_format="%.8f"
)

# ---------------------------------------------------------------
# 7. Provenance manifest
# ---------------------------------------------------------------

def hash_file7(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest7 = {
    "figure": stem7,
    "dataset": "Synthetic_NAF_v2",
    "comparison": "Competing-Risk TCN minus Competing-Risk LSTM",
    "difference_convention": "TCN - LSTM",
    "main_bootstrap_replicates": bootstrap_n7,
    "official_test_used": False,
    "models_trained": 0,
    "sources": [
        {
            "path": str(p.relative_to(PROJECT7)),
            "sha256": hash_file7(p)
        }
        for p in [MAIN_CI7, SOURCE_CI7]
    ],
    "generated_outputs": [
        {
            "path": str(p.relative_to(PROJECT7)),
            "sha256": hash_file7(p)
        }
        for p in figure_files7 + [table_path7]
    ],
    "limitations": [
        "Bootstrap results were read from saved validation reports.",
        "Uncertainty depends on original resampling assumptions.",
        "Window overlap may create additional dependence.",
        "Conditional family and ETA metrics assume a true future event.",
        "Observed intervals do not establish deployment performance.",
        "No official TEST data was accessed."
    ]
}

manifest_path7 = MANIFESTS7 / (
    f"statistical_comparison_manifest_{stamp7}.json"
)

with manifest_path7.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest7,
        handle,
        indent=2
    )

# ---------------------------------------------------------------
# 8. Summary and verification
# ---------------------------------------------------------------

print("\n" + "=" * 90)
print("B. VERIFIED OUTPUTS")
print("=" * 90)

for path in figure_files7 + [
    table_path7,
    manifest_path7
]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(" -", path.relative_to(OUTPUT7))

print("\nINTERVALS EXCLUDING ZERO")
print("-" * 90)

for _, row in combined_table7.loc[
    combined_table7["CI_Excludes_Zero"]
].iterrows():

    print(
        f"{row['Metric']}: "
        f"{row['TCN_Minus_LSTM']:+.4f} "
        f"[{row['CI_Lower_95']:+.4f}, "
        f"{row['CI_Upper_95']:+.4f}]"
    )

print("\n" + "=" * 90)
print("CELL 7 COMPLETE")
print("=" * 90)

print("Panels generated: 5")
print("Figure formats: PDF, SVG, PNG")
print("Research tables: 1")
print("Provenance manifests: 1")

print("\nModels trained: 0")
print("Official TEST accessed: NO")
print("Existing saved models modified: NO")
print("Original reports overwritten: NO")

print("=" * 90)


In [ ]:

# ============================================================
# NETGUARD — CELL 7B
# FIX INCOMPLETE STATISTICAL COMPARISON MANIFEST
#
# No model training
# No figure regeneration
# No source file modifications
# Repairs ONLY Cell 7's incomplete JSON manifest
# ============================================================

import json
import numpy as np
from pathlib import Path

print("=" * 75)
print("CELL 7B — REPAIR STATISTICAL MANIFEST")
print("=" * 75)

required = [
    "manifest7",
    "manifest_path7",
    "figure_files7",
    "table_path7"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Cell 7 variables are not available."
        )

# Convert NumPy scalar types into JSON-safe Python types.
def json_safe7(value):

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(k): json_safe7(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe7(v) for v in value]

    return value


manifest_path7 = Path(manifest_path7)

# Verify the previously generated outputs exist.
for path in list(figure_files7) + [table_path7]:

    path = Path(path)

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Missing or empty research output: {path}"
        )

# Serialize fully IN MEMORY before modifying anything.
fixed_manifest7 = json_safe7(manifest7)

serialized7 = json.dumps(
    fixed_manifest7,
    indent=2,
    ensure_ascii=False,
    allow_nan=False
)

# Only repair the manifest if it is missing or invalid.
if manifest_path7.exists():

    try:
        with manifest_path7.open(
            "r", encoding="utf-8"
        ) as handle:
            existing = json.load(handle)

        print("\nManifest already valid. No overwrite performed.")

    except (json.JSONDecodeError, UnicodeDecodeError):

        with manifest_path7.open(
            "w", encoding="utf-8"
        ) as handle:
            handle.write(serialized7)

        print("\nIncomplete manifest repaired.")

else:

    with manifest_path7.open(
        "x", encoding="utf-8"
    ) as handle:
        handle.write(serialized7)

    print("\nMissing manifest created.")

# Reload to verify valid JSON.
with manifest_path7.open(
    "r", encoding="utf-8"
) as handle:

    verified_manifest7 = json.load(handle)

assert verified_manifest7["figure"] == manifest7["figure"]

print("\nMANIFEST VALIDATION: PASS")
print("Manifest:", manifest_path7)

print("\nEXISTING RESEARCH OUTPUTS:")

for path in list(figure_files7) + [table_path7]:
    print("PASS:", Path(path).name)

print("\n" + "=" * 75)
print("CELL 7B COMPLETE")
print("=" * 75)

print("Models retrained: 0")
print("Graphs regenerated: 0")
print("Existing source reports modified: NO")
print("Research manifest valid: YES")
print("=" * 75)


In [ ]:

# ================================================================
# NETGUARD RESEARCH OUTPUTS
# CELL 8 — ATTACK-FAMILY CONFUSION & GENERALIZATION
#
# F04: Four-panel source-specific confusion atlas
# F05: Family recall and model-difference atlas
#
# Data: Saved competing-risk LSTM / TCN validation reports
#
# Scientific safeguards:
#   - Validate confusion-matrix shapes
#   - Verify per-class support
#   - Verify recall against source-family reports
#   - Mark absent classes as N/A
#   - No model training
#   - No official TEST access
#   - No modification of existing artifacts
#
# OUTPUT:
#   01_Figures/Forecasting/
#   02_Tables/
#   05_Manifests/
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

print("=" * 95)
print("CELL 8 — ATTACK-FAMILY CONFUSION & GENERALIZATION")
print("=" * 95)

# ------------------------------------------------------------
# 1. EXISTING RESEARCH DIRECTORIES
# ------------------------------------------------------------

PROJECT8 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

AUDIT8 = (
    PROJECT8
    / "Synthetic_NAF_v2"
    / "models"
    / "FINAL_VALIDATION_AUDIT"
)

OUTPUT8 = PROJECT8 / "Research_Outputs"

FIGURES8 = OUTPUT8 / "01_Figures" / "Forecasting"
TABLES8 = OUTPUT8 / "02_Tables"
MANIFESTS8 = OUTPUT8 / "05_Manifests"

for folder in [
    AUDIT8, FIGURES8, TABLES8, MANIFESTS8
]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

# ------------------------------------------------------------
# 2. EXPLICIT, VERIFIED VALIDATION FILES
# ------------------------------------------------------------

MATRIX_PATHS8 = {
    ("LSTM", "CIC_IDS2017"):
        AUDIT8 / "CONFUSION_LSTM_CIC_IDS2017.csv",

    ("TCN", "CIC_IDS2017"):
        AUDIT8 / "CONFUSION_TCN_CIC_IDS2017.csv",

    ("LSTM", "UNSW_NB15"):
        AUDIT8 / "CONFUSION_LSTM_UNSW_NB15.csv",

    ("TCN", "UNSW_NB15"):
        AUDIT8 / "CONFUSION_TCN_UNSW_NB15.csv"
}

FAMILY_REPORT8 = (
    AUDIT8 / "SOURCE_FAMILY_PER_CLASS.csv"
)

ALL_SOURCES8 = (
    list(MATRIX_PATHS8.values())
    + [FAMILY_REPORT8]
)

for path in ALL_SOURCES8:
    if not path.is_file():
        raise FileNotFoundError(path)

FAMILIES8 = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

PRETTY8 = {
    "BRUTE_AUTH": "Brute/Auth",
    "DOS_DDOS": "DoS/DDoS",
    "EXPLOIT_APP": "Exploit/App",
    "FUZZING": "Fuzzing",
    "RECON_SCAN": "Recon/Scan"
}

SOURCE_LABEL8 = {
    "CIC_IDS2017": "CIC-IDS2017 derived",
    "UNSW_NB15": "UNSW-NB15 derived"
}

MODEL_LABEL8 = {
    "LSTM": "Competing-Risk LSTM",
    "TCN": "Competing-Risk TCN"
}

# ------------------------------------------------------------
# 3. LOAD AND VALIDATE CLASS-SPECIFIC METRICS
# ------------------------------------------------------------

family_report8 = pd.read_csv(FAMILY_REPORT8)

needed8 = {
    "Model", "Source", "Family",
    "Support", "Precision", "Recall", "F1"
}

if not needed8.issubset(family_report8.columns):
    raise RuntimeError(
        "Unexpected SOURCE_FAMILY_PER_CLASS schema."
    )

if family_report8.duplicated(
    ["Model", "Source", "Family"]
).any():
    raise RuntimeError("Duplicate source/family rows.")

if len(family_report8) != 20:
    raise RuntimeError(
        f"Expected 20 family rows, got {len(family_report8)}."
    )

for col in ["Support", "Precision", "Recall", "F1"]:
    family_report8[col] = pd.to_numeric(
        family_report8[col],
        errors="raise"
    )

if not (
    family_report8["Support"] >= 0
).all():
    raise RuntimeError("Negative family support.")

# ------------------------------------------------------------
# 4. LOAD ALL FOUR CONFUSION MATRICES
# ------------------------------------------------------------

MATRICES8 = {}
NORMALIZED8 = {}
SUPPORT8 = {}
RECALL8 = {}
VALIDATION8 = []

print("\nA. CONFUSION MATRIX VALIDATION")
print("-" * 95)

for (model, source), path in MATRIX_PATHS8.items():

    frame = pd.read_csv(
        path,
        index_col=0
    )

    expected_rows = [
        f"True_{family}"
        for family in FAMILIES8
    ]

    expected_columns = [
        f"Pred_{family}"
        for family in FAMILIES8
    ]

    if set(frame.index) != set(expected_rows):
        raise RuntimeError(
            f"Unexpected true-label schema: {path.name}"
        )

    if set(frame.columns) != set(expected_columns):
        raise RuntimeError(
            f"Unexpected prediction schema: {path.name}"
        )

    frame = frame.loc[
        expected_rows,
        expected_columns
    ]

    counts = frame.apply(
        pd.to_numeric,
        errors="raise"
    ).to_numpy(dtype=np.float64)

    if counts.shape != (5, 5):
        raise RuntimeError("Unexpected matrix shape.")

    if not np.isfinite(counts).all():
        raise RuntimeError("Invalid matrix values.")

    if (counts < 0).any():
        raise RuntimeError("Negative confusion counts.")

    if not np.allclose(
        counts,
        np.round(counts),
        atol=0
    ):
        raise RuntimeError(
            "Confusion counts are not integers."
        )

    counts = counts.astype(np.int64)

    row_support = counts.sum(axis=1)

    normalized = np.divide(
        counts.astype(float),
        row_support[:, None],
        out=np.full(
            counts.shape,
            np.nan,
            dtype=float
        ),
        where=(row_support[:, None] > 0)
    )

    diagonal_recall = np.diag(normalized)

    subset = family_report8.loc[
        (family_report8["Model"] == MODEL_LABEL8[model])
        & (family_report8["Source"] == source)
    ].set_index("Family")

    if set(subset.index) != set(FAMILIES8):
        raise RuntimeError(
            f"Missing family report rows: {model}/{source}"
        )

    subset = subset.loc[FAMILIES8]

    np.testing.assert_array_equal(
        row_support,
        subset["Support"].to_numpy(dtype=np.int64),
        err_msg=(
            f"Support mismatch for {model}/{source}"
        )
    )

    supported = row_support > 0

    np.testing.assert_allclose(
        diagonal_recall[supported],
        subset["Recall"].to_numpy()[supported],
        atol=1e-9,
        rtol=1e-8,
        err_msg=(
            f"Recall mismatch for {model}/{source}"
        )
    )

    MATRICES8[(model, source)] = counts
    NORMALIZED8[(model, source)] = normalized
    SUPPORT8[(model, source)] = row_support
    RECALL8[(model, source)] = diagonal_recall

    VALIDATION8.append({
        "model": MODEL_LABEL8[model],
        "source": source,
        "total_event_windows": int(counts.sum()),
        "absent_class_count": int((row_support == 0).sum()),
        "support_verified": True,
        "recall_verified": True
    })

    print(
        f"PASS | {model:<5} | {source:<12} "
        f"| windows={counts.sum():>5} "
        f"| zero-support classes={(row_support == 0).sum()}"
    )

print("\nAll support and recall checks passed.")

# ------------------------------------------------------------
# 5. PUBLICATION STYLE
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.titleweight": "semibold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

CMAP8 = plt.get_cmap("YlGnBu").copy()
CMAP8.set_bad("#E8EBEF")

MODEL_COLORS8 = {
    "LSTM": "#167A83",
    "TCN": "#925AA4"
}

STAMP8 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

CREATED8 = []

def save_figure8(figure, stem, destination):

    output_files = []

    for extension in ["pdf", "svg", "png"]:

        path = destination / f"{stem}_{STAMP8}.{extension}"

        if path.exists():
            raise FileExistsError(path)

        figure.savefig(
            path,
            dpi=420 if extension == "png" else 160,
            bbox_inches="tight",
            facecolor="white"
        )

        output_files.append(path)

    return output_files

# ============================================================
# FIGURE F04 — FOUR CONFUSION MATRICES
# ============================================================

print("\nB. GENERATING F04 — CONFUSION ATLAS")
print("-" * 95)

fig8a, axes8a = plt.subplots(
    2, 2,
    figsize=(16, 12),
    sharex=True,
    sharey=True,
    gridspec_kw={
        "hspace": 0.24,
        "wspace": 0.22
    }
)

fig8a.suptitle(
    "Attack-family confusion across forecasting models and sources",
    x=0.11,
    y=0.987,
    ha="left",
    fontsize=17,
    fontweight="bold"
)

fig8a.text(
    0.11,
    0.953,
    "Row-normalized confusion matrices  |  "
    "Each cell shows recall share and raw event-window count",
    fontsize=10,
    color="#596778"
)

plot_order8 = [
    ("LSTM", "CIC_IDS2017"),
    ("TCN", "CIC_IDS2017"),
    ("LSTM", "UNSW_NB15"),
    ("TCN", "UNSW_NB15")
]

panel_letters8 = ["A", "B", "C", "D"]

for ax, (model, source), letter in zip(
    axes8a.flat,
    plot_order8,
    panel_letters8
):

    counts = MATRICES8[(model, source)]
    matrix = NORMALIZED8[(model, source)]
    supports = SUPPORT8[(model, source)]

    masked = np.ma.masked_invalid(matrix)

    image = ax.imshow(
        masked,
        cmap=CMAP8,
        vmin=0,
        vmax=1,
        aspect="equal",
        interpolation="nearest"
    )

    ax.set_title(
        f"{letter}   {MODEL_LABEL8[model]} · "
        f"{SOURCE_LABEL8[source]}",
        loc="left",
        pad=14,
        fontsize=11.5
    )

    ax.set_xticks(range(5))
    ax.set_yticks(range(5))

    ax.set_xticklabels(
        [PRETTY8[f] for f in FAMILIES8],
        rotation=35,
        ha="right"
    )

    ax.set_yticklabels(
        [
            f"{PRETTY8[f]}  (n={supports[i]})"
            for i, f in enumerate(FAMILIES8)
        ]
    )

    ax.set_xticks(
        np.arange(-0.5, 5, 1),
        minor=True
    )
    ax.set_yticks(
        np.arange(-0.5, 5, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=2.0
    )

    ax.tick_params(
        which="both",
        length=0
    )

    for i in range(5):
        for j in range(5):

            if supports[i] == 0:
                label = "N/A"
                text_color = "#647080"

            elif counts[i, j] == 0:
                label = "·"
                text_color = "#637486"

            else:
                pct = matrix[i, j] * 100

                label = (
                    f"{pct:.1f}%\n"
                    f"({counts[i, j]})"
                )

                text_color = (
                    "white"
                    if matrix[i, j] > 0.53
                    else "#15263A"
                )

            ax.text(
                j, i,
                label,
                ha="center",
                va="center",
                fontsize=8.4,
                color=text_color,
                fontweight=(
                    "bold" if i == j else "normal"
                )
            )

        # Outline diagonal only if class has support.
        if supports[i] > 0:

            ax.add_patch(
                Rectangle(
                    (i - 0.48, i - 0.48),
                    0.96,
                    0.96,
                    fill=False,
                    edgecolor="#DA8840",
                    linewidth=1.7
                )
            )

    ax.set_xlabel("Predicted future attack family")

    if source == "CIC_IDS2017":
        ax.set_ylabel("True future attack family")
    else:
        ax.set_ylabel("True future attack family")

fig8a.subplots_adjust(
    left=0.13,
    right=0.90,
    top=0.88,
    bottom=0.10
)

cbar_ax8 = fig8a.add_axes([
    0.925, 0.21,
    0.015, 0.57
])

colorbar8 = fig8a.colorbar(
    ScalarMappable(
        norm=Normalize(0, 1),
        cmap=CMAP8
    ),
    cax=cbar_ax8
)

colorbar8.set_label(
    "Share of true-class windows",
    labelpad=10
)

fig8a.text(
    0.13, 0.027,
    "Synthetic NAF v2 validation. Family classification "
    "is conditional on a true future event. "
    "Grey rows have zero source-specific support; "
    "N/A does not indicate zero recall.",
    fontsize=9,
    color="#5D6978"
)

F04_PATHS8 = save_figure8(
    fig8a,
    "F04_Source_Family_Confusion_Atlas",
    FIGURES8
)

CREATED8.extend(F04_PATHS8)

plt.show()
plt.close(fig8a)

# ============================================================
# FIGURE F05 — SOURCE-SPECIFIC RECALL COMPARISON
# ============================================================

print("\nC. GENERATING F05 — RECALL COMPARISON")
print("-" * 95)

fig8b, axes8b = plt.subplots(
    1, 2,
    figsize=(15, 6.6),
    sharex=True,
    sharey=True,
    gridspec_kw={
        "wspace": 0.30
    }
)

fig8b.suptitle(
    "Attack-family recall and model disagreement by source",
    x=0.07,
    y=0.98,
    ha="left",
    fontsize=16.5,
    fontweight="bold"
)

fig8b.text(
    0.07, 0.915,
    "Paired source-specific family recall  |  "
    "Competing-Risk LSTM versus TCN",
    fontsize=10,
    color="#596778"
)

recall_rows8 = []

for idx, source in enumerate([
    "CIC_IDS2017",
    "UNSW_NB15"
]):

    ax = axes8b[idx]

    ax.set_title(
        f"{'A' if idx == 0 else 'B'}   "
        f"{SOURCE_LABEL8[source]}",
        loc="left",
        pad=16
    )

    ax.set_facecolor("#FAFBFC")
    ax.set_xlim(-0.035, 1.08)
    ax.set_ylim(-0.7, 4.7)

    ax.set_xticks(
        [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    )

    ax.set_yticks(range(5))
    ax.set_yticklabels(
        [PRETTY8[f] for f in FAMILIES8]
    )

    ax.invert_yaxis()

    ax.grid(
        axis="x",
        linestyle=":",
        color="#B6C0CC",
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.set_xlabel(
        "Recall (fraction of true family correctly classified)",
        labelpad=12
    )
    ax.tick_params(
        axis="y",
        length=0,
        pad=8
    )

    for family_idx, family in enumerate(FAMILIES8):

        recall_lstm = RECALL8[
            ("LSTM", source)
        ][family_idx]

        recall_tcn = RECALL8[
            ("TCN", source)
        ][family_idx]

        support = int(
            SUPPORT8[
                ("LSTM", source)
            ][family_idx]
        )

        if support == 0:

            ax.text(
                0.5,
                family_idx,
                "N/A — no true-class examples",
                ha="center",
                va="center",
                fontsize=9,
                color="#75808D"
            )

            continue

        ax.plot(
            [recall_lstm, recall_tcn],
            [family_idx, family_idx],
            color="#AEB8C4",
            lw=2.8,
            zorder=2
        )

        ax.scatter(
            recall_lstm,
            family_idx - 0.095,
            s=75,
            color=MODEL_COLORS8["LSTM"],
            edgecolor="white",
            linewidth=1.0,
            zorder=4
        )

        ax.scatter(
            recall_tcn,
            family_idx + 0.095,
            s=75,
            marker="s",
            color=MODEL_COLORS8["TCN"],
            edgecolor="white",
            linewidth=1.0,
            zorder=4
        )

        difference = (
            recall_tcn - recall_lstm
        )

        recall_rows8.append({
            "Source": source,
            "Family": family,
            "Support": support,
            "LSTM_Recall": float(recall_lstm),
            "TCN_Recall": float(recall_tcn),
            "TCN_Minus_LSTM": float(difference)
        })

        # Explicitly annotate the unusually large
        # UNSW exploitation error.
        if (
            source == "UNSW_NB15"
            and family == "EXPLOIT_APP"
        ):

            ax.text(
                0.38,
                family_idx - 0.20,
                f"Δ = {difference:+.3f}",
                fontsize=9.5,
                fontweight="bold",
                color="#9D484A"
            )

fig8b.legend(
    handles=[
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            markerfacecolor=MODEL_COLORS8["LSTM"],
            markeredgecolor="white",
            markersize=9,
            label="Competing-Risk LSTM"
        ),
        Line2D(
            [0], [0],
            marker="s",
            color="none",
            markerfacecolor=MODEL_COLORS8["TCN"],
            markeredgecolor="white",
            markersize=9,
            label="Competing-Risk TCN"
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.55, 0.895),
    ncol=2,
    frameon=False
)

fig8b.subplots_adjust(
    left=0.13,
    right=0.96,
    top=0.80,
    bottom=0.19
)

fig8b.text(
    0.07, 0.045,
    "Only families present in the source are plotted. "
    "Recall is conditional on a true future event; "
    "the difference plot is descriptive, "
    "not a significance test.",
    fontsize=9,
    color="#5D6978"
)

F05_PATHS8 = save_figure8(
    fig8b,
    "F05_Source_Family_Recall_Comparison",
    FIGURES8
)

CREATED8.extend(F05_PATHS8)

plt.show()
plt.close(fig8b)

# ------------------------------------------------------------
# 6. EXPORT TABLES
# ------------------------------------------------------------

print("\nD. EXPORTING SOURCE-SPECIFIC RESEARCH TABLES")
print("-" * 95)

LONG_ROWS8 = []

for (model, source), counts in MATRICES8.items():

    row_support = SUPPORT8[(model, source)]

    for i, true_family in enumerate(FAMILIES8):
        for j, predicted_family in enumerate(FAMILIES8):

            LONG_ROWS8.append({
                "Model": MODEL_LABEL8[model],
                "Source": source,
                "True_Family": true_family,
                "Predicted_Family": predicted_family,
                "Count": int(counts[i, j]),
                "True_Class_Support": int(row_support[i]),
                "Row_Fraction": (
                    float(counts[i, j] / row_support[i])
                    if row_support[i] > 0
                    else np.nan
                )
            })

confusion_table8 = pd.DataFrame(LONG_ROWS8)

TABLE_F04_8 = (
    TABLES8
    / f"T04_Source_Family_Confusion_{STAMP8}.csv"
)

confusion_table8.to_csv(
    TABLE_F04_8,
    index=False,
    mode="x"
)

recall_table8 = pd.DataFrame(recall_rows8)

TABLE_F05_8 = (
    TABLES8
    / f"T05_Source_Family_Recall_{STAMP8}.csv"
)

recall_table8.to_csv(
    TABLE_F05_8,
    index=False,
    mode="x"
)

CREATED8.extend([
    TABLE_F04_8,
    TABLE_F05_8
])

# ------------------------------------------------------------
# 7. VERIFY THE KEY EXPLOIT/APP FINDING
# ------------------------------------------------------------

exploit_idx8 = FAMILIES8.index("EXPLOIT_APP")
fuzz_idx8 = FAMILIES8.index("FUZZING")

unsw_tcn8 = MATRICES8[
    ("TCN", "UNSW_NB15")
]

unsw_lstm8 = MATRICES8[
    ("LSTM", "UNSW_NB15")
]

print("\nE. EXPLOIT/APP FAILURE DIAGNOSTIC")
print("-" * 95)

print(
    "UNSW true Exploit/App windows:",
    int(unsw_tcn8[exploit_idx8].sum())
)

print(
    "LSTM correctly classified:",
    int(unsw_lstm8[exploit_idx8, exploit_idx8])
)

print(
    "TCN correctly classified:",
    int(unsw_tcn8[exploit_idx8, exploit_idx8])
)

print(
    "TCN misclassified as Fuzzing:",
    int(unsw_tcn8[exploit_idx8, fuzz_idx8])
)

# ------------------------------------------------------------
# 8. HASHES AND PROVENANCE
# ------------------------------------------------------------

def sha256_8(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

MANIFEST8 = (
    MANIFESTS8
    / f"family_confusion_research_manifest_{STAMP8}.json"
)

manifest_data8 = {
    "project": "NetGuard",
    "stage": "Research Outputs — Cell 8",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "experiment": "Synthetic_NAF_v2",
    "evaluation": "saved_validation",

    "models": [
        "Competing-Risk LSTM",
        "Competing-Risk TCN"
    ],

    "sources": [
        "CIC_IDS2017",
        "UNSW_NB15"
    ],

    "checks": VALIDATION8,

    "input_files": [
        {
            "path": str(p.relative_to(PROJECT8)),
            "sha256": sha256_8(p)
        }
        for p in ALL_SOURCES8
    ],

    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT8)),
            "sha256": sha256_8(p)
        }
        for p in CREATED8
    ],

    "limitations": [
        "Forecasting family classification, not current attack detection.",
        "Class predictions evaluated conditional on a true future event.",
        "No source-specific class support means recall is N/A.",
        "Rows normalized by true-class support.",
        "Source-wise differences are descriptive; statistical inference requires an appropriate paired analysis.",
        "All evaluated data are from the synthetic forecasting benchmark.",
        "No official TEST dataset accessed."
    ],

    "official_test_used": False,
    "models_retrained": 0
}

# Convert NumPy scalars if present, BEFORE writing.
def json_safe8(obj):

    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {
            str(k): json_safe8(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            json_safe8(item)
            for item in obj
        ]

    return obj

json_content8 = json.dumps(
    json_safe8(manifest_data8),
    indent=2,
    allow_nan=False
)

with MANIFEST8.open(
    "x", encoding="utf-8"
) as handle:
    handle.write(json_content8)

# ------------------------------------------------------------
# 9. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("CELL 8 COMPLETE")
print("=" * 95)

for path in CREATED8 + [MANIFEST8]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        "PASS:",
        path.relative_to(OUTPUT8)
    )

print("\nConfusion matrices validated: 4")
print("Family-support checks: PASS")
print("Family-recall checks: PASS")

print("Figures generated: 2")
print("Figure files: 6")
print("Research tables generated: 2")
print("Manifest generated: 1")

print("\nModels trained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Source reports modified: NO")
print("Existing research outputs overwritten: NO")

print("=" * 95)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 8B — CORRECT SOURCE-SPECIFIC CONFUSION ATLAS
#
# Fixes incorrect shared y-axis support labels.
# Uses verified Cell 8 matrices.
#
# No retraining.
# No official TEST access.
# No source file modifications.
# No overwriting previous figures.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle

print("=" * 90)
print("CELL 8B — CORRECTED ATTACK-FAMILY CONFUSION ATLAS")
print("=" * 90)

required8b = [
    "MATRICES8",
    "NORMALIZED8",
    "SUPPORT8",
    "FAMILIES8",
    "PRETTY8",
    "SOURCE_LABEL8",
    "MODEL_LABEL8",
    "CMAP8"
]

for name in required8b:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Cell 8 first."
        )

PROJECT8B = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

FIGURES8B = (
    PROJECT8B / "Research_Outputs"
    / "01_Figures" / "Forecasting"
)

MANIFESTS8B = (
    PROJECT8B / "Research_Outputs"
    / "05_Manifests"
)

for folder in [FIGURES8B, MANIFESTS8B]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

# ------------------------------------------------------------
# 1. Verify exact source-specific supports
# ------------------------------------------------------------

expected_support8b = {
    "CIC_IDS2017": [675, 345, 345, 0, 345],
    "UNSW_NB15": [0, 330, 330, 675, 330]
}

plot_order8b = [
    ("LSTM", "CIC_IDS2017"),
    ("TCN", "CIC_IDS2017"),
    ("LSTM", "UNSW_NB15"),
    ("TCN", "UNSW_NB15")
]

for model, source in plot_order8b:

    np.testing.assert_array_equal(
        SUPPORT8[(model, source)],
        expected_support8b[source],
        err_msg=f"Incorrect support: {model}/{source}"
    )

print("\nSource-specific support checks: PASS")

# ------------------------------------------------------------
# 2. Create axes WITHOUT shared tick labels
# ------------------------------------------------------------

fig8b_fix, axes8b_fix = plt.subplots(
    2, 2,
    figsize=(16, 12),
    sharex=False,
    sharey=False
)

fig8b_fix.patch.set_facecolor("white")

fig8b_fix.suptitle(
    "Source-dependent attack-family classification errors",
    x=0.10,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig8b_fix.text(
    0.10,
    0.945,
    "Conditional future attack-family classification | "
    "Row-normalized confusion matrices",
    fontsize=10.5,
    color="#5D6978"
)

letters8b = ["A", "B", "C", "D"]

for ax, (model, source), letter in zip(
    axes8b_fix.flat,
    plot_order8b,
    letters8b
):

    counts = MATRICES8[(model, source)]

    normalized = NORMALIZED8[(model, source)]

    support = SUPPORT8[(model, source)]

    masked = np.ma.masked_invalid(normalized)

    ax.imshow(
        masked,
        cmap=CMAP8,
        vmin=0,
        vmax=1,
        aspect="equal",
        interpolation="nearest"
    )

    ax.set_title(
        f"{letter}  {MODEL_LABEL8[model]}\n"
        f"{SOURCE_LABEL8[source]}",
        loc="left",
        fontsize=11,
        fontweight="bold",
        pad=13
    )

    ax.set_xticks(range(5))
    ax.set_yticks(range(5))

    ax.set_xticklabels(
        [PRETTY8[f] for f in FAMILIES8],
        rotation=35,
        ha="right",
        fontsize=9
    )

    # FIX: each panel gets its OWN source-specific labels.
    correct_y_labels = [
        f"{PRETTY8[f]} (n={int(support[i])})"
        for i, f in enumerate(FAMILIES8)
    ]

    ax.set_yticklabels(
        correct_y_labels,
        fontsize=9
    )

    # Verify displayed labels were assigned correctly.
    actual_labels = [
        label.get_text()
        for label in ax.get_yticklabels()
    ]

    assert actual_labels == correct_y_labels

    ax.set_xlabel(
        "Predicted attack family",
        fontsize=10
    )

    ax.set_ylabel(
        "True attack family",
        fontsize=10
    )

    # Delineate cells.
    ax.set_xticks(
        np.arange(-0.5, 5, 1),
        minor=True
    )
    ax.set_yticks(
        np.arange(-0.5, 5, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=1.6
    )

    ax.tick_params(
        which="both",
        length=0
    )

    # Annotate support-aware percentages and counts.
    for i in range(5):
        for j in range(5):

            if support[i] == 0:

                label = "N/A"
                color = "#687584"

            elif counts[i, j] == 0:

                label = "·"
                color = "#687584"

            else:

                percentage = normalized[i, j] * 100

                label = (
                    f"{percentage:.1f}%\n"
                    f"({int(counts[i, j])})"
                )

                color = (
                    "white"
                    if normalized[i, j] > 0.53
                    else "#182B40"
                )

            ax.text(
                j, i,
                label,
                ha="center",
                va="center",
                color=color,
                fontsize=8.5,
                fontweight=(
                    "bold" if i == j else "normal"
                )
            )

        if support[i] > 0:

            ax.add_patch(
                Rectangle(
                    (i - 0.48, i - 0.48),
                    0.96,
                    0.96,
                    fill=False,
                    edgecolor="#D78B43",
                    linewidth=1.7
                )
            )

# ------------------------------------------------------------
# 3. Shared colorbar and publication caption
# ------------------------------------------------------------

fig8b_fix.subplots_adjust(
    left=0.15,
    right=0.88,
    top=0.88,
    bottom=0.13,
    hspace=0.50,
    wspace=0.43
)

colorbar_ax8b = fig8b_fix.add_axes(
    [0.91, 0.24, 0.016, 0.50]
)

cb8b = fig8b_fix.colorbar(
    ScalarMappable(
        norm=Normalize(0, 1),
        cmap=CMAP8
    ),
    cax=colorbar_ax8b
)

cb8b.set_label(
    "Fraction of true-class windows",
    fontsize=10,
    labelpad=9
)

fig8b_fix.text(
    0.10,
    0.035,
    "Synthetic NAF v2 validation. "
    "Each nonempty cell reports row percentage and count. "
    "Support is source-specific; N/A indicates no true "
    "examples of that family. Family predictions are "
    "conditional on a true future event.",
    fontsize=9,
    color="#5D6978"
)

# ------------------------------------------------------------
# 4. Save corrected figure under a NEW filename
# ------------------------------------------------------------

stamp8b = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem8b = (
    f"F04_Source_Family_Confusion_CORRECTED_{stamp8b}"
)

saved8b = []

for extension in ["pdf", "svg", "png"]:

    path = FIGURES8B / f"{stem8b}.{extension}"

    if path.exists():
        raise FileExistsError(path)

    fig8b_fix.savefig(
        path,
        dpi=450 if extension == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    saved8b.append(path)

plt.show()
plt.close(fig8b_fix)

# ------------------------------------------------------------
# 5. New provenance manifest
# ------------------------------------------------------------

def sha256_8b(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest8b = {
    "project": "NetGuard",
    "figure": stem8b,
    "fix": (
        "Independent y-axis tick labels for each "
        "source-specific confusion matrix"
    ),
    "source_specific_supports": expected_support8b,
    "support_validation": "PASS",
    "data_source": (
        "Previously validated Cell 8 confusion matrices"
    ),
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT8B)),
            "sha256": sha256_8b(p)
        }
        for p in saved8b
    ],
    "official_test_used": False,
    "models_trained": 0
}

manifest_path8b = (
    MANIFESTS8B /
    f"corrected_confusion_manifest_{stamp8b}.json"
)

with manifest_path8b.open(
    "x", encoding="utf-8"
) as f:
    json.dump(
        manifest8b,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 8B COMPLETE")
print("=" * 90)

print("Corrected confusion panels: 4")
print("Source-specific support labels: VERIFIED")
print("Models retrained: 0")
print("Official TEST accessed: NO")
print("Existing figures overwritten: NO")

print("\nNew files:")

for path in saved8b + [manifest_path8b]:
    assert path.is_file()
    print(" -", path.relative_to(PROJECT8B))

print("=" * 90)


In [ ]:

# ================================================================
# NETGUARD RESEARCH OUTPUTS
# CELL 9 — PROBABILITY CALIBRATION ATLAS
#
# F06: 3 horizons x 2 competing-risk models
# Each panel contains:
#   - Reliability curve
#   - Ideal calibration reference
#   - Validation sample distribution per probability bin
#   - Expected Calibration Error (ECE)
#
# Also retires the three known incorrect F04 figure exports,
# after verifying the corrected F04 exists.
#
# NO training / NO TEST / NO checkpoint changes
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

print("=" * 92)
print("CELL 9 — FORECASTING PROBABILITY CALIBRATION")
print("=" * 92)

PROJECT9 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

MODELS9 = PROJECT9 / "Synthetic_NAF_v2" / "models"
OUT9 = PROJECT9 / "Research_Outputs"

FIG9 = OUT9 / "01_Figures" / "Forecasting"
TAB9 = OUT9 / "02_Tables"
MAN9 = OUT9 / "05_Manifests"

for folder in [FIG9, TAB9, MAN9]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

SOURCES9 = {
    "Competing-Risk LSTM": (
        MODELS9 / "LSTM_V2_CompetingRisk"
        / "official_validation" / "CALIBRATION_10_BINS.csv"
    ),
    "Competing-Risk TCN": (
        MODELS9 / "TCN_V2_CompetingRisk"
        / "official_validation" / "CALIBRATION_10_BINS.csv"
    )
}

for name, path in SOURCES9.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

HORIZONS9 = ["H5", "H10", "H15"]

COLORS9 = {
    "Competing-Risk LSTM": "#176E83",
    "Competing-Risk TCN": "#935BA0"
}

# ------------------------------------------------------------
# A. LOAD, NORMALIZE AND VALIDATE SOURCE REPORTS
# ------------------------------------------------------------

DATA9 = {}
SUMMARY9 = []
BINS9 = []

print("\nA. VALIDATING CALIBRATION REPORTS")
print("-" * 92)

required9 = {
    "horizon", "bin", "lower", "upper",
    "count", "mean_prediction", "observed_rate"
}

for model, path in SOURCES9.items():

    frame = pd.read_csv(path)

    # LSTM and TCN use different CSV capitalization.
    frame.columns = [
        str(c).strip().lower()
        for c in frame.columns
    ]

    if not required9.issubset(frame.columns):
        raise RuntimeError(
            f"{model}: missing {required9 - set(frame.columns)}"
        )

    if len(frame) != 30:
        raise RuntimeError(
            f"{model}: expected 30 calibration rows."
        )

    if frame.duplicated(["horizon", "bin"]).any():
        raise RuntimeError(f"{model}: duplicate bins.")

    frame["horizon"] = (
        frame["horizon"].astype(str).str.strip()
    )

    numeric_cols = [
        "bin", "lower", "upper", "count",
        "mean_prediction", "observed_rate"
    ]

    for col in numeric_cols:
        frame[col] = pd.to_numeric(
            frame[col], errors="raise"
        )

    if not np.isfinite(
        frame[numeric_cols].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(f"{model}: nonfinite values.")

    if set(frame["horizon"]) != set(HORIZONS9):
        raise RuntimeError(f"{model}: unexpected horizons.")

    DATA9[model] = {}

    for horizon in HORIZONS9:

        part = (
            frame.loc[frame["horizon"] == horizon]
            .sort_values("bin")
            .copy()
        )

        if not np.array_equal(
            part["bin"].to_numpy(),
            np.arange(10)
        ):
            raise RuntimeError(
                f"{model}/{horizon}: bin IDs mismatch."
            )

        lower = part["lower"].to_numpy()
        upper = part["upper"].to_numpy()
        count = part["count"].to_numpy()
        predicted = part["mean_prediction"].to_numpy()
        observed = part["observed_rate"].to_numpy()

        if not np.allclose(
            count, np.round(count)
        ):
            raise RuntimeError("Noninteger bin counts.")

        if np.any(count < 0):
            raise RuntimeError("Negative bin count.")

        if not (
            np.all((predicted >= 0) & (predicted <= 1))
            and np.all((observed >= 0) & (observed <= 1))
        ):
            raise RuntimeError("Invalid probability/rate.")

        if not (
            np.isclose(lower[0], 0)
            and np.isclose(upper[-1], 1)
            and np.allclose(lower[1:], upper[:-1])
        ):
            raise RuntimeError(
                f"{model}/{horizon}: bin boundaries invalid."
            )

        if np.any(upper <= lower):
            raise RuntimeError("Invalid bin widths.")

        total = int(count.sum())

        if total != 18839:
            raise RuntimeError(
                f"{model}/{horizon}: expected "
                f"18,839 windows, got {total}"
            )

        if np.any(
            (count > 0)
            & ((predicted < lower - 1e-8)
               | (predicted > upper + 1e-8))
        ):
            raise RuntimeError(
                f"{model}/{horizon}: mean outside bin."
            )

        # Weighted ECE over the 10 fixed bins.
        ece = float(
            np.sum(
                count * np.abs(predicted - observed)
            ) / total
        )

        signed_bias = float(
            np.sum(
                count * (predicted - observed)
            ) / total
        )

        prevalence = float(
            np.sum(count * observed) / total
        )

        mean_risk = float(
            np.sum(count * predicted) / total
        )

        DATA9[model][horizon] = part

        SUMMARY9.append({
            "Model": model,
            "Horizon": horizon,
            "Windows": total,
            "ECE_10": ece,
            "Mean_Predicted_Risk": mean_risk,
            "Observed_Event_Rate": prevalence,
            "Signed_Calibration_Bias": signed_bias
        })

        export_part = part.copy()
        export_part.insert(0, "Model", model)
        BINS9.append(export_part)

        print(
            f"PASS | {model:<22} {horizon} "
            f"| N={total:,} "
            f"| ECE={ece:.5f} "
            f"| prevalence={prevalence:.4f}"
        )

SUMMARY9 = pd.DataFrame(SUMMARY9)
BINS9 = pd.concat(BINS9, ignore_index=True)

# Both models must be evaluated against the same prevalence.
for horizon in HORIZONS9:

    subset = SUMMARY9.loc[
        SUMMARY9["Horizon"] == horizon
    ]

    if not np.allclose(
        subset["Observed_Event_Rate"].to_numpy(),
        subset["Observed_Event_Rate"].iloc[0],
        atol=1e-9
    ):
        raise RuntimeError(
            f"Observed prevalence disagrees for {horizon}."
        )

print("\nAll six calibration datasets validated.")

# ------------------------------------------------------------
# B. PUBLICATION FIGURE: SIX COMPOUND PANELS
#
# Each compound panel has:
#   Top    — reliability
#   Bottom — probability-bin occupancy (log count)
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig9 = plt.figure(
    figsize=(15.5, 16.5),
    facecolor="white"
)

outer9 = GridSpec(
    3, 2,
    figure=fig9,
    left=0.10,
    right=0.96,
    top=0.86,
    bottom=0.085,
    wspace=0.23,
    hspace=0.40
)

fig9.suptitle(
    "Reliability of future-attack probability forecasts",
    x=0.10,
    y=0.975,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig9.text(
    0.10,
    0.942,
    "Three prediction horizons × two competing-risk "
    "architectures  |  18,839 validation windows per panel",
    fontsize=11,
    color="#5A6573"
)

model_order9 = [
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

panel_letters9 = "ABCDEF"

for i, horizon in enumerate(HORIZONS9):
    for j, model in enumerate(model_order9):

        panel_id = i * 2 + j

        inner = outer9[i, j].subgridspec(
            2, 1,
            height_ratios=[3.6, 1.1],
            hspace=0.07
        )

        ax = fig9.add_subplot(inner[0])
        ax_count = fig9.add_subplot(
            inner[1], sharex=ax
        )

        part = DATA9[model][horizon]
        valid = part["count"] > 0

        x = part.loc[
            valid, "mean_prediction"
        ].to_numpy()

        y = part.loc[
            valid, "observed_rate"
        ].to_numpy()

        n = part.loc[
            valid, "count"
        ].to_numpy()

        color = COLORS9[model]

        # Ideal calibration reference.
        ax.plot(
            [0, 1], [0, 1],
            linestyle=(0, (4, 4)),
            color="#778491",
            linewidth=1.35,
            zorder=1
        )

        # Actual reliability curve.
        ax.plot(
            x, y,
            color=color,
            linewidth=2.15,
            alpha=0.90,
            zorder=3
        )

        # Marker area encodes bin sample count.
        sizes = (
            30 + 125 * np.sqrt(
                n / max(int(n.max()), 1)
            )
        )

        ax.scatter(
            x, y,
            s=sizes,
            color=color,
            edgecolor="white",
            linewidth=0.9,
            zorder=4
        )

        ax.set_xlim(-0.025, 1.025)
        ax.set_ylim(-0.025, 1.025)

        ax.set_aspect("equal", adjustable="box")

        ax.grid(
            linestyle=":",
            color="#BBC5D0",
            alpha=0.45
        )

        ax.set_axisbelow(True)
        ax.set_facecolor("#FBFCFD")

        ax.set_xticks(
            np.linspace(0, 1, 6)
        )
        ax.set_yticks(
            np.linspace(0, 1, 6)
        )

        ax.tick_params(
            labelbottom=False,
            labelsize=8.5
        )

        ax.set_ylabel(
            "Observed event rate",
            fontsize=9.5
        )

        row = SUMMARY9.loc[
            (SUMMARY9["Model"] == model)
            & (SUMMARY9["Horizon"] == horizon)
        ].iloc[0]

        ax.set_title(
            f"{panel_letters9[panel_id]}  "
            f"{model}  ·  {horizon}",
            loc="left",
            pad=12,
            fontsize=11.5,
            fontweight="bold"
        )

        ax.text(
            0.035, 0.96,
            f"ECE = {row['ECE_10']:.4f}\n"
            f"Event rate = {row['Observed_Event_Rate']:.3f}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=9,
            color="#263746",
            bbox={
                "facecolor": "white",
                "edgecolor": "#E0E5EA",
                "alpha": 0.95,
                "boxstyle": "round,pad=0.4"
            }
        )

        # Distribution of predicted probabilities.
        centers = (
            part["lower"].to_numpy()
            + part["upper"].to_numpy()
        ) / 2

        widths = (
            part["upper"].to_numpy()
            - part["lower"].to_numpy()
        ) * 0.86

        counts = part["count"].to_numpy()

        # Log scale highlights low-occupancy bins.
        ax_count.bar(
            centers,
            np.maximum(counts - 1, 0),
            width=widths,
            bottom=1,
            color=color,
            alpha=0.68,
            edgecolor="white",
            linewidth=0.5
        )

        ax_count.set_yscale("log")
        ax_count.set_ylim(1, 50000)

        ax_count.set_yticks(
            [1, 100, 10000]
        )
        ax_count.set_yticklabels(
            ["1", "100", "10k"]
        )

        ax_count.set_xlim(-0.025, 1.025)
        ax_count.set_xticks(
            np.linspace(0, 1, 6)
        )

        ax_count.set_ylabel(
            "Count\n(log)",
            fontsize=8.5
        )

        ax_count.set_xlabel(
            "Predicted attack probability",
            fontsize=9.5
        )

        ax_count.tick_params(labelsize=8)
        ax_count.set_facecolor("#FBFCFD")

        ax_count.grid(
            axis="y",
            linestyle=":",
            alpha=0.35
        )

        ax_count.set_axisbelow(True)

fig9.legend(
    handles=[
        Line2D(
            [0], [0],
            color="#778491",
            linestyle=(0, (4, 4)),
            linewidth=1.5,
            label="Perfect calibration"
        ),
        Line2D(
            [0], [0],
            color=COLORS9["Competing-Risk LSTM"],
            marker="o",
            linewidth=2,
            label="LSTM reliability"
        ),
        Line2D(
            [0], [0],
            color=COLORS9["Competing-Risk TCN"],
            marker="o",
            linewidth=2,
            label="TCN reliability"
        )
    ],
    loc="upper left",
    bbox_to_anchor=(0.10, 0.917),
    ncol=3,
    frameon=False,
    fontsize=10
)

fig9.text(
    0.10,
    0.032,
    "Synthetic NAF v2 development validation. "
    "ECE is frequency-weighted over ten fixed probability bins. "
    "Marker size indicates bin support; lower panels use "
    "logarithmic counts. Observations come from overlapping "
    "windows, so no independent-bin confidence intervals "
    "are implied.",
    fontsize=9,
    color="#5A6573",
    wrap=True
)

# ------------------------------------------------------------
# C. SAVE THE NEW FIGURE AND RESEARCH TABLES
# ------------------------------------------------------------

STAMP9 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM9 = f"F06_Probability_Calibration_Atlas_{STAMP9}"

FIGURE_FILES9 = []

for ext in ["pdf", "svg", "png"]:

    path = FIG9 / f"{STEM9}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig9.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES9.append(path)

plt.show()
plt.close(fig9)

SUMMARY_PATH9 = (
    TAB9 / f"T06_Calibration_Summary_{STAMP9}.csv"
)

BINS_PATH9 = (
    TAB9 / f"T07_Calibration_Bin_Details_{STAMP9}.csv"
)

SUMMARY9.to_csv(
    SUMMARY_PATH9,
    index=False,
    mode="x",
    float_format="%.8f"
)

BINS9.to_csv(
    BINS_PATH9,
    index=False,
    mode="x",
    float_format="%.8f"
)

# ------------------------------------------------------------
# D. RETIRE ONLY THE KNOWN INCORRECT F04 EXPORTS
# ------------------------------------------------------------

old_run9 = "20261027_061113_471823"
corrected_run9 = "20261027_061555_018961"

old_stem9 = (
    f"F04_Source_Family_Confusion_Atlas_{old_run9}"
)

corrected_stem9 = (
    "F04_Source_Family_Confusion_CORRECTED_"
    + corrected_run9
)

corrected_files9 = [
    FIG9 / f"{corrected_stem9}.{ext}"
    for ext in ["pdf", "svg", "png"]
]

obsolete_files9 = [
    FIG9 / f"{old_stem9}.{ext}"
    for ext in ["pdf", "svg", "png"]
]

retired9 = []

if all(
    p.is_file() and p.stat().st_size > 0
    for p in corrected_files9
):

    for path in obsolete_files9:
        if path.is_file():
            path.unlink()
            retired9.append(path.name)

    print("\nObsolete F04 figure exports retired:")
    for name in retired9:
        print(" -", name)

else:
    print(
        "\nOld F04 exports preserved: "
        "corrected files could not all be verified."
    )

# Original manifests remain available as historical provenance.

# ------------------------------------------------------------
# E. MANIFEST WITH SOURCE CHECKSUMS
# ------------------------------------------------------------

def hash_file9(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

generated9 = (
    FIGURE_FILES9
    + [SUMMARY_PATH9, BINS_PATH9]
)

manifest9 = {
    "project": "NetGuard",
    "figure": STEM9,
    "stage": "Research Outputs Cell 9",
    "evaluation": "Synthetic_NAF_v2 validation",
    "models": model_order9,
    "horizons": HORIZONS9,
    "windows_per_model_horizon": 18839,
    "ece_definition": (
        "sum_b(count_b * abs(mean_prediction_b "
        "- observed_rate_b)) / total_windows"
    ),
    "source_files": [
        {
            "path": str(p.relative_to(PROJECT9)),
            "sha256": hash_file9(p)
        }
        for p in SOURCES9.values()
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT9)),
            "sha256": hash_file9(p)
        }
        for p in generated9
    ],
    "retired_incorrect_figure_exports": retired9,
    "replacement_confusion_figure": corrected_stem9,
    "scientific_notes": [
        "Calibration based on fixed ten-bin validation reports.",
        "Low aggregate ECE does not guarantee local calibration.",
        "Window overlap introduces statistical dependence.",
        "Histogram counts use a logarithmic axis.",
        "No uncertainty intervals were estimated here.",
        "No official TEST data accessed."
    ],
    "models_trained": 0,
    "official_test_accessed": False
}

MANIFEST_PATH9 = (
    MAN9 / f"calibration_research_manifest_{STAMP9}.json"
)

with MANIFEST_PATH9.open(
    "x", encoding="utf-8"
) as f:
    json.dump(
        manifest9,
        f,
        indent=2,
        allow_nan=False
    )

# ------------------------------------------------------------
# F. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("CELL 9 COMPLETE")
print("=" * 92)

for p in generated9 + [MANIFEST_PATH9]:

    assert p.is_file() and p.stat().st_size > 0
    print("PASS:", p.relative_to(OUT9))

print("\nCalibration panels: 6")
print("Bin histograms: 6")
print("Source validation: PASS")
print("Research tables: 2")
print("Figure exports: PDF, SVG, PNG")
print("Obsolete F04 files deleted:", len(retired9))

print("\nModels trained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Source reports modified: NO")

print("=" * 92)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 9B — REVISED PUBLICATION CALIBRATION ATLAS
#
# FIXES:
# - Reliability plots and histograms have identical width
# - No artificially shrunken square axes
# - Better panel spacing and typography
# - Consistent x/y probability ranges
# - Shows ECE, event prevalence, and bin support
#
# OUTPUT:
#   Revised F06 as PDF / SVG / PNG
#   New JSON provenance manifest
#
# Original F06 is NOT deleted or overwritten.
# NO TRAINING. NO TEST ACCESS.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, FuncFormatter

print("=" * 86)
print("CELL 9B — REVISED FORECASTING CALIBRATION ATLAS")
print("=" * 86)

PROJECT = Path("/content/drive/MyDrive/network-attack-forecasting")
OUTPUT = PROJECT / "Research_Outputs"
FIGURES = OUTPUT / "01_Figures" / "Forecasting"
MANIFESTS = OUTPUT / "05_Manifests"
MODELS = PROJECT / "Synthetic_NAF_v2" / "models"

for folder in [FIGURES, MANIFESTS]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

sources = {
    "Competing-Risk LSTM": (
        MODELS / "LSTM_V2_CompetingRisk"
        / "official_validation" / "CALIBRATION_10_BINS.csv"
    ),
    "Competing-Risk TCN": (
        MODELS / "TCN_V2_CompetingRisk"
        / "official_validation" / "CALIBRATION_10_BINS.csv"
    )
}

horizons = ["H5", "H10", "H15"]
model_names = list(sources.keys())

colors = {
    "Competing-Risk LSTM": "#176F7B",
    "Competing-Risk TCN": "#86559C"
}

# ------------------------------------------------------------
# 1. Load and validate the original calibration reports
# ------------------------------------------------------------

results = {}
summary = []

for model_name, path in sources.items():

    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    df["horizon"] = df["horizon"].astype(str).str.strip()

    required = {
        "horizon", "bin", "lower", "upper", "count",
        "mean_prediction", "observed_rate"
    }

    if not required.issubset(df.columns):
        raise RuntimeError(
            f"{model_name}: unexpected calibration schema"
        )

    if df.duplicated(["horizon", "bin"]).any():
        raise RuntimeError("Duplicate calibration bins")

    results[model_name] = {}

    for horizon in horizons:

        part = (
            df.loc[df["horizon"] == horizon]
            .sort_values("bin")
            .copy()
        )

        if len(part) != 10:
            raise RuntimeError(
                f"{model_name} {horizon}: expected ten bins"
            )

        for col in [
            "bin", "lower", "upper", "count",
            "mean_prediction", "observed_rate"
        ]:
            part[col] = pd.to_numeric(
                part[col], errors="raise"
            )

        if not np.array_equal(
            part["bin"].to_numpy(), np.arange(10)
        ):
            raise RuntimeError("Invalid bin IDs")

        if not np.isfinite(
            part[
                ["lower", "upper", "count",
                 "mean_prediction", "observed_rate"]
            ].to_numpy()
        ).all():
            raise RuntimeError("Nonfinite calibration data")

        counts = part["count"].to_numpy(dtype=int)
        predicted = part["mean_prediction"].to_numpy()
        observed = part["observed_rate"].to_numpy()

        if counts.sum() != 18839 or (counts < 0).any():
            raise RuntimeError("Unexpected validation count")

        if (
            (predicted < 0).any()
            or (predicted > 1).any()
            or (observed < 0).any()
            or (observed > 1).any()
        ):
            raise RuntimeError("Invalid probabilities")

        ece = np.sum(
            counts * np.abs(predicted - observed)
        ) / counts.sum()

        prevalence = np.sum(
            counts * observed
        ) / counts.sum()

        results[model_name][horizon] = part

        summary.append({
            "model": model_name,
            "horizon": horizon,
            "ece": float(ece),
            "prevalence": float(prevalence)
        })

summary_df = pd.DataFrame(summary)

print("\nValidated calibration combinations: 6")
print(summary_df.round(5).to_string(index=False))

# ------------------------------------------------------------
# 2. Consistent research-paper styling
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig = plt.figure(figsize=(14.5, 14.5), facecolor="white")

fig.suptitle(
    "Forecast probability calibration across prediction horizons",
    x=0.08, y=0.985,
    ha="left", fontsize=18, fontweight="bold"
)

fig.text(
    0.08, 0.953,
    "Reliability and probability-bin occupancy "
    "for competing-risk forecasting models",
    fontsize=10.5,
    color="#556274"
)

outer = fig.add_gridspec(
    3, 2,
    left=0.09,
    right=0.96,
    top=0.865,
    bottom=0.09,
    wspace=0.20,
    hspace=0.47
)

letters = "ABCDEF"

# ------------------------------------------------------------
# 3. Draw six properly aligned compound panels
# ------------------------------------------------------------

for row, horizon in enumerate(horizons):

    for col, model_name in enumerate(model_names):

        inner = outer[row, col].subgridspec(
            2, 1,
            height_ratios=[3.3, 1.05],
            hspace=0.09
        )

        ax = fig.add_subplot(inner[0])
        ax_hist = fig.add_subplot(inner[1], sharex=ax)

        part = results[model_name][horizon]

        x = part["mean_prediction"].to_numpy()
        y = part["observed_rate"].to_numpy()
        n = part["count"].to_numpy(dtype=int)

        color = colors[model_name]
        info = summary_df.loc[
            (summary_df["model"] == model_name)
            & (summary_df["horizon"] == horizon)
        ].iloc[0]

        # Reliability diagram
        ax.plot(
            [0, 1], [0, 1],
            color="#778495",
            linestyle=(0, (4, 4)),
            linewidth=1.35,
            label="Perfect calibration"
        )

        ax.plot(
            x, y,
            color=color,
            linewidth=2.1,
            marker="o",
            markersize=5.1,
            markeredgecolor="white",
            markeredgewidth=0.7,
            zorder=3
        )

        # IMPORTANT: No set_aspect("equal").
        # The two axes now retain identical panel widths.
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xticks(np.linspace(0, 1, 6))
        ax.set_yticks(np.linspace(0, 1, 6))

        ax.grid(
            linestyle=":",
            linewidth=0.65,
            alpha=0.35
        )
        ax.set_axisbelow(True)
        ax.set_facecolor("#FBFCFE")

        label = (
            f"{letters[row * 2 + col]}   "
            f"{'LSTM' if col == 0 else 'TCN'}"
            f"  |  {horizon}"
        )

        ax.set_title(
            label,
            loc="left",
            fontweight="bold",
            pad=12
        )

        ax.set_ylabel("Observed event frequency")

        ax.tick_params(
            axis="x",
            labelbottom=False,
            length=0
        )

        ax.text(
            0.025, 0.96,
            f"ECE: {info['ece']:.4f}\n"
            f"Prevalence: {info['prevalence']:.3f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox={
                "facecolor": "white",
                "edgecolor": "#DDE3E9",
                "boxstyle": "round,pad=0.35"
            }
        )

        # Aligned bin-support histogram
        centers = (
            part["lower"].to_numpy()
            + part["upper"].to_numpy()
        ) / 2

        widths = (
            part["upper"].to_numpy()
            - part["lower"].to_numpy()
        ) * 0.85

        ax_hist.bar(
            centers,
            np.maximum(n - 1, 0),
            bottom=1,
            width=widths,
            color=color,
            alpha=0.72,
            edgecolor="white",
            linewidth=0.6
        )

        ax_hist.set_yscale("log")
        ax_hist.set_ylim(1, 50000)

        ax_hist.set_yticks([1, 100, 10000])
        ax_hist.yaxis.set_major_formatter(
            FuncFormatter(
                lambda v, pos:
                f"{int(v):,}" if v >= 1 else ""
            )
        )

        ax_hist.set_xticks(
            np.linspace(0, 1, 6)
        )

        ax_hist.set_xlabel(
            "Predicted probability"
        )
        ax_hist.set_ylabel("Bin count\n(log)", fontsize=9)

        ax_hist.grid(
            axis="y",
            linestyle=":",
            alpha=0.28
        )

        ax_hist.set_axisbelow(True)
        ax_hist.set_facecolor("#FBFCFE")

# ------------------------------------------------------------
# 4. Shared legend and interpretation
# ------------------------------------------------------------

fig.legend(
    handles=[
        Line2D(
            [0], [0],
            color="#778495",
            linestyle=(0, (4, 4)),
            label="Perfect calibration"
        ),
        Line2D(
            [0], [0],
            color=colors["Competing-Risk LSTM"],
            marker="o",
            label="LSTM"
        ),
        Line2D(
            [0], [0],
            color=colors["Competing-Risk TCN"],
            marker="o",
            label="TCN"
        )
    ],
    loc="upper left",
    bbox_to_anchor=(0.08, 0.929),
    ncol=3,
    frameon=False,
    fontsize=10
)

fig.text(
    0.08, 0.035,
    "Synthetic NAF v2 validation (18,839 windows per model "
    "and horizon). ECE uses ten probability bins. "
    "Bin counts are logarithmic. Overlapping windows "
    "are dependent; uncertainty bands are not estimated.",
    fontsize=9,
    color="#566371"
)

# ------------------------------------------------------------
# 5. Save revised F06 without overwriting old files
# ------------------------------------------------------------

stamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem = f"F06_Calibration_Atlas_V2_{stamp}"
created = []

for ext in ["pdf", "svg", "png"]:

    destination = FIGURES / f"{stem}.{ext}"

    if destination.exists():
        raise FileExistsError(destination)

    fig.savefig(
        destination,
        dpi=450 if ext == "png" else 150,
        bbox_inches="tight",
        facecolor="white"
    )

    created.append(destination)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 6. Provenance manifest
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "project": "NetGuard",
    "figure": stem,
    "stage": "Cell 9B — layout revision",
    "replaces_visual_design_of": "F06_Probability_Calibration_Atlas",
    "data_or_metric_changes": False,
    "models": model_names,
    "horizons": horizons,
    "summary": summary,
    "input_files": [
        {
            "path": str(p.relative_to(PROJECT)),
            "sha256": sha256_file(p)
        }
        for p in sources.values()
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT)),
            "sha256": sha256_file(p)
        }
        for p in created
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

manifest_path = MANIFESTS / (
    f"calibration_atlas_v2_manifest_{stamp}.json"
)

with manifest_path.open("x", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, allow_nan=False)

print("\n" + "=" * 86)
print("CELL 9B COMPLETE")
print("=" * 86)

for p in created + [manifest_path]:
    assert p.is_file() and p.stat().st_size > 0
    print("PASS:", p.relative_to(OUTPUT))

print("\nCalibration values changed: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")
print("Original F06 overwritten or deleted: NO")
print("=" * 86)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 10 — OPERATIONAL FORECASTING & FALSE-ALERT ANALYSIS
#
# F07: 3 x 3 publication-style operating-characteristics atlas
#
# Columns:
#     H5, H10, H15
#
# Rows:
#     1. Precision and recall versus threshold
#     2. Window-level false-positive rate versus threshold
#     3. Whole-benign-episode alert fraction versus threshold
#
# Uses saved FINAL_VALIDATION_AUDIT only.
#
# Saves NEW:
#   01_Figures/Forecasting/  -> PDF, SVG, PNG
#   02_Tables/              -> detailed operating-point CSV
#   03_Metrics/             -> validated JSON metrics
#   04_Reports/             -> textual scientific report
#   05_Manifests/           -> SHA256 provenance manifest
#
# SAFETY:
#   NO TRAINING
#   NO OFFICIAL TEST ACCESS
#   NO MODEL CHECKPOINT MODIFICATIONS
#   NO OVERWRITING OR DELETING EXISTING OUTPUTS
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

print("=" * 96)
print("CELL 10 — OPERATIONAL FORECASTING & FALSE-ALERT ANALYSIS")
print("=" * 96)

# ----------------------------------------------------------------------
# 1. EXACT EXISTING PROJECT PATHS
# ----------------------------------------------------------------------

PROJECT10 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

AUDIT10 = (
    PROJECT10
    / "Synthetic_NAF_v2"
    / "models"
    / "FINAL_VALIDATION_AUDIT"
)

OUTPUT10 = PROJECT10 / "Research_Outputs"

FIGURES10 = OUTPUT10 / "01_Figures" / "Forecasting"
TABLES10 = OUTPUT10 / "02_Tables"
METRICS10 = OUTPUT10 / "03_Metrics"
REPORTS10 = OUTPUT10 / "04_Reports"
MANIFESTS10 = OUTPUT10 / "05_Manifests"

for directory in [
    AUDIT10,
    FIGURES10,
    TABLES10,
    METRICS10,
    REPORTS10,
    MANIFESTS10
]:
    if not directory.is_dir():
        raise FileNotFoundError(directory)

SOURCE10 = AUDIT10 / "FIXED_THRESHOLD_ALERT_AUDIT.csv"

if not SOURCE10.is_file():
    raise FileNotFoundError(SOURCE10)

# ----------------------------------------------------------------------
# 2. LOAD THE SAVED THRESHOLD REPORT
# ----------------------------------------------------------------------

df10 = pd.read_csv(SOURCE10)

REQUIRED10 = {
    "Model",
    "Horizon",
    "Threshold",
    "TP",
    "FP",
    "TN",
    "FN",
    "Precision",
    "Recall",
    "False_Positive_Rate",
    "Whole_Benign_Episodes",
    "Benign_Episodes_Alerted",
    "Benign_Episode_Alert_Fraction"
}

missing10 = REQUIRED10 - set(df10.columns)

if missing10:
    raise RuntimeError(
        f"Unexpected threshold report schema: {missing10}"
    )

MODEL_ORDER10 = [
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

HORIZONS10 = ["H5", "H10", "H15"]
THRESHOLDS10 = [0.1, 0.2, 0.5]

if set(df10["Model"]) != set(MODEL_ORDER10):
    raise RuntimeError("Unexpected model names.")

if set(df10["Horizon"]) != set(HORIZONS10):
    raise RuntimeError("Unexpected horizons.")

if len(df10) != 18:
    raise RuntimeError(
        f"Expected 18 operating points; got {len(df10)}."
    )

COUNT_COLS10 = [
    "TP", "FP", "TN", "FN",
    "Whole_Benign_Episodes",
    "Benign_Episodes_Alerted"
]

RATE_COLS10 = [
    "Precision",
    "Recall",
    "False_Positive_Rate",
    "Benign_Episode_Alert_Fraction"
]

df10["Threshold"] = pd.to_numeric(
    df10["Threshold"], errors="raise"
)

for col in COUNT_COLS10 + RATE_COLS10:

    df10[col] = pd.to_numeric(
        df10[col], errors="raise"
    )

if not np.isfinite(
    df10[
        ["Threshold"] + COUNT_COLS10 + RATE_COLS10
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite evaluation values.")

for col in COUNT_COLS10:

    values = df10[col].to_numpy()

    if (values < 0).any():
        raise RuntimeError(f"Negative count: {col}")

    if not np.array_equal(
        values, np.round(values)
    ):
        raise RuntimeError(f"Noninteger count: {col}")

if not df10[RATE_COLS10].ge(0).all().all():
    raise RuntimeError("Negative evaluation rate.")

if not df10[RATE_COLS10].le(1).all().all():
    raise RuntimeError("Evaluation rate exceeds one.")

if df10.duplicated(
    ["Model", "Horizon", "Threshold"]
).any():
    raise RuntimeError("Duplicate operating points.")

for (model, horizon), group in df10.groupby(
    ["Model", "Horizon"]
):

    observed_thresholds = sorted(
        group["Threshold"].tolist()
    )

    if not np.allclose(
        observed_thresholds,
        THRESHOLDS10,
        atol=1e-12
    ):
        raise RuntimeError(
            f"Threshold mismatch for {model}/{horizon}."
        )

# ----------------------------------------------------------------------
# 3. RECOMPUTE AND VERIFY EVERY OPERATING METRIC
# ----------------------------------------------------------------------

def safe_rate10(numerator, denominator):

    numerator = np.asarray(
        numerator, dtype=float
    )

    denominator = np.asarray(
        denominator, dtype=float
    )

    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(
            numerator, dtype=float
        ),
        where=(denominator > 0)
    )

df10["Calculated_Precision"] = safe_rate10(
    df10["TP"],
    df10["TP"] + df10["FP"]
)

df10["Calculated_Recall"] = safe_rate10(
    df10["TP"],
    df10["TP"] + df10["FN"]
)

df10["Calculated_FPR"] = safe_rate10(
    df10["FP"],
    df10["FP"] + df10["TN"]
)

df10["Calculated_Episode_Fraction"] = safe_rate10(
    df10["Benign_Episodes_Alerted"],
    df10["Whole_Benign_Episodes"]
)

# Saved reports may round to several decimal places.
# These tolerances are for checking reported rounding,
# not for changing any metric.

VERIFY10 = {
    "Precision": "Calculated_Precision",
    "Recall": "Calculated_Recall",
    "False_Positive_Rate": "Calculated_FPR",
    "Benign_Episode_Alert_Fraction":
        "Calculated_Episode_Fraction"
}

for reported, calculated in VERIFY10.items():

    deviation = np.abs(
        df10[reported] - df10[calculated]
    ).max()

    if deviation > 5e-4:
        raise RuntimeError(
            f"{reported}: reported/recomputed "
            f"values disagree, max error={deviation}."
        )

df10["Total_Windows"] = (
    df10["TP"] + df10["FP"]
    + df10["TN"] + df10["FN"]
)

if not (df10["Total_Windows"] == 18839).all():
    raise RuntimeError(
        "Expected 18,839 validation windows per operating point."
    )

if (
    df10["Benign_Episodes_Alerted"]
    > df10["Whole_Benign_Episodes"]
).any():
    raise RuntimeError(
        "Benign-episode alert count exceeds eligible episodes."
    )

# Check for monotonicity of decision outcomes as threshold rises.
# Do not assume precision must improve monotonically.

for (model, horizon), group in df10.groupby(
    ["Model", "Horizon"]
):

    ordered = group.sort_values("Threshold")

    for col in [
        "Calculated_Recall",
        "Calculated_FPR",
        "Calculated_Episode_Fraction"
    ]:

        values = ordered[col].to_numpy()

        if np.any(np.diff(values) > 1e-9):
            raise RuntimeError(
                f"Unexpected threshold monotonicity: "
                f"{model}/{horizon}/{col}"
            )

print("\nA. DATA QUALITY VERIFICATION")
print("-" * 96)

print("Saved operating points:", len(df10))
print("Validation windows per point: 18,839")
print("Precision validation: PASS")
print("Recall validation: PASS")
print("False-positive rate validation: PASS")
print("Benign-episode fraction validation: PASS")
print("Threshold monotonicity: PASS")

print("\nB. VERIFIED OPERATING POINTS")
print("-" * 96)

display10 = df10[
    [
        "Model",
        "Horizon",
        "Threshold",
        "Calculated_Precision",
        "Calculated_Recall",
        "Calculated_FPR",
        "Calculated_Episode_Fraction",
        "FP",
        "FN"
    ]
].copy()

print(
    display10
    .sort_values(
        ["Horizon", "Model", "Threshold"]
    )
    .round(4)
    .to_string(index=False)
)

# ----------------------------------------------------------------------
# 4. SCIENTIFIC FIGURE DESIGN
# ----------------------------------------------------------------------

COLORS10 = {
    "Competing-Risk LSTM": "#147784",
    "Competing-Risk TCN": "#9258A2"
}

SHORT10 = {
    "Competing-Risk LSTM": "CR-LSTM",
    "Competing-Risk TCN": "CR-TCN"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig10, axes10 = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(17, 12.8),
    squeeze=False
)

fig10.patch.set_facecolor("white")

fig10.suptitle(
    "Operating characteristics of network attack forecasts",
    x=0.085,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig10.text(
    0.085,
    0.951,
    "Fixed decision thresholds   |   "
    "Two competing-risk architectures   |   "
    "Three prediction horizons",
    fontsize=11,
    color="#586575"
)

# Common row scales allow honest cross-horizon comparison.

max_fpr10 = df10["Calculated_FPR"].max()
max_ep10 = df10["Calculated_Episode_Fraction"].max()

fpr_ylim10 = min(
    1.0,
    max(0.02, max_fpr10 * 1.20)
)

episode_ylim10 = min(
    1.0,
    max(0.02, max_ep10 * 1.20)
)

PANEL_LETTERS10 = list("ABCDEFGHI")

def style_panel10(ax):

    ax.set_facecolor("#FAFBFC")

    ax.grid(
        linestyle=":",
        linewidth=0.70,
        color="#CBD3DD",
        alpha=0.60
    )

    ax.set_axisbelow(True)

    ax.tick_params(
        length=3,
        width=0.65,
        pad=6
    )

    ax.set_xlim(0.075, 0.525)

    ax.set_xticks(THRESHOLDS10)

    ax.set_xticklabels(
        ["0.10", "0.20", "0.50"]
    )

    ax.set_xlabel(
        "Decision threshold",
        labelpad=9
    )

ROW_TITLES10 = [
    "Precision and event recall",
    "Window-level false-positive rate",
    "Whole-benign-episode alert fraction"
]

# ----------------------------------------------------------------------
# 5. GENERATE NINE COORDINATED PANELS
# ----------------------------------------------------------------------

for c, horizon in enumerate(HORIZONS10):

    for r in range(3):

        ax = axes10[r, c]

        style_panel10(ax)

        letter = PANEL_LETTERS10[r * 3 + c]

        ax.set_title(
            f"{letter}   {horizon} · {ROW_TITLES10[r]}",
            loc="left",
            fontsize=10.8,
            fontweight="bold",
            pad=14
        )

        if r == 0:
            ax.set_ylim(0, 1.035)
            ax.set_ylabel("Precision / recall")

        elif r == 1:
            ax.set_ylim(0, fpr_ylim10)
            ax.set_ylabel(
                "False-positive fraction\nof true-negative windows"
            )

        else:
            ax.set_ylim(0, episode_ylim10)
            ax.set_ylabel(
                "Fraction of wholly benign\nepisodes with ≥1 alert"
            )

        for model in MODEL_ORDER10:

            subset = (
                df10.loc[
                    (df10["Model"] == model)
                    & (df10["Horizon"] == horizon)
                ]
                .sort_values("Threshold")
            )

            x = subset["Threshold"].to_numpy()

            color = COLORS10[model]

            if r == 0:

                ax.plot(
                    x,
                    subset["Calculated_Recall"],
                    color=color,
                    linestyle="-",
                    linewidth=2.2,
                    marker="o",
                    markersize=6,
                    markeredgecolor="white",
                    markeredgewidth=0.7,
                    zorder=4
                )

                ax.plot(
                    x,
                    subset["Calculated_Precision"],
                    color=color,
                    linestyle=(0, (5, 2.5)),
                    linewidth=1.75,
                    marker="s",
                    markersize=5,
                    markeredgecolor="white",
                    markeredgewidth=0.6,
                    zorder=3
                )

            elif r == 1:

                ax.plot(
                    x,
                    subset["Calculated_FPR"],
                    color=color,
                    linestyle="-",
                    linewidth=2.2,
                    marker="o",
                    markersize=6,
                    markeredgecolor="white",
                    markeredgewidth=0.7
                )

            else:

                ax.plot(
                    x,
                    subset["Calculated_Episode_Fraction"],
                    color=color,
                    linestyle="-",
                    linewidth=2.2,
                    marker="D",
                    markersize=5.7,
                    markeredgecolor="white",
                    markeredgewidth=0.7
                )

# ----------------------------------------------------------------------
# 6. LEGEND AND SCIENTIFIC CAPTION
# ----------------------------------------------------------------------

LEGEND10 = [
    Line2D(
        [0], [0],
        color=COLORS10["Competing-Risk LSTM"],
        linewidth=2.4,
        label="Competing-Risk LSTM"
    ),
    Line2D(
        [0], [0],
        color=COLORS10["Competing-Risk TCN"],
        linewidth=2.4,
        label="Competing-Risk TCN"
    ),
    Line2D(
        [0], [0],
        color="#485563",
        linestyle="-",
        marker="o",
        linewidth=1.7,
        label="Recall / rate"
    ),
    Line2D(
        [0], [0],
        color="#485563",
        linestyle=(0, (5, 2.5)),
        marker="s",
        linewidth=1.7,
        label="Precision (top row)"
    )
]

fig10.legend(
    handles=LEGEND10,
    loc="upper left",
    bbox_to_anchor=(0.085, 0.927),
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=2.8,
    columnspacing=2.0
)

fig10.subplots_adjust(
    left=0.105,
    right=0.975,
    top=0.855,
    bottom=0.115,
    wspace=0.31,
    hspace=0.42
)

fig10.text(
    0.085,
    0.059,
    "Synthetic NAF v2 validation; each operating point uses "
    "18,839 overlapping decision windows. Thresholds are "
    "fixed diagnostic values, not optimized deployment policies.",
    fontsize=9.0,
    color="#576472"
)

fig10.text(
    0.085,
    0.041,
    "Whole-benign-episode alert fraction and window-level "
    "false-positive rate use different denominators and "
    "must not be interpreted as interchangeable.",
    fontsize=9.0,
    color="#576472"
)

# ----------------------------------------------------------------------
# 7. SAVE PDF / SVG / HIGH-RESOLUTION PNG
# ----------------------------------------------------------------------

RUN10 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM10 = f"F07_Operational_Threshold_Atlas_{RUN10}"

FIGURE_FILES10 = []

for ext in ["pdf", "svg", "png"]:

    destination = FIGURES10 / f"{STEM10}.{ext}"

    if destination.exists():
        raise FileExistsError(destination)

    fig10.savefig(
        destination,
        dpi=450 if ext == "png" else 150,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES10.append(destination)

plt.show()
plt.close(fig10)

# ----------------------------------------------------------------------
# 8. SAVE DETAILED NUMERICAL RESEARCH TABLE
# ----------------------------------------------------------------------

TABLE_PATH10 = (
    TABLES10
    / f"T08_Operational_Threshold_Metrics_{RUN10}.csv"
)

table10 = df10[
    [
        "Model",
        "Horizon",
        "Threshold",
        "TP", "FP", "TN", "FN",
        "Total_Windows",
        "Calculated_Precision",
        "Calculated_Recall",
        "Calculated_FPR",
        "Whole_Benign_Episodes",
        "Benign_Episodes_Alerted",
        "Calculated_Episode_Fraction"
    ]
].copy()

table10.to_csv(
    TABLE_PATH10,
    index=False,
    mode="x",
    float_format="%.8f"
)

# ----------------------------------------------------------------------
# 9. SAVE METRIC SNAPSHOT AS JSON
# ----------------------------------------------------------------------

METRIC_PATH10 = (
    METRICS10
    / f"operational_threshold_metrics_{RUN10}.json"
)

metric_content10 = {
    "project": "NetGuard",
    "benchmark": "Synthetic_NAF_v2",
    "evaluation": "saved_validation",
    "fixed_thresholds": THRESHOLDS10,
    "prediction_horizons": HORIZONS10,
    "models": MODEL_ORDER10,
    "records_per_operating_point": 18839,
    "operating_points": (
        table10
        .to_dict(orient="records")
    ),
    "definitions": {
        "precision": "TP / (TP + FP)",
        "recall": "TP / (TP + FN)",
        "fpr": "FP / (FP + TN)",
        "benign_episode_alert_fraction": (
            "wholly benign episodes alerted / "
            "all wholly benign episodes"
        )
    }
}

with METRIC_PATH10.open(
    "x", encoding="utf-8"
) as f:

    json.dump(
        metric_content10,
        f,
        indent=2,
        allow_nan=False
    )

# ----------------------------------------------------------------------
# 10. GENERATE HUMAN-READABLE RESEARCH REPORT
# ----------------------------------------------------------------------

REPORT_PATH10 = (
    REPORTS10
    / f"operational_threshold_analysis_{RUN10}.txt"
)

report_lines10 = [
    "NETGUARD — FIXED-THRESHOLD OPERATIONAL ANALYSIS",
    "=" * 88,
    "",
    "Benchmark: Synthetic NAF v2 development validation",
    "Models: Competing-Risk LSTM, Competing-Risk TCN",
    "Horizons: H5, H10, H15",
    "Thresholds: 0.10, 0.20, 0.50",
    "Validation windows per operating point: 18,839",
    "",
    "SCIENTIFIC INTERPRETATION",
    "-" * 88,
    "Raising an alert threshold generally trades event recall",
    "against the frequency of false-positive alerts.",
    "",
    "The false-positive rate is computed over true-negative",
    "decision windows, whereas benign-episode alert fraction",
    "is computed over whole benign episodes.",
    "",
    "The thresholds are fixed evaluation points and must not",
    "be presented as tuned operational deployment thresholds.",
    "",
    "The synthetic validation set contains temporally overlapping",
    "windows; these reported operating points do not establish",
    "live-network false-alert behavior.",
    "",
    "DETAILED OPERATING POINTS",
    "-" * 88,
    table10.round(6).to_string(index=False),
    "",
    "SOURCE",
    str(SOURCE10.relative_to(PROJECT10)),
    "",
    "Models trained: 0",
    "Official TEST data accessed: No",
]

with REPORT_PATH10.open(
    "x", encoding="utf-8"
) as f:
    f.write("\n".join(report_lines10))

# ----------------------------------------------------------------------
# 11. PROVENANCE MANIFEST
# ----------------------------------------------------------------------

def sha256_10(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

generated10 = (
    FIGURE_FILES10
    + [
        TABLE_PATH10,
        METRIC_PATH10,
        REPORT_PATH10
    ]
)

manifest10 = {
    "project": "NetGuard",
    "stage": "Research Outputs Cell 10",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "figure": STEM10,
    "source": {
        "path": str(SOURCE10.relative_to(PROJECT10)),
        "sha256": sha256_10(SOURCE10)
    },
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT10)),
            "sha256": sha256_10(p)
        }
        for p in generated10
    ],
    "checks": {
        "precision_recomputed": True,
        "recall_recomputed": True,
        "fpr_recomputed": True,
        "episode_fraction_recomputed": True,
        "threshold_monotonicity_verified": True,
        "validation_windows": 18839
    },
    "limitations": [
        "Fixed thresholds are not deployment-tuned.",
        "Overlapping validation windows are statistically dependent.",
        "Window-level FPR is not episode-level false-alert burden.",
        "This is synthetic benchmark validation, not live deployment.",
        "No official TEST dataset accessed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

MANIFEST_PATH10 = (
    MANIFESTS10
    / f"operational_threshold_manifest_{RUN10}.json"
)

with MANIFEST_PATH10.open(
    "x", encoding="utf-8"
) as f:

    json.dump(
        manifest10,
        f,
        indent=2,
        allow_nan=False
    )

# ----------------------------------------------------------------------
# 12. FINAL VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "=" * 96)
print("CELL 10 COMPLETE")
print("=" * 96)

print("Validated models:", len(MODEL_ORDER10))
print("Horizons:", len(HORIZONS10))
print("Thresholds per horizon:", len(THRESHOLDS10))
print("Operating points:", len(df10))
print("Figure panels:", 9)

print("\nSAVED OUTPUTS")
print("-" * 96)

for p in generated10 + [MANIFEST_PATH10]:

    assert p.is_file()
    assert p.stat().st_size > 0

    print("PASS:", p.relative_to(OUTPUT10))

print("\nModel training: NONE")
print("Official TEST accessed: NO")
print("Original source reports modified: NO")
print("Existing checkpoints modified: NO")
print("Previous research outputs overwritten: NO")

print("=" * 96)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 11 — OBSERVED-SIGNAL ALIGNMENT AUDIT
#
# Purpose:
#   Find real synthetic traffic-feature traces for the wide
#   first column of our reference-style research figure.
#
# Source:
#   Synthetic_NAF_v2/release_candidate_corrected/
#   minute_predictors.parquet
#
# Safety:
#   - Only selected VALIDATION episode IDs requested
#   - Filtered Parquet read; no TEST rows returned
#   - No model training
#   - No existing files modified
#   - No figures written yet
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
except ImportError:
    raise RuntimeError(
        "Install pyarrow with: %pip install pyarrow"
    )

print("=" * 90)
print("CELL 11 — SYNTHETIC TRAFFIC SIGNAL ALIGNMENT")
print("=" * 90)

PROJECT11 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

SIGNAL_FILE11 = (
    PROJECT11
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
    / "minute_predictors.parquet"
)

if not SIGNAL_FILE11.is_file():
    raise FileNotFoundError(SIGNAL_FILE11)

if "selected6" not in globals():
    raise RuntimeError(
        "Run Cell 6 first. We need its five selected "
        "validation episode IDs."
    )

if "reference5" not in globals():
    raise RuntimeError(
        "Run Cell 5 first. We need verified validation keys."
    )

# ------------------------------------------------------------
# A. Inspect only Parquet metadata / column names
# ------------------------------------------------------------

schema11 = pq.read_schema(SIGNAL_FILE11)
columns11 = list(schema11.names)

print("\nA. ORIGINAL TRAFFIC FEATURE SCHEMA")
print("-" * 90)

print("File:", SIGNAL_FILE11.relative_to(PROJECT11))
print("Size MB:", round(SIGNAL_FILE11.stat().st_size / 1048576, 2))

for i, name in enumerate(columns11, 1):
    print(f"{i:02d}. {name}")

def choose_column11(names):
    lookup = {
        c.lower(): c
        for c in columns11
    }
    return next(
        (lookup[n.lower()] for n in names if n.lower() in lookup),
        None
    )

episode_col11 = choose_column11([
    "Episode_ID", "episode_id", "episode",
    "episode_key"
])

minute_col11 = choose_column11([
    "Minute", "minute", "minute_idx",
    "time_idx", "time_index", "t"
])

split_col11 = choose_column11([
    "Split", "split", "dataset_split",
    "partition"
])

feature_names11 = [
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Flow_Count",
    "Total_Packets",
    "Total_Bytes",
    "Flow_Duration_Mean"
]

features11 = [
    c for name in feature_names11
    if (c := choose_column11([name])) is not None
]

print("\nB. AVAILABLE ALIGNMENT FIELDS")
print("-" * 90)
print("Episode identifier:", episode_col11)
print("Minute identifier:", minute_col11)
print("Split identifier:", split_col11)
print("Traffic features:", features11)

if episode_col11 is None or minute_col11 is None:
    print(
        "\nSTOP: Cannot join traffic features without "
        "explicit episode and minute identifiers."
    )
    print("No traffic-feature data has been read.")
    raise RuntimeError(
        "Review the schema printed above before continuing."
    )

if not features11:
    raise RuntimeError(
        "No expected traffic features found. "
        "Review the printed schema."
    )

# ------------------------------------------------------------
# C. Validate the requested episode IDs
# ------------------------------------------------------------

episode_ids11 = (
    selected6["Episode_ID"]
    .astype(str)
    .tolist()
)

if len(episode_ids11) != 5:
    raise RuntimeError(
        "Expected five selected validation episodes."
    )

if len(set(episode_ids11)) != 5:
    raise RuntimeError(
        "Selected episode IDs are not unique."
    )

reference_keys11 = (
    reference5
    .reset_index()
    .loc[
        lambda f:
            f["Episode_ID"].astype(str).isin(episode_ids11),
        ["Episode_ID", "Minute"]
    ]
    .copy()
)

reference_keys11["Episode_ID"] = (
    reference_keys11["Episode_ID"].astype(str)
)

reference_keys11["Minute"] = pd.to_numeric(
    reference_keys11["Minute"],
    errors="raise"
)

if reference_keys11.duplicated(
    ["Episode_ID", "Minute"]
).any():
    raise RuntimeError(
        "Duplicate reference validation keys."
    )

# ------------------------------------------------------------
# D. Read ONLY requested validation episodes
#
# This predicate is applied by the Arrow dataset scanner.
# If a split column exists, additionally require VALIDATION.
# Never fall back to an unfiltered full-file read.
# ------------------------------------------------------------

dataset11 = ds.dataset(
    str(SIGNAL_FILE11),
    format="parquet"
)

filter11 = ds.field(episode_col11).isin(
    episode_ids11
)

if split_col11 is not None:
    filter11 = (
        filter11
        & ds.field(split_col11).isin(
            ["VALIDATION", "validation", "Validation"]
        )
    )

read_columns11 = list(dict.fromkeys(
    [
        episode_col11,
        minute_col11,
        *features11
    ]
    + ([split_col11] if split_col11 else [])
))

print("\nC. FILTERED VALIDATION FEATURE READ")
print("-" * 90)
print("Requested episode IDs:", episode_ids11)
print("Selected columns:", read_columns11)
print("Filter:", filter11)

signal_frame11 = dataset11.to_table(
    columns=read_columns11,
    filter=filter11
).to_pandas()

print("Rows returned:", len(signal_frame11))

if signal_frame11.empty:
    print(
        "No matching records were returned. "
        "Check episode ID encoding or split values."
    )
    raise RuntimeError(
        "Traffic-feature alignment requires investigation."
    )

signal_frame11 = signal_frame11.rename(
    columns={
        episode_col11: "Episode_ID",
        minute_col11: "Minute"
    }
)

signal_frame11["Episode_ID"] = (
    signal_frame11["Episode_ID"]
    .astype(str)
    .str.strip()
)

signal_frame11["Minute"] = pd.to_numeric(
    signal_frame11["Minute"],
    errors="raise"
)

# Every returned record must belong to a validated episode.
if not set(signal_frame11["Episode_ID"]).issubset(
    set(episode_ids11)
):
    raise RuntimeError(
        "A record outside the selected validation IDs was returned."
    )

if split_col11 is not None:

    if not signal_frame11[split_col11].astype(
        str
    ).str.upper().eq("VALIDATION").all():
        raise RuntimeError(
            "Non-validation split record returned."
        )

if signal_frame11.duplicated(
    ["Episode_ID", "Minute"]
).any():
    raise RuntimeError(
        "Duplicate episode-minute feature records."
    )

# ------------------------------------------------------------
# E. Join validation keys and quantify coverage
# ------------------------------------------------------------

joined11 = reference_keys11.merge(
    signal_frame11,
    on=["Episode_ID", "Minute"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

match_count11 = int(
    joined11["_merge"].eq("both").sum()
)

coverage11 = (
    match_count11 / len(reference_keys11)
)

print("\nD. FEATURE / FORECAST ALIGNMENT")
print("-" * 90)
print("Reference validation rows:", len(reference_keys11))
print("Matched traffic-feature rows:", match_count11)
print("Alignment coverage:", f"{coverage11:.2%}")

print("\nPER-EPISODE COVERAGE")

for episode_id in episode_ids11:

    part = joined11.loc[
        joined11["Episode_ID"] == episode_id
    ]

    matched = int(
        part["_merge"].eq("both").sum()
    )

    print(
        f"{episode_id:<37} "
        f"{matched:>3}/{len(part):<3} "
        f"({matched/len(part):.1%})"
    )

print("\nE. ACTUAL TRAFFIC-SIGNAL STATISTICS")
print("-" * 90)

for feature in features11:

    values = pd.to_numeric(
        joined11[feature],
        errors="coerce"
    )

    valid = values.dropna()

    print(
        f"{feature:<28} "
        f"non-null={len(valid):>4} "
        f"min={valid.min() if len(valid) else 'N/A'} "
        f"median={valid.median() if len(valid) else 'N/A'} "
        f"max={valid.max() if len(valid) else 'N/A'}"
    )

print("\n" + "=" * 90)
print("CELL 11 COMPLETE")
print("=" * 90)
print("Requested validation episodes:", len(episode_ids11))
print("Alignment coverage:", f"{coverage11:.2%}")
print("Traffic features available:", features11)
print("Full unfiltered Parquet loaded: NO")
print("TEST records returned: NO")
print("Existing files modified: NO")
print("Models trained: 0")
print("New files written: 0")
print("=" * 90)


In [ ]:

# =====================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 12 — TRAFFIC SIGNAL + FOUR-MODEL FORECASTING ATLAS
#
# Five rows: Brute/Auth, DoS/DDoS, Exploit/App, Fuzzing, Recon/Scan
# Five columns: traffic signal + four forecasting architectures
#
# Observed signal: saved synthetic Packets_Per_Second
# Model predictions: aligned H5, H10, H15 probabilities
#
# Exports:
#   F08 figure: PDF, SVG, PNG
#   T09 underlying plotted values: CSV
#   Provenance: JSON
#
# NO retraining
# NO official TEST access
# NO source modifications
# NO existing figure overwrites
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

print("=" * 94)
print("CELL 12 — REFERENCE-STYLE TRAFFIC AND FORECAST ATLAS")
print("=" * 94)

# ---------------------------------------------------------------------
# A. VERIFY PREVIOUSLY AUDITED VARIABLES
# ---------------------------------------------------------------------

required12 = [
    "joined11",
    "selected6",
    "reference5",
    "REF_INDEX5",
    "TRUTH5",
    "MODEL_PROB5",
    "SIGNAL_FILE11",
    "SOURCES5"
]

for variable in required12:
    if variable not in globals():
        raise RuntimeError(
            f"Missing {variable}. "
            "Run the successful Cells 5, 6 and 11 first."
        )

PROJECT12 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUT12 = PROJECT12 / "Research_Outputs"
FIGURES12 = OUTPUT12 / "01_Figures" / "Forecasting"
TABLES12 = OUTPUT12 / "02_Tables"
MANIFESTS12 = OUTPUT12 / "05_Manifests"

for folder in [FIGURES12, TABLES12, MANIFESTS12]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

# We intentionally use four models to keep panels readable.
MODELS12 = [
    "Single LSTM",
    "Stacked LSTM",
    "CR-LSTM",
    "CR-TCN"
]

HORIZONS12 = [5, 10, 15]

for model in MODELS12:

    if model not in MODEL_PROB5:
        raise RuntimeError(f"Missing predictions for {model}")

    if not all(
        h in MODEL_PROB5[model]
        for h in HORIZONS12
    ):
        raise RuntimeError(
            f"{model}: missing one of H5/H10/H15."
        )

FAMILY_NAMES12 = {
    "BRUTE_AUTH": "Brute / Auth",
    "DOS_DDOS": "DoS / DDoS",
    "EXPLOIT_APP": "Exploit / App",
    "FUZZING": "Fuzzing",
    "RECON_SCAN": "Recon / Scan"
}

SOURCE_NAMES12 = {
    "CIC_IDS2017": "CIC-derived",
    "UNSW_NB15": "UNSW-derived"
}

# ---------------------------------------------------------------------
# B. VALIDATE MATCHED TRAFFIC FEATURE RECORDS
# ---------------------------------------------------------------------

signal12 = joined11.copy()

needed_signal12 = {
    "Episode_ID",
    "Minute",
    "Packets_Per_Second",
    "_merge"
}

if not needed_signal12.issubset(signal12.columns):
    raise RuntimeError(
        f"Missing traffic columns: "
        f"{needed_signal12 - set(signal12.columns)}"
    )

if not signal12["_merge"].astype(str).eq("both").all():
    raise RuntimeError(
        "Not every selected forecasting window "
        "has a matched traffic-feature record."
    )

signal12["Episode_ID"] = (
    signal12["Episode_ID"].astype(str).str.strip()
)

signal12["Minute"] = pd.to_numeric(
    signal12["Minute"],
    errors="raise"
)

signal12["Packets_Per_Second"] = pd.to_numeric(
    signal12["Packets_Per_Second"],
    errors="raise"
)

if signal12.duplicated(
    ["Episode_ID", "Minute"]
).any():
    raise RuntimeError(
        "Duplicate traffic-feature alignment keys."
    )

if not np.isfinite(
    signal12["Packets_Per_Second"]
).all():
    raise RuntimeError("Nonfinite packet-rate measurements.")

if (signal12["Packets_Per_Second"] < 0).any():
    raise RuntimeError("Negative packet-rate measurement.")

selected_ids12 = selected6[
    "Episode_ID"
].astype(str).tolist()

if len(selected_ids12) != 5:
    raise RuntimeError("Expected five selected episodes.")

if set(signal12["Episode_ID"]) != set(selected_ids12):
    raise RuntimeError(
        "Signal episode IDs differ from Cell 6 selection."
    )

signal12 = signal12.set_index(
    ["Episode_ID", "Minute"],
    verify_integrity=True
).sort_index()

# Compare selected traffic keys with the verified
# forecasting decision-window keys.

forecast_keys12 = REF_INDEX5[
    REF_INDEX5.get_level_values("Episode_ID").isin(
        selected_ids12
    )
]

if len(signal12.index.difference(forecast_keys12)):
    raise RuntimeError("Unexpected extra matched signal keys.")

if len(forecast_keys12.difference(signal12.index)):
    raise RuntimeError("Missing traffic-feature keys.")

if len(signal12) != len(forecast_keys12):
    raise RuntimeError("Traffic/prediction row-count mismatch.")

print("\nA. ALIGNMENT VERIFICATION")
print("-" * 94)

print("Selected attack-family episodes:", len(selected_ids12))
print("Verified common decision windows:", len(forecast_keys12))
print("Traffic-feature alignment: 100%")
print("Model trajectories:", len(MODELS12))
print("Horizons per model:", len(HORIZONS12))

# ---------------------------------------------------------------------
# C. VISUAL DESIGN
# ---------------------------------------------------------------------

TRAFFIC_COLOR12 = "#2876A8"

HORIZON_COLORS12 = {
    5: "#8655A4",
    10: "#25865C",
    15: "#DA8439"
}

HORIZON_STYLES12 = {
    5: (0, (2, 2)),
    10: "-",
    15: (0, (5, 2))
}

HORIZON_WIDTHS12 = {
    5: 1.25,
    10: 1.75,
    15: 1.35
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig12 = plt.figure(
    figsize=(23, 13.7),
    facecolor="white"
)

# First column deliberately wider, like your example.
grid12 = GridSpec(
    nrows=5,
    ncols=5,
    figure=fig12,
    width_ratios=[2.35, 1, 1, 1, 1],
    left=0.115,
    right=0.985,
    top=0.84,
    bottom=0.13,
    wspace=0.22,
    hspace=0.31
)

fig12.suptitle(
    "Traffic dynamics and multi-horizon network attack forecasts",
    x=0.115,
    y=0.976,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig12.text(
    0.115,
    0.944,
    "Aligned synthetic validation episodes | "
    "Five attack families | Four forecasting architectures",
    fontsize=10.7,
    color="#596678"
)

COLUMN_LABELS12 = [
    "Synthetic traffic signal",
    "Single LSTM",
    "Stacked LSTM",
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

def get_positive_spans12(minutes, truth):

    positive = np.sort(
        np.unique(
            np.asarray(minutes)[np.asarray(truth) == 1]
        )
    )

    if len(positive) == 0:
        return []

    breaks = np.flatnonzero(np.diff(positive) > 1)

    start_indices = np.r_[0, breaks + 1]
    end_indices = np.r_[breaks, len(positive) - 1]

    return [
        (
            float(positive[a]) - 0.45,
            float(positive[b]) + 0.45
        )
        for a, b in zip(start_indices, end_indices)
    ]

def style_axis12(ax, minutes, spans, risk_axis=True):

    ax.set_facecolor("#FBFCFD")

    for start, end in spans:
        ax.axvspan(
            start,
            end,
            facecolor="#E5BA70",
            alpha=0.23,
            linewidth=0,
            zorder=0
        )

    ax.set_xlim(
        float(np.min(minutes)),
        float(np.max(minutes))
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(nbins=4, integer=True)
    )

    ax.grid(
        axis="y",
        linestyle=":",
        linewidth=0.65,
        color="#A9B3BF",
        alpha=0.45
    )

    ax.set_axisbelow(True)

    ax.tick_params(
        axis="both",
        length=2,
        pad=3,
        labelsize=8
    )

    if risk_axis:

        ax.set_ylim(-0.025, 1.025)
        ax.set_yticks([0, 0.5, 1])

        # Reference only; not a deployment-tuned threshold.
        ax.axhline(
            0.5,
            linestyle=":",
            linewidth=0.8,
            color="#84909B",
            alpha=0.5,
            zorder=1
        )

# ---------------------------------------------------------------------
# D. PLOT EACH EPISODE WITH EXACT KEY ALIGNMENT
# ---------------------------------------------------------------------

plot_rows12 = []
axes12 = []

print("\nB. RENDERING ALIGNED ATTACK-FAMILY PANELS")
print("-" * 94)

for row_number, (_, episode) in enumerate(
    selected6.reset_index(drop=True).iterrows()
):

    episode_id = str(episode["Episode_ID"])
    family = str(episode["Family"])
    source = str(episode["Source"])

    selection_mask = (
        REF_INDEX5.get_level_values("Episode_ID")
        == episode_id
    )

    episode_minutes = (
        REF_INDEX5.get_level_values("Minute")
        .to_numpy()[selection_mask]
    )

    sort_order = np.argsort(episode_minutes)
    episode_minutes = episode_minutes[sort_order]

    reference_episode_index = REF_INDEX5[
        selection_mask
    ][sort_order]

    # Reindex feature data using the EXACT same keys.
    traffic_episode = signal12.reindex(
        reference_episode_index
    )

    pps = traffic_episode[
        "Packets_Per_Second"
    ].to_numpy(dtype=float)

    if not np.isfinite(pps).all():
        raise RuntimeError(
            f"Unmatched traffic signal: {episode_id}"
        )

    actual12 = {
        h: TRUTH5[h][selection_mask][sort_order]
        for h in HORIZONS12
    }

    spans = get_positive_spans12(
        episode_minutes,
        actual12[10]
    )

    row_axes = [
        fig12.add_subplot(grid12[row_number, col])
        for col in range(5)
    ]

    axes12.append(row_axes)

    # ---------------------------------------------------------
    # FIRST COLUMN: real saved synthetic traffic measurements
    # ---------------------------------------------------------

    ax_traffic = row_axes[0]

    style_axis12(
        ax_traffic,
        episode_minutes,
        spans,
        risk_axis=False
    )

    ax_traffic.plot(
        episode_minutes,
        pps,
        color=TRAFFIC_COLOR12,
        linewidth=1.35,
        alpha=0.95,
        zorder=4
    )

    ax_traffic.fill_between(
        episode_minutes,
        pps,
        0,
        color=TRAFFIC_COLOR12,
        alpha=0.055,
        zorder=2
    )

    # Raw packet rates are strictly positive here.
    # Use a logarithmic axis for comparable orders of
    # magnitude without inventing normalization.
    ax_traffic.set_yscale("log")

    ax_traffic.set_ylabel(
        "Packets / second",
        fontsize=9.5,
        labelpad=7
    )

    row_name = FAMILY_NAMES12.get(family, family)
    source_name = SOURCE_NAMES12.get(source, source)

    ax_traffic.annotate(
        f"{row_name}\n"
        f"{source_name}\n"
        f"n = {len(episode_minutes)}",
        xy=(0, 0.5),
        xycoords="axes fraction",
        xytext=(-0.20, 0.5),
        textcoords="axes fraction",
        ha="right",
        va="center",
        color="#374353",
        fontsize=9,
        fontweight="medium",
        annotation_clip=False
    )

    # ---------------------------------------------------------
    # REMAINING COLUMNS: real model probabilities
    # ---------------------------------------------------------

    for col_number, model in enumerate(MODELS12, start=1):

        ax = row_axes[col_number]

        style_axis12(
            ax,
            episode_minutes,
            spans,
            risk_axis=True
        )

        for horizon in HORIZONS12:

            probabilities = (
                MODEL_PROB5[model][horizon]
                [selection_mask][sort_order]
            )

            if not np.isfinite(probabilities).all():
                raise RuntimeError(
                    f"{model}: invalid probabilities."
                )

            ax.plot(
                episode_minutes,
                probabilities,
                linestyle=HORIZON_STYLES12[horizon],
                linewidth=HORIZON_WIDTHS12[horizon],
                color=HORIZON_COLORS12[horizon],
                alpha=0.95,
                zorder=4
            )

            for minute, predicted, true_label in zip(
                episode_minutes,
                probabilities,
                actual12[horizon]
            ):

                plot_rows12.append({
                    "Episode_ID": episode_id,
                    "Source": source,
                    "Family": family,
                    "Minute": float(minute),
                    "Packets_Per_Second": float(
                        pps[np.where(
                            episode_minutes == minute
                        )[0][0]]
                    ),
                    "Model": model,
                    "Horizon": f"H{horizon}",
                    "Predicted_Probability": float(predicted),
                    "True_Label": int(true_label)
                })

        if col_number == 1:

            ax.set_ylabel(
                "Future attack risk",
                fontsize=9,
                labelpad=6
            )

    if row_number == 0:

        for ax, heading in zip(
            row_axes,
            COLUMN_LABELS12
        ):

            ax.set_title(
                heading,
                fontsize=11,
                fontweight="bold",
                pad=13
            )

    if row_number == 4:

        for ax in row_axes:
            ax.set_xlabel(
                "Decision minute",
                fontsize=9,
                labelpad=7
            )

    print(
        f"PASS | {family:<13} "
        f"| {source:<12} "
        f"| {len(episode_minutes):>3} aligned minutes"
    )

# ---------------------------------------------------------------------
# E. LEGEND AND FIGURE CAPTION
# ---------------------------------------------------------------------

legend12 = [
    Line2D(
        [0], [0],
        color=TRAFFIC_COLOR12,
        linewidth=2,
        label="Synthetic packet rate"
    )
]

for horizon in HORIZONS12:

    legend12.append(
        Line2D(
            [0], [0],
            color=HORIZON_COLORS12[horizon],
            linestyle=HORIZON_STYLES12[horizon],
            linewidth=2,
            label=f"H{horizon} forecast"
        )
    )

legend12.append(
    Patch(
        facecolor="#E5BA70",
        alpha=0.35,
        label="Positive H10 target"
    )
)

fig12.legend(
    handles=legend12,
    loc="upper left",
    bbox_to_anchor=(0.115, 0.918),
    ncol=5,
    frameon=False,
    fontsize=10,
    columnspacing=2.0,
    handlelength=3.0
)

fig12.text(
    0.115,
    0.059,
    "Synthetic NAF v2 development validation. "
    "Left: measured synthetic packet-rate feature "
    "(logarithmic y-axis). Right: predicted probability "
    "of an attack within 5, 10 or 15 minutes.",
    fontsize=9.1,
    color="#566575"
)

fig12.text(
    0.115,
    0.041,
    "All panels use matched Episode_ID + Minute keys. "
    "Gold shading denotes positive H10 forecast targets, "
    "not an exact attack-onset timestamp. "
    "Episodes selected using labels, not model scores.",
    fontsize=9.1,
    color="#566575"
)

# ---------------------------------------------------------------------
# F. EXPORT FIGURE, VALUES AND PROVENANCE
# ---------------------------------------------------------------------

RUN12 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM12 = (
    f"F08_Traffic_And_Forecasting_Atlas_{RUN12}"
)

saved_figures12 = []

for ext in ["pdf", "svg", "png"]:

    path = FIGURES12 / f"{STEM12}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig12.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    saved_figures12.append(path)

plt.show()
plt.close(fig12)

# Each CSV row represents one model/horizon/minute.
plot_data12 = pd.DataFrame(plot_rows12)

table_path12 = TABLES12 / (
    f"T09_F08_Underlying_Plot_Data_{RUN12}.csv"
)

plot_data12.to_csv(
    table_path12,
    index=False,
    mode="x",
    float_format="%.8f"
)

def hash_file12(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

source_paths12 = {
    "Synthetic traffic measurements": Path(SIGNAL_FILE11)
}

for model in MODELS12:
    source_paths12[model] = Path(SOURCES5[model])

manifest12 = {
    "project": "NetGuard",
    "figure": STEM12,
    "benchmark": "Synthetic_NAF_v2",
    "evaluation": "development_validation",
    "figure_layout": "5 rows x 5 columns",
    "traffic_measurement": "Packets_Per_Second",
    "traffic_axis": "logarithmic",
    "forecast_horizons": HORIZONS12,
    "models": MODELS12,
    "aligned_decision_windows": int(
        len(forecast_keys12)
    ),
    "episodes": selected6[
        ["Episode_ID", "Source", "Family", "Windows"]
    ].to_dict("records"),
    "source_files": [
        {
            "label": name,
            "path": str(
                path.relative_to(PROJECT12)
            ),
            "sha256": hash_file12(path)
        }
        for name, path in source_paths12.items()
    ],
    "generated_files": [
        {
            "path": str(path.relative_to(PROJECT12)),
            "sha256": hash_file12(path)
        }
        for path in saved_figures12 + [table_path12]
    ],
    "limitations": [
        "Traffic features are synthetically generated.",
        "The traffic signal is not a real captured network trace.",
        "Only matched validation decision minutes are shown.",
        "The H10 positive band is not exact attack onset.",
        "Episodes are illustrative and not aggregate model evaluation.",
        "No official TEST data was accessed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

manifest_path12 = MANIFESTS12 / (
    f"traffic_forecasting_atlas_{RUN12}.json"
)

with manifest_path12.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest12,
        handle,
        indent=2,
        default=str,
        allow_nan=False
    )

# ---------------------------------------------------------------------
# G. VERIFY FINAL OUTPUTS
# ---------------------------------------------------------------------

print("\n" + "=" * 94)
print("CELL 12 COMPLETE")
print("=" * 94)

print("Attack-family episodes:", len(selected6))
print("Forecasting models:", len(MODELS12))
print("Figure panels:", 25)
print("Verified traffic windows:", len(forecast_keys12))
print("Underlying plot-data rows:", len(plot_data12))

print("\nSAVED FILES")
print("-" * 94)

for path in saved_figures12 + [
    table_path12,
    manifest_path12
]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print("PASS:", path.relative_to(OUTPUT12))

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Synthetic source files modified: NO")
print("Forecasting checkpoints modified: NO")
print("Existing research outputs overwritten: NO")

print("=" * 94)


In [ ]:

# =====================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 13 — MULTI-MODEL PRECISION–RECALL BENCHMARK
#
# FIGURE F09
#   A, C, E: Precision–Recall curves for H5 / H10 / H15
#   B, D, F: Model AP rankings for H5 / H10 / H15
#
# Seven models:
#   Single LSTM, Stacked LSTM, XGBoost V2,
#   Competing-Risk LSTM, Competing-Risk TCN,
#   GCN-LSTM, GAT-LSTM
#
# Uses Cell 5's VERIFIED aligned validation predictions.
#
# Writes NEW:
#   01_Figures/Model_Comparisons/ -> PDF / SVG / PNG
#   02_Tables/ -> AP summary + PR curve coordinates
#   03_Metrics/ -> benchmark JSON
#   05_Manifests/ -> provenance JSON
#
# SAFETY:
#   No retraining
#   No official TEST access
#   No source-file modifications
#   No overwrites
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
)

print("=" * 96)
print("CELL 13 — SEVEN-MODEL PRECISION–RECALL BENCHMARK")
print("=" * 96)

# ---------------------------------------------------------------------
# 1. VERIFY CELL 5'S ALIGNED VALIDATION RESULTS
# ---------------------------------------------------------------------

required13 = [
    "MODEL_PROB5",
    "TRUTH5",
    "REF_INDEX5",
    "SOURCES5"
]

for variable in required13:

    if variable not in globals():
        raise RuntimeError(
            f"Missing {variable}. "
            "Run the successful Cell 5 first."
        )

PROJECT13 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUT13 = PROJECT13 / "Research_Outputs"

FIGURES13 = (
    OUTPUT13
    / "01_Figures"
    / "Model_Comparisons"
)

TABLES13 = OUTPUT13 / "02_Tables"
METRICS13 = OUTPUT13 / "03_Metrics"
MANIFESTS13 = OUTPUT13 / "05_Manifests"

for folder in [
    FIGURES13,
    TABLES13,
    METRICS13,
    MANIFESTS13
]:

    if not folder.is_dir():
        raise FileNotFoundError(folder)

MODELS13 = [
    "Single LSTM",
    "Stacked LSTM",
    "XGBoost V2",
    "CR-LSTM",
    "CR-TCN",
    "GCN-LSTM",
    "GAT-LSTM"
]

HORIZONS13 = [5, 10, 15]

COLORS13 = {
    "Single LSTM": "#4876A6",
    "Stacked LSTM": "#254D89",
    "XGBoost V2": "#C27C3B",
    "CR-LSTM": "#17877D",
    "CR-TCN": "#9857A5",
    "GCN-LSTM": "#708B45",
    "GAT-LSTM": "#C65D75"
}

for model in MODELS13:

    if model not in MODEL_PROB5:
        raise RuntimeError(
            f"Missing aligned predictions for {model}"
        )

if len(REF_INDEX5) != 18839:
    raise RuntimeError(
        "Unexpected validation window count."
    )

if len(set(REF_INDEX5.tolist())) != len(REF_INDEX5):
    raise RuntimeError(
        "Duplicate validation keys detected."
    )

# Check that H15 is genuinely absent from XGBoost.
if 15 in MODEL_PROB5["XGBoost V2"]:
    raise RuntimeError(
        "Unexpected XGBoost H15 predictions. "
        "Review saved metadata before proceeding."
    )

print("\nA. VALIDATION INPUTS")
print("-" * 96)

print("Aligned validation windows:", len(REF_INDEX5))
print("Models:", len(MODELS13))
print("Horizons:", HORIZONS13)
print("XGBoost H15: NOT AVAILABLE")

# ---------------------------------------------------------------------
# 2. VALIDATE TARGETS AND RECOMPUTE PR CURVES
# ---------------------------------------------------------------------

BENCHMARK_ROWS13 = []
CURVE_ROWS13 = []
CURVES13 = {}

for horizon in HORIZONS13:

    y_true = np.asarray(
        TRUTH5[horizon]
    ).reshape(-1)

    if len(y_true) != len(REF_INDEX5):
        raise RuntimeError(
            f"H{horizon}: target length mismatch."
        )

    if not np.isin(y_true, [0, 1]).all():
        raise RuntimeError(
            f"H{horizon}: nonbinary ground truth."
        )

    y_true = y_true.astype(np.int8)

    positives = int(y_true.sum())
    negatives = int(len(y_true) - positives)

    if positives == 0 or negatives == 0:
        raise RuntimeError(
            f"H{horizon}: both classes required."
        )

    prevalence = positives / len(y_true)

    for model in MODELS13:

        if horizon not in MODEL_PROB5[model]:

            BENCHMARK_ROWS13.append({
                "Model": model,
                "Horizon": f"H{horizon}",
                "Average_Precision": np.nan,
                "Positive_Windows": positives,
                "Negative_Windows": negatives,
                "Prevalence": prevalence,
                "AP_Over_Prevalence": np.nan,
                "Available": False
            })

            continue

        probabilities = np.asarray(
            MODEL_PROB5[model][horizon],
            dtype=float
        ).reshape(-1)

        if len(probabilities) != len(y_true):
            raise RuntimeError(
                f"{model}/H{horizon}: prediction length mismatch."
            )

        if not np.isfinite(probabilities).all():
            raise RuntimeError(
                f"{model}/H{horizon}: invalid probability."
            )

        if (
            (probabilities < -1e-8).any()
            or (probabilities > 1 + 1e-8).any()
        ):
            raise RuntimeError(
                f"{model}/H{horizon}: probability outside [0,1]."
            )

        probabilities = np.clip(
            probabilities, 0, 1
        )

        # AP is NOT the trapezoidal PR AUC.
        ap = float(
            average_precision_score(
                y_true,
                probabilities
            )
        )

        precision, recall, thresholds = (
            precision_recall_curve(
                y_true,
                probabilities
            )
        )

        # sklearn returns recall in descending order.
        # Reverse for plotting left-to-right.
        plot_recall = recall[::-1]
        plot_precision = precision[::-1]

        if not np.all(
            np.diff(plot_recall) >= -1e-10
        ):
            raise RuntimeError(
                f"{model}/H{horizon}: invalid recall curve."
            )

        CURVES13[(model, horizon)] = {
            "recall": plot_recall,
            "precision": plot_precision,
            "ap": ap,
            "prevalence": prevalence
        }

        BENCHMARK_ROWS13.append({
            "Model": model,
            "Horizon": f"H{horizon}",
            "Average_Precision": ap,
            "Positive_Windows": positives,
            "Negative_Windows": negatives,
            "Prevalence": prevalence,
            "AP_Over_Prevalence": ap / prevalence,
            "Available": True
        })

        # Save every genuine curve coordinate.
        for k, (r, p) in enumerate(
            zip(plot_recall, plot_precision)
        ):

            CURVE_ROWS13.append({
                "Model": model,
                "Horizon": f"H{horizon}",
                "Point_Index": k,
                "Recall": float(r),
                "Precision": float(p)
            })

BENCHMARK13 = pd.DataFrame(
    BENCHMARK_ROWS13
)

PR_POINTS13 = pd.DataFrame(
    CURVE_ROWS13
)

print("\nB. RECOMPUTED AVERAGE PRECISION")
print("-" * 96)

pivot13 = BENCHMARK13.pivot(
    index="Model",
    columns="Horizon",
    values="Average_Precision"
).reindex(MODELS13)

print(
    pivot13[
        ["H5", "H10", "H15"]
    ].round(6).to_string()
)

print("\nEVENT PREVALENCE")

for h in HORIZONS13:

    subset = BENCHMARK13.loc[
        BENCHMARK13["Horizon"] == f"H{h}"
    ]

    print(
        f"H{h:<2} | "
        f"positive={int(subset['Positive_Windows'].iloc[0]):,} "
        f"| prevalence={subset['Prevalence'].iloc[0]:.4%}"
    )

# ---------------------------------------------------------------------
# 3. OPTIONAL CONSISTENCY COMPARISON WITH SAVED BENCHMARK
#
# This is diagnostic only:
# do not silently overwrite newly recomputed AP.
# ---------------------------------------------------------------------

AUDIT13 = (
    PROJECT13
    / "Synthetic_NAF_v2"
    / "models"
    / "FINAL_VALIDATION_AUDIT"
)

comparison_path13 = (
    AUDIT13
    / "FOUR_MODEL_VALIDATION_COMPARISON.csv"
)

NAME_MAP13 = {
    "XGBoost V2": "XGBoost V2",
    "Stacked LSTM": "Stacked Horizon LSTM V2",
    "CR-LSTM": "Competing-Risk LSTM",
    "CR-TCN": "Competing-Risk TCN"
}

COMPARISON_WARNINGS13 = []

if comparison_path13.is_file():

    historical13 = pd.read_csv(
        comparison_path13
    ).set_index("Model")

    print("\nC. SAVED BENCHMARK CONSISTENCY")
    print("-" * 96)

    for model, report_name in NAME_MAP13.items():

        if report_name not in historical13.index:
            continue

        for horizon in HORIZONS13:

            metric = f"H{horizon}_AP"

            if metric not in historical13.columns:
                continue

            reported = pd.to_numeric(
                historical13.loc[
                    report_name,
                    metric
                ],
                errors="coerce"
            )

            if pd.isna(reported):
                continue

            current = CURVES13.get(
                (model, horizon)
            )

            if current is None:
                continue

            measured = current["ap"]
            difference = measured - float(reported)

            status = (
                "MATCH"
                if abs(difference) <= 5e-4
                else "REVIEW"
            )

            print(
                f"{model:<17} H{horizon:<2} "
                f"reported={reported:.6f} "
                f"recomputed={measured:.6f} "
                f"{status}"
            )

            if status == "REVIEW":

                COMPARISON_WARNINGS13.append({
                    "model": model,
                    "horizon": horizon,
                    "reported": float(reported),
                    "recomputed": measured,
                    "difference": difference
                })

# ---------------------------------------------------------------------
# 4. PUBLICATION FIGURE STYLE
# ---------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig13, axes13 = plt.subplots(
    3, 2,
    figsize=(17, 14),
    gridspec_kw={
        "width_ratios": [1.28, 1],
        "wspace": 0.29,
        "hspace": 0.40
    }
)

fig13.suptitle(
    "Precision–recall discrimination across forecasting architectures",
    x=0.085,
    y=0.986,
    ha="left",
    fontsize=18.5,
    fontweight="bold"
)

fig13.text(
    0.085,
    0.957,
    "Seven architectures  |  "
    "H5, H10 and H15 event forecasting  |  "
    "18,839 aligned validation windows",
    fontsize=10.5,
    color="#596678"
)

# ---------------------------------------------------------------------
# 5. RENDER THREE PR CURVES + THREE RANKING PANELS
# ---------------------------------------------------------------------

LETTERS13 = "ABCDEF"

for row, horizon in enumerate(HORIZONS13):

    horizon_name = f"H{horizon}"

    ax_pr = axes13[row, 0]
    ax_rank = axes13[row, 1]

    subset = BENCHMARK13.loc[
        (BENCHMARK13["Horizon"] == horizon_name)
        & (BENCHMARK13["Available"])
    ]

    prevalence = float(
        subset["Prevalence"].iloc[0]
    )

    # ---------------------------------------------------------
    # LEFT: FULL PRECISION–RECALL CURVES
    # ---------------------------------------------------------

    ax_pr.set_facecolor("#FAFBFC")

    for model in MODELS13:

        data = CURVES13.get(
            (model, horizon)
        )

        if data is None:
            continue

        ax_pr.plot(
            data["recall"],
            data["precision"],
            color=COLORS13[model],
            linewidth=1.85,
            alpha=0.92,
            label=model,
            zorder=3
        )

    # Random-ranking baseline equals prevalence.
    ax_pr.axhline(
        prevalence,
        color="#5A6573",
        linestyle=(0, (4, 3)),
        linewidth=1.25,
        label=f"Prevalence = {prevalence:.3f}",
        zorder=1
    )

    ax_pr.set_xlim(0, 1.0)
    ax_pr.set_ylim(0, 1.02)

    ax_pr.set_xticks(
        np.linspace(0, 1, 6)
    )

    ax_pr.set_yticks(
        np.linspace(0, 1, 6)
    )

    ax_pr.grid(
        linestyle=":",
        linewidth=0.7,
        alpha=0.35
    )

    ax_pr.set_axisbelow(True)

    ax_pr.set_title(
        f"{LETTERS13[row*2]}   "
        f"{horizon_name}: Precision–recall curves",
        loc="left",
        fontweight="bold",
        pad=13
    )

    ax_pr.set_xlabel(
        "Recall",
        labelpad=8
    )

    ax_pr.set_ylabel(
        "Precision",
        labelpad=8
    )

    ax_pr.text(
        0.03,
        0.055,
        f"Positive windows: "
        f"{int(subset['Positive_Windows'].iloc[0]):,}",
        transform=ax_pr.transAxes,
        fontsize=9,
        color="#576472",
        bbox={
            "facecolor": "white",
            "edgecolor": "#DEE4EA",
            "boxstyle": "round,pad=0.4"
        }
    )

    # ---------------------------------------------------------
    # RIGHT: MODEL AP COMPARISON
    # ---------------------------------------------------------

    ranking = (
        subset
        .sort_values("Average_Precision")
        .reset_index(drop=True)
    )

    y_positions = np.arange(
        len(ranking)
    )

    bars = ax_rank.barh(
        y_positions,
        ranking["Average_Precision"],
        color=[
            COLORS13[name]
            for name in ranking["Model"]
        ],
        height=0.64,
        alpha=0.90,
        edgecolor="white",
        linewidth=0.6
    )

    ax_rank.axvline(
        prevalence,
        color="#5A6573",
        linestyle=(0, (4, 3)),
        linewidth=1.2,
        zorder=2
    )

    ax_rank.set_yticks(y_positions)

    ax_rank.set_yticklabels(
        ranking["Model"],
        fontsize=9.3
    )

    ax_rank.set_xlim(0, 1.10)
    ax_rank.set_xticks(
        np.linspace(0, 1, 6)
    )

    ax_rank.set_xlabel(
        "Average precision (AP)",
        labelpad=8
    )

    ax_rank.set_title(
        f"{LETTERS13[row*2+1]}   "
        f"{horizon_name}: Model AP ranking",
        loc="left",
        fontweight="bold",
        pad=13
    )

    ax_rank.set_facecolor("#FAFBFC")

    ax_rank.grid(
        axis="x",
        linestyle=":",
        alpha=0.35
    )

    ax_rank.set_axisbelow(True)

    ax_rank.tick_params(
        axis="y",
        length=0,
        pad=8
    )

    for bar, value in zip(
        bars,
        ranking["Average_Precision"]
    ):

        ax_rank.text(
            value + 0.014,
            bar.get_y() + bar.get_height()/2,
            f"{value:.4f}",
            fontsize=9,
            va="center",
            color="#253347"
        )

    if horizon == 15:

        ax_rank.text(
            0.97, 0.03,
            "XGBoost H15: N/A",
            transform=ax_rank.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            color="#98633E"
        )

# ---------------------------------------------------------------------
# 6. SHARED LEGEND & EXPLANATORY NOTES
# ---------------------------------------------------------------------

from matplotlib.lines import Line2D

legend_handles13 = [
    Line2D(
        [0], [0],
        color=COLORS13[model],
        linewidth=2,
        label=model
    )
    for model in MODELS13
]

legend_handles13.append(
    Line2D(
        [0], [0],
        color="#5A6573",
        linestyle=(0, (4, 3)),
        linewidth=1.5,
        label="Class-prevalence baseline"
    )
)

fig13.legend(
    handles=legend_handles13,
    loc="upper left",
    bbox_to_anchor=(0.085, 0.935),
    ncol=4,
    frameon=False,
    fontsize=9.5,
    handlelength=2.3,
    columnspacing=1.8
)

fig13.subplots_adjust(
    left=0.105,
    right=0.96,
    top=0.865,
    bottom=0.105
)

fig13.text(
    0.085,
    0.052,
    "Synthetic NAF v2 development validation. "
    "PR curves and AP are recomputed from aligned saved "
    "predictions, using sklearn average_precision_score. "
    "AP is not overall accuracy.",
    fontsize=9,
    color="#586574"
)

fig13.text(
    0.085,
    0.035,
    "Horizons have different event prevalences. "
    "XGBoost V2 has no saved H15 forecast. "
    "Validation windows overlap in time; confidence "
    "intervals are not implied.",
    fontsize=9,
    color="#586574"
)

# ---------------------------------------------------------------------
# 7. SAVE RESEARCH FIGURE
# ---------------------------------------------------------------------

RUN13 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM13 = (
    f"F09_Seven_Model_Precision_Recall_{RUN13}"
)

FIGURE_FILES13 = []

for ext in ["pdf", "svg", "png"]:

    path = FIGURES13 / f"{STEM13}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig13.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES13.append(path)

plt.show()
plt.close(fig13)

# ---------------------------------------------------------------------
# 8. SAVE RESEARCH TABLES
# ---------------------------------------------------------------------

AP_TABLE13 = (
    TABLES13
    / f"T10_Seven_Model_AP_Benchmark_{RUN13}.csv"
)

CURVE_TABLE13 = (
    TABLES13
    / f"T11_Precision_Recall_Curve_Points_{RUN13}.csv.gz"
)

BENCHMARK13.to_csv(
    AP_TABLE13,
    index=False,
    mode="x",
    float_format="%.8f"
)

# Compression keeps full reproducible PR coordinates
# without creating an unnecessarily huge plain CSV.

PR_POINTS13.to_csv(
    CURVE_TABLE13,
    index=False,
    mode="x",
    compression="gzip",
    float_format="%.8f"
)

# ---------------------------------------------------------------------
# 9. SAVE MACHINE-READABLE METRICS
# ---------------------------------------------------------------------

METRICS_PATH13 = (
    METRICS13
    / f"seven_model_pr_metrics_{RUN13}.json"
)

records13 = []

for _, row in BENCHMARK13.iterrows():

    records13.append({
        "model": str(row["Model"]),
        "horizon": str(row["Horizon"]),
        "average_precision": (
            float(row["Average_Precision"])
            if pd.notna(row["Average_Precision"])
            else None
        ),
        "positive_windows": int(row["Positive_Windows"]),
        "negative_windows": int(row["Negative_Windows"]),
        "prevalence": float(row["Prevalence"]),
        "available": bool(row["Available"])
    })

metrics_data13 = {
    "project": "NetGuard",
    "benchmark": "Synthetic_NAF_v2",
    "split": "development_validation",
    "aligned_windows": int(len(REF_INDEX5)),
    "models": MODELS13,
    "horizons": HORIZONS13,
    "metric": "average_precision_score",
    "results": records13,
    "historical_comparison_warnings":
        COMPARISON_WARNINGS13,
    "official_test_accessed": False,
    "models_retrained": 0
}

with METRICS_PATH13.open(
    "x",
    encoding="utf-8"
) as file:

    json.dump(
        metrics_data13,
        file,
        indent=2,
        allow_nan=False
    )

# ---------------------------------------------------------------------
# 10. REPRODUCIBILITY MANIFEST
# ---------------------------------------------------------------------

def sha256_13(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

created13 = (
    FIGURE_FILES13
    + [
        AP_TABLE13,
        CURVE_TABLE13,
        METRICS_PATH13
    ]
)

source_items13 = []

for model in MODELS13:

    path = Path(SOURCES5[model])

    source_items13.append({
        "model": model,
        "path": str(path.relative_to(PROJECT13)),
        "sha256": sha256_13(path)
    })

manifest_data13 = {
    "project": "NetGuard",
    "figure": STEM13,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "evaluation": "Synthetic_NAF_v2 development validation",
    "alignment_keys": [
        "Episode_ID",
        "Minute"
    ],
    "alignment_verified_in": "Cell 5",
    "metric_definition": (
        "Average precision computed from sklearn "
        "average_precision_score on saved probabilities"
    ),
    "input_prediction_files": source_items13,
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT13)),
            "sha256": sha256_13(p)
        }
        for p in created13
    ],
    "scientific_limitations": [
        "Synthetic attack precursor dynamics.",
        "Overlapping validation windows.",
        "AP is distinct from accuracy.",
        "Class prevalence differs across forecasting horizons.",
        "No XGBoost H15 prediction available.",
        "Curves are descriptive, without uncertainty intervals.",
        "Generalization to live traffic is unestablished."
    ],
    "official_test_accessed": False,
    "models_retrained": 0
}

MANIFEST_PATH13 = (
    MANIFESTS13
    / f"precision_recall_manifest_{RUN13}.json"
)

with MANIFEST_PATH13.open(
    "x",
    encoding="utf-8"
) as file:

    json.dump(
        manifest_data13,
        file,
        indent=2,
        allow_nan=False
    )

# ---------------------------------------------------------------------
# 11. FINAL VERIFICATION
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("CELL 13 COMPLETE")
print("=" * 96)

print("Forecasting models:", len(MODELS13))
print("Prediction horizons:", len(HORIZONS13))
print("PR curves generated:", len(CURVES13))
print("Figure panels:", 6)

print("\nBEST MODEL PER HORIZON")
print("-" * 96)

for horizon in HORIZONS13:

    best = (
        BENCHMARK13.loc[
            (BENCHMARK13["Horizon"] == f"H{horizon}")
            & BENCHMARK13["Available"]
        ]
        .sort_values(
            "Average_Precision",
            ascending=False
        )
        .iloc[0]
    )

    print(
        f"H{horizon:<2} | "
        f"{best['Model']:<18} "
        f"AP={best['Average_Precision']:.6f}"
    )

print("\nSAVED RESEARCH OUTPUTS")
print("-" * 96)

for path in created13 + [MANIFEST_PATH13]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        "PASS:",
        path.relative_to(OUTPUT13)
    )

print("\nHistorical comparison warnings:",
      len(COMPARISON_WARNINGS13))

print("Models retrained: 0")
print("Official TEST accessed: NO")
print("Saved predictions modified: NO")
print("Forecast checkpoints modified: NO")
print("Existing outputs overwritten: NO")

print("=" * 96)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 14 — TIME-TO-ATTACK (ETA) EVALUATION
#
# F10: Six-panel conditional ETA analysis
# Uses SAVED validation predictions from CR-LSTM and CR-TCN.
#
# A/B: Actual vs predicted conditional ETA, density
# C:   Absolute-error cumulative distribution
# D:   MAE by true time-to-attack
# E:   Signed prediction bias by true time-to-attack
# F:   Source-specific conditional ETA MAE
#
# Checks raw predictions against independent saved reports.
# No training / no TEST access / no overwriting.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

print("=" * 90)
print("CELL 14 — CONDITIONAL TIME-TO-ATTACK EVALUATION")
print("=" * 90)

PROJECT14 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

ROOT14 = PROJECT14 / "Synthetic_NAF_v2" / "models"
OUT14 = PROJECT14 / "Research_Outputs"

FIG14 = OUT14 / "01_Figures" / "Forecasting"
TAB14 = OUT14 / "02_Tables"
MET14 = OUT14 / "03_Metrics"
MAN14 = OUT14 / "05_Manifests"

for folder in [FIG14, TAB14, MET14, MAN14]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

SOURCES14 = {
    "CR-LSTM": ROOT14 / "LSTM_V2_CompetingRisk"
                     / "official_validation",
    "CR-TCN": ROOT14 / "TCN_V2_CompetingRisk"
                    / "official_validation"
}

COLORS14 = {
    "CR-LSTM": "#16817F",
    "CR-TCN": "#9257A4"
}

CMAPS14 = {
    "CR-LSTM": "GnBu",
    "CR-TCN": "Purples"
}

# ---------------------------------------------------------------
# A. LOAD SAVED PREDICTIONS AND INDEPENDENT REPORTS
# ---------------------------------------------------------------

TABLES14 = {}
MINUTE_REPORTS14 = {}
SOURCE_REPORTS14 = {}
INPUT_FILES14 = []

for model, folder in SOURCES14.items():

    pred_path = folder / "VALIDATION_PREDICTIONS.csv"
    minute_path = folder / "ETA_BY_TRUE_MINUTE.csv"
    source_path = folder / "SOURCE_FAMILY_ETA_METRICS.csv"

    for path in [pred_path, minute_path, source_path]:
        if not path.is_file():
            raise FileNotFoundError(path)
        INPUT_FILES14.append(path)

    df = pd.read_csv(pred_path)

    required = {
        "Episode_ID",
        "Minute",
        "Source",
        "True_Onset_Minute",
        "True_H15",
        "Expected_ETA_If_Event",
        "Predicted_Onset_Minute_If_Event"
    }

    if not required.issubset(df.columns):
        raise RuntimeError(
            f"{model}: missing columns {required - set(df.columns)}"
        )

    if len(df) != 18839:
        raise RuntimeError(
            f"{model}: unexpected validation row count"
        )

    df["Episode_ID"] = (
        df["Episode_ID"].astype(str).str.strip()
    )

    numeric = [
        "Minute",
        "True_Onset_Minute",
        "True_H15",
        "Expected_ETA_If_Event",
        "Predicted_Onset_Minute_If_Event"
    ]

    for col in numeric:
        df[col] = pd.to_numeric(df[col], errors="raise")

    if not np.isfinite(df[numeric].to_numpy()).all():
        raise RuntimeError(f"{model}: nonfinite values")

    if df.duplicated(["Episode_ID", "Minute"]).any():
        raise RuntimeError(f"{model}: duplicated observation keys")

    if not np.isin(
        df["True_Onset_Minute"].to_numpy(),
        np.arange(16)
    ).all():
        raise RuntimeError(
            f"{model}: invalid true onset-minute values"
        )

    if not np.array_equal(
        (df["True_Onset_Minute"].to_numpy() > 0).astype(int),
        df["True_H15"].to_numpy().astype(int)
    ):
        raise RuntimeError(
            f"{model}: H15 event indicator mismatch"
        )

    if not df["Expected_ETA_If_Event"].between(1, 15).all():
        raise RuntimeError(
            f"{model}: ETA predictions outside [1,15]"
        )

    df = df.set_index(
        ["Episode_ID", "Minute"],
        verify_integrity=True
    ).sort_index()

    TABLES14[model] = df
    MINUTE_REPORTS14[model] = pd.read_csv(minute_path)
    SOURCE_REPORTS14[model] = pd.read_csv(source_path)

# Verify identical keys and actual labels.
lstm14 = TABLES14["CR-LSTM"]
tcn14 = TABLES14["CR-TCN"]

if not lstm14.index.equals(tcn14.index):
    raise RuntimeError("LSTM and TCN observation keys differ")

for col in ["Source", "True_Onset_Minute", "True_H15"]:

    if not np.array_equal(
        lstm14[col].to_numpy(),
        tcn14[col].to_numpy()
    ):
        raise RuntimeError(
            f"Ground-truth mismatch: {col}"
        )

if "REF_INDEX5" in globals():

    if not lstm14.index.equals(REF_INDEX5):
        raise RuntimeError(
            "Cell 5 alignment differs from ETA validation"
        )

print("\nA. VALIDATION ALIGNMENT")
print("-" * 90)

print("Aligned validation windows:", len(lstm14))
print("Models:", list(TABLES14))

# ---------------------------------------------------------------
# B. ISOLATE TRUE FUTURE-EVENT WINDOWS
#
# ETA is CONDITIONAL on a true event within 15 minutes.
# Do not evaluate ETA on no-event windows.
# ---------------------------------------------------------------

EVENT14 = {}
PER_MINUTE14 = {}
PER_SOURCE14 = {}
STATS14 = []

for model, df in TABLES14.items():

    event = df.loc[
        df["True_Onset_Minute"] > 0
    ].copy()

    event["True_ETA"] = (
        event["True_Onset_Minute"].astype(float)
    )

    event["Predicted_ETA"] = (
        event["Expected_ETA_If_Event"].astype(float)
    )

    event["Signed_Error"] = (
        event["Predicted_ETA"] - event["True_ETA"]
    )

    event["Absolute_Error"] = (
        event["Signed_Error"].abs()
    )

    EVENT14[model] = event

    by_minute = (
        event
        .reset_index()
        .groupby("True_Onset_Minute", as_index=False)
        .agg(
            Windows=("Absolute_Error", "size"),
            ETA_MAE=("Absolute_Error", "mean"),
            Mean_Signed_Error=("Signed_Error", "mean"),
            Mean_Predicted_ETA=("Predicted_ETA", "mean")
        )
    )

    PER_MINUTE14[model] = by_minute

    source_stats = (
        event
        .reset_index()
        .groupby("Source", as_index=False)
        .agg(
            Event_Windows=("Absolute_Error", "size"),
            ETA_MAE=("Absolute_Error", "mean")
        )
    )

    PER_SOURCE14[model] = source_stats

    report = MINUTE_REPORTS14[model].copy()

    required_minute = {
        "True_Onset_Minute", "Windows",
        "Expected_ETA_MAE"
    }

    if not required_minute.issubset(report.columns):
        raise RuntimeError(
            f"{model}: unexpected per-minute report schema"
        )

    check = by_minute.merge(
        report,
        on="True_Onset_Minute",
        how="outer",
        validate="one_to_one",
        suffixes=("_recomputed", "_reported")
    )

    if check.isna().any().any():
        raise RuntimeError(
            f"{model}: incomplete minute-level comparison"
        )

    np.testing.assert_array_equal(
        check["Windows_recomputed"],
        check["Windows_reported"]
    )

    np.testing.assert_allclose(
        check["ETA_MAE"],
        check["Expected_ETA_MAE"],
        rtol=1e-6,
        atol=1e-6,
        err_msg=f"{model}: per-minute ETA MAE mismatch"
    )

    source_report = SOURCE_REPORTS14[model]

    for _, row in source_stats.iterrows():

        matching = source_report.loc[
            source_report["Source"] == row["Source"]
        ]

        if len(matching) != 1:
            raise RuntimeError(
                f"{model}: missing source report for {row['Source']}"
            )

        np.testing.assert_allclose(
            row["ETA_MAE"],
            float(matching["ETA_MAE"].iloc[0]),
            rtol=1e-6,
            atol=1e-6,
            err_msg=f"{model}: source-level ETA mismatch"
        )

    STATS14.append({
        "Model": model,
        "Event_Windows": int(len(event)),
        "ETA_MAE": float(event["Absolute_Error"].mean()),
        "ETA_Median_Absolute_Error":
            float(event["Absolute_Error"].median()),
        "Mean_Signed_Error":
            float(event["Signed_Error"].mean())
    })

STATS14 = pd.DataFrame(STATS14)

if not all(len(EVENT14[m]) == 3375 for m in SOURCES14):
    raise RuntimeError("Unexpected conditional event count")

print("\nB. INDEPENDENT ETA METRIC VERIFICATION")
print("-" * 90)
print(STATS14.round(5).to_string(index=False))
print("\nPer-minute reported MAE: MATCH")
print("Source-specific reported MAE: MATCH")

# ---------------------------------------------------------------
# C. PUBLICATION FIGURE
# ---------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig14, axes14 = plt.subplots(
    3, 2,
    figsize=(15.5, 14.5),
    gridspec_kw={
        "hspace": 0.43,
        "wspace": 0.28
    }
)

fig14.suptitle(
    "Conditional time-to-attack prediction accuracy",
    x=0.09,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig14.text(
    0.09,
    0.95,
    "Competing-risk LSTM versus TCN  |  "
    "3,375 true-event windows  |  "
    "ETA horizon: 1–15 minutes",
    fontsize=10.5,
    color="#586575"
)

def style14(ax):
    ax.set_facecolor("#FBFCFD")
    ax.grid(
        linestyle=":",
        linewidth=0.7,
        color="#C5CED8",
        alpha=0.55
    )
    ax.set_axisbelow(True)
    ax.tick_params(length=3, pad=6)

# A/B: True vs predicted ETA densities
for j, model in enumerate(["CR-LSTM", "CR-TCN"]):

    ax = axes14[0, j]
    event = EVENT14[model]

    hb = ax.hexbin(
        event["True_ETA"],
        event["Predicted_ETA"],
        gridsize=28,
        extent=(0.5, 15.5, 0.5, 15.5),
        cmap=CMAPS14[model],
        mincnt=1,
        bins="log",
        linewidths=0
    )

    ax.plot(
        [1, 15], [1, 15],
        color="#D1873C",
        linestyle=(0, (5, 3)),
        linewidth=1.5,
        label="Perfect ETA"
    )

    ax.set_xlim(0.5, 15.5)
    ax.set_ylim(0.5, 15.5)
    ax.set_xticks([1, 3, 5, 7, 9, 11, 13, 15])
    ax.set_yticks([1, 3, 5, 7, 9, 11, 13, 15])

    ax.set_title(
        f"{'A' if j == 0 else 'B'}   "
        f"{model}: Actual vs predicted ETA",
        loc="left",
        fontweight="bold",
        pad=13
    )

    ax.set_xlabel("True time until attack (minutes)")
    ax.set_ylabel("Expected ETA prediction (minutes)")

    fig14.colorbar(
        hb, ax=ax,
        fraction=0.046,
        pad=0.02,
        label="Window density (log count)"
    )

# C: Absolute-error cumulative distribution
ax = axes14[1, 0]
style14(ax)

for model in SOURCES14:

    errors = np.sort(
        EVENT14[model]["Absolute_Error"].to_numpy()
    )

    cdf = (
        np.arange(1, len(errors)+1) / len(errors)
    )

    ax.plot(
        errors,
        cdf,
        color=COLORS14[model],
        linewidth=2.1,
        label=model
    )

ax.set_xlim(left=0)
ax.set_ylim(0, 1.02)
ax.set_title(
    "C   Conditional ETA absolute-error distribution",
    loc="left",
    fontweight="bold",
    pad=13
)
ax.set_xlabel("Absolute ETA error (minutes)")
ax.set_ylabel("Fraction of event windows")
ax.legend(frameon=False)

# D: MAE by true ETA
ax = axes14[1, 1]
style14(ax)

for model in SOURCES14:

    minute = PER_MINUTE14[model]

    ax.plot(
        minute["True_Onset_Minute"],
        minute["ETA_MAE"],
        marker="o",
        markersize=4,
        linewidth=2,
        color=COLORS14[model],
        label=model
    )

ax.set_title(
    "D   Error versus actual time remaining",
    loc="left",
    fontweight="bold",
    pad=13
)
ax.set_xticks(range(1, 16, 2))
ax.set_xlabel("True time until attack (minutes)")
ax.set_ylabel("Conditional ETA MAE (minutes)")

# E: Signed-error bias by true ETA
ax = axes14[2, 0]
style14(ax)

ax.axhline(
    0,
    color="#7B8794",
    linewidth=1.1,
    linestyle=(0, (4, 3))
)

for model in SOURCES14:

    minute = PER_MINUTE14[model]

    ax.plot(
        minute["True_Onset_Minute"],
        minute["Mean_Signed_Error"],
        marker="o",
        markersize=4,
        linewidth=2,
        color=COLORS14[model],
        label=model
    )

ax.set_title(
    "E   Systematic early/late ETA bias",
    loc="left",
    fontweight="bold",
    pad=13
)
ax.set_xticks(range(1, 16, 2))
ax.set_xlabel("True time until attack (minutes)")
ax.set_ylabel("Predicted minus true ETA (minutes)")

# F: Source-specific conditional MAE
ax = axes14[2, 1]
style14(ax)

source_order = ["CIC_IDS2017", "UNSW_NB15"]
source_labels = ["CIC-derived", "UNSW-derived"]

xx = np.arange(len(source_order))
width = 0.33

for j, model in enumerate(["CR-LSTM", "CR-TCN"]):

    frame = PER_SOURCE14[model].set_index("Source")

    values = [
        float(frame.loc[s, "ETA_MAE"])
        for s in source_order
    ]

    bars = ax.bar(
        xx + (j-0.5)*width,
        values,
        width=width,
        color=COLORS14[model],
        edgecolor="white",
        label=model,
        alpha=0.90
    )

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height()+0.035,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax.set_xticks(xx)
ax.set_xticklabels(source_labels)
ax.set_ylim(0, 3.25)
ax.set_ylabel("Conditional ETA MAE (minutes)")
ax.set_title(
    "F   ETA accuracy by source dataset",
    loc="left",
    fontweight="bold",
    pad=13
)
ax.legend(frameon=False, loc="upper left")

fig14.subplots_adjust(
    left=0.10,
    right=0.95,
    top=0.88,
    bottom=0.10
)

fig14.text(
    0.09,
    0.038,
    "Synthetic NAF v2 validation. ETA is evaluated "
    "only on windows with a true attack within 15 minutes. "
    "Predictions use conditional expected ETA, not "
    "the discrete most-likely onset class.",
    fontsize=8.8,
    color="#586575"
)

fig14.text(
    0.09,
    0.022,
    "Positive signed error means the model predicts "
    "more time remaining than actually exists. "
    "No independent-window uncertainty intervals are inferred.",
    fontsize=8.8,
    color="#586575"
)

# ---------------------------------------------------------------
# D. SAVE FIGURE AND SCIENTIFIC TABLES
# ---------------------------------------------------------------

stamp14 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem14 = f"F10_Conditional_ETA_Analysis_{stamp14}"

created14 = []

for ext in ["pdf", "svg", "png"]:

    path = FIG14 / f"{stem14}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig14.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    created14.append(path)

plt.show()
plt.close(fig14)

minute_export14 = pd.concat(
    [
        frame.assign(Model=model)
        for model, frame in PER_MINUTE14.items()
    ],
    ignore_index=True
)

source_export14 = pd.concat(
    [
        frame.assign(Model=model)
        for model, frame in PER_SOURCE14.items()
    ],
    ignore_index=True
)

table_minute14 = (
    TAB14 / f"T12_ETA_By_True_Minute_{stamp14}.csv"
)

table_source14 = (
    TAB14 / f"T13_ETA_By_Source_{stamp14}.csv"
)

minute_export14.to_csv(
    table_minute14,
    index=False,
    mode="x",
    float_format="%.8f"
)

source_export14.to_csv(
    table_source14,
    index=False,
    mode="x",
    float_format="%.8f"
)

created14.extend([
    table_minute14,
    table_source14
])

# ---------------------------------------------------------------
# E. METRICS SNAPSHOT AND REPRODUCIBILITY MANIFEST
# ---------------------------------------------------------------

def hash14(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024*1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

def json_safe14(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {
            str(k): json_safe14(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_safe14(v) for v in value]
    return value

metric_path14 = (
    MET14 / f"conditional_eta_metrics_{stamp14}.json"
)

metric_content14 = {
    "project": "NetGuard",
    "evaluation": "Synthetic_NAF_v2 validation",
    "definition": (
        "MAE of Expected_ETA_If_Event, conditional "
        "on True_Onset_Minute in [1,15]"
    ),
    "models": STATS14.to_dict(orient="records"),
    "per_minute_report_verified": True,
    "per_source_report_verified": True,
    "official_test_accessed": False
}

with metric_path14.open("x", encoding="utf-8") as f:
    json.dump(
        json_safe14(metric_content14),
        f,
        indent=2,
        allow_nan=False
    )

created14.append(metric_path14)

manifest_path14 = (
    MAN14 / f"eta_analysis_manifest_{stamp14}.json"
)

manifest14 = {
    "figure": stem14,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "models": list(SOURCES14),
    "event_windows_per_model": 3375,
    "evaluation_only_on_true_event_windows": True,
    "data_inputs": [
        {
            "path": str(p.relative_to(PROJECT14)),
            "sha256": hash14(p)
        }
        for p in INPUT_FILES14
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT14)),
            "sha256": hash14(p)
        }
        for p in created14
    ],
    "limitations": [
        "Conditional ETA is not end-to-end attack-onset accuracy.",
        "Synthetic validation windows are temporally dependent.",
        "ETA predictions are conditional on an event.",
        "No independent uncertainty estimates are implied.",
        "No official TEST data accessed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with manifest_path14.open("x", encoding="utf-8") as f:
    json.dump(
        json_safe14(manifest14),
        f,
        indent=2,
        allow_nan=False
    )

print("\n" + "=" * 90)
print("CELL 14 COMPLETE")
print("=" * 90)

print("Models evaluated:", len(SOURCES14))
print("True-event windows per model:", 3375)
print("Figure panels:", 6)
print("Per-minute MAE verification: PASS")
print("Source-level MAE verification: PASS")

print("\nSaved files:")

for path in created14 + [manifest_path14]:
    assert path.is_file()
    assert path.stat().st_size > 0
    print("PASS:", path.relative_to(OUT14))

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Source validation reports modified: NO")
print("Existing research outputs overwritten: NO")
print("=" * 90)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 15 — GRAPH FORECASTING ARCHITECTURE COMPARISON
# FULL CORRECTED VERSION
#
# FIGURE F11:
#   A. Horizon-wise average precision across six architectures
#   B. Graph-model AP difference versus Stacked LSTM
#   C. H10 recall versus false-positive rate at threshold 0.5
#   D. Original versus optimized Graph LSTM trade-offs
#
# RESEARCH SOURCES:
#   GCN report:  9 rows, 3 graph models x 3 horizons
#   GAT report: 12 rows, 4 graph models x 3 horizons
#   Original/optimized Graph LSTM comparison
#   Previously verified seven-model AP benchmark
#
# OUTPUTS:
#   F11: PDF, SVG, PNG
#   T14: Graph-model metrics CSV
#   Metrics: JSON
#   Provenance: JSON
#
# SAFETY:
#   No retraining
#   No official TEST access
#   No checkpoint changes
#   No original-source modifications
#   No existing-output overwrites
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

print("=" * 100)
print("CELL 15 — GRAPH FORECASTING ARCHITECTURE COMPARISON")
print("=" * 100)

# ======================================================================
# A. PROJECT PATHS
# ======================================================================

PROJECT15 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

GRAPH_ROOT15 = (
    PROJECT15
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
)

OUTPUT15 = PROJECT15 / "Research_Outputs"

FIGURES15 = (
    OUTPUT15
    / "01_Figures"
    / "Model_Comparisons"
)

TABLES15 = OUTPUT15 / "02_Tables"
METRICS15 = OUTPUT15 / "03_Metrics"
MANIFESTS15 = OUTPUT15 / "05_Manifests"

for folder in [
    GRAPH_ROOT15,
    FIGURES15,
    TABLES15,
    METRICS15,
    MANIFESTS15
]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

print("\nA. SOURCE DIRECTORY VERIFICATION")
print("-" * 100)

print("Project:", PROJECT15)
print("Graph models:", GRAPH_ROOT15)
print("Research outputs:", OUTPUT15)

# ======================================================================
# B. EXACT SAVED REPORTS
# ======================================================================

GAT_PATH15 = (
    GRAPH_ROOT15
    / "GAT_LSTM"
    / "validation_v1"
    / "GAT_VAL_METRICS.csv"
)

GCN_PATH15 = (
    GRAPH_ROOT15
    / "GCN_LSTM"
    / "validation_v1"
    / "GCN_VAL_METRICS.csv"
)

GRAPH_COMPARISON_PATH15 = (
    GRAPH_ROOT15
    / "Graph_LSTM"
    / "validation_comparison_v1"
    / "ORIGINAL_VS_C02_METRICS.csv"
)

BASELINE_PATH15 = (
    OUTPUT15
    / "02_Tables"
    / "T10_Seven_Model_AP_Benchmark_20261027_064511_790770.csv"
)

SOURCE_FILES15 = [
    GAT_PATH15,
    GCN_PATH15,
    GRAPH_COMPARISON_PATH15,
    BASELINE_PATH15
]

for path in SOURCE_FILES15:

    if not path.is_file():
        raise FileNotFoundError(
            f"Required saved report missing:\n{path}"
        )

print("\nB. VERIFIED SOURCE FILES")
print("-" * 100)

for path in SOURCE_FILES15:

    print(
        "FOUND |",
        path.relative_to(PROJECT15)
    )

# ======================================================================
# C. LOAD SAVED METRICS
# ======================================================================

gat15 = pd.read_csv(GAT_PATH15)
gcn15 = pd.read_csv(GCN_PATH15)

original_comparison15 = pd.read_csv(
    GRAPH_COMPARISON_PATH15
)

baseline15 = pd.read_csv(
    BASELINE_PATH15
)

for frame in [
    gat15,
    gcn15,
    original_comparison15,
    baseline15
]:
    frame.columns = (
        frame.columns.astype(str).str.strip()
    )

print("\nC. SOURCE SHAPES")
print("-" * 100)

print("GAT full benchmark:", gat15.shape)
print("GCN earlier benchmark:", gcn15.shape)
print("Graph LSTM original/optimized:", original_comparison15.shape)
print("Seven-model baseline AP:", baseline15.shape)

# ======================================================================
# D. VERIFY THE 9-ROW / 12-ROW DIFFERENCE
# ======================================================================

REQUIRED_GRAPH15 = {
    "Model",
    "Horizon",
    "AP",
    "AUROC",
    "Brier",
    "Precision@0.5",
    "Recall@0.5",
    "F1@0.5",
    "FPR@0.5",
    "TP",
    "FP",
    "FN",
    "TN"
}

for name, frame, expected_rows in [
    ("GAT", gat15, 12),
    ("GCN", gcn15, 9)
]:

    missing = REQUIRED_GRAPH15 - set(frame.columns)

    if missing:
        raise RuntimeError(
            f"{name}: missing columns {sorted(missing)}"
        )

    if len(frame) != expected_rows:
        raise RuntimeError(
            f"{name}: expected {expected_rows} rows, "
            f"found {len(frame)}"
        )

    if frame.duplicated(
        ["Model", "Horizon"]
    ).any():
        raise RuntimeError(
            f"{name}: duplicate model/horizon rows"
        )

GRAPH_NAME_MAPPING15 = {
    "Original Graph LSTM": "Original Graph LSTM",
    "Optimized Graph LSTM": "Optimized Graph LSTM",
    "GCN + LSTM": "GCN-LSTM",
    "GAT + LSTM": "GAT-LSTM"
}

SHARED_RAW_MODELS15 = [
    "Original Graph LSTM",
    "Optimized Graph LSTM",
    "GCN + LSTM"
]

GRAPH_MODELS15 = [
    "Original Graph LSTM",
    "Optimized Graph LSTM",
    "GCN-LSTM",
    "GAT-LSTM"
]

HORIZONS15 = ["H5", "H10", "H15"]

if set(gat15["Model"]) != set(GRAPH_NAME_MAPPING15):
    raise RuntimeError(
        "GAT report does not contain the expected four models"
    )

if set(gcn15["Model"]) != set(SHARED_RAW_MODELS15):
    raise RuntimeError(
        "GCN report does not contain the expected three models"
    )

if set(gat15["Horizon"]) != set(HORIZONS15):
    raise RuntimeError(
        "Unexpected horizons in GAT report"
    )

if set(gcn15["Horizon"]) != set(HORIZONS15):
    raise RuntimeError(
        "Unexpected horizons in GCN report"
    )

# The GAT report is newer and contains the three models
# previously reported in the GCN audit, plus GAT-LSTM.

gat_shared15 = (
    gat15.loc[
        gat15["Model"].isin(SHARED_RAW_MODELS15)
    ]
    .sort_values(["Model", "Horizon"])
    .reset_index(drop=True)
)

gcn_shared15 = (
    gcn15
    .sort_values(["Model", "Horizon"])
    .reset_index(drop=True)
)

if len(gat_shared15) != 9:
    raise RuntimeError(
        "Expected nine shared GAT/GCN report rows"
    )

# Compare only the common validated metric fields.
# Do not require unrelated report-specific columns.

COMPARISON_COLUMNS15 = [
    "Model",
    "Horizon",
    "AP",
    "AUROC",
    "Brier",
    "Precision@0.5",
    "Recall@0.5",
    "F1@0.5",
    "FPR@0.5",
    "TP",
    "FP",
    "FN",
    "TN"
]

pd.testing.assert_frame_equal(
    gat_shared15[COMPARISON_COLUMNS15],
    gcn_shared15[COMPARISON_COLUMNS15],
    check_exact=False,
    check_dtype=False,
    rtol=1e-7,
    atol=1e-9
)

print("\nD. GRAPH REPORT RECONCILIATION")
print("-" * 100)

print("GCN report rows:", len(gcn15))
print("GAT report rows:", len(gat15))
print("Shared results:", len(gat_shared15))
print("GAT-only results:", len(gat15) - len(gat_shared15))
print("Shared metric agreement: PASS")
print("Complete graph benchmark: GAT_VAL_METRICS.csv")

# ======================================================================
# E. COMPLETE GRAPH-MODEL EVALUATION DATA
# ======================================================================

graph15 = gat15.copy()

graph15["Model"] = (
    graph15["Model"].map(GRAPH_NAME_MAPPING15)
)

if graph15["Model"].isna().any():
    raise RuntimeError(
        "Unknown graph-model name in complete report"
    )

NUMERIC_METRICS15 = [
    "AP",
    "AUROC",
    "Brier",
    "Precision@0.5",
    "Recall@0.5",
    "F1@0.5",
    "FPR@0.5"
]

COUNT_COLUMNS15 = [
    "TP", "FP", "FN", "TN"
]

for col in NUMERIC_METRICS15 + COUNT_COLUMNS15:

    graph15[col] = pd.to_numeric(
        graph15[col],
        errors="raise"
    )

if not np.isfinite(
    graph15[
        NUMERIC_METRICS15 + COUNT_COLUMNS15
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Nonfinite graph-model evaluation values"
    )

if not graph15[NUMERIC_METRICS15].ge(0).all().all():
    raise RuntimeError(
        "Negative graph-model metric"
    )

if not graph15[NUMERIC_METRICS15].le(1).all().all():
    raise RuntimeError(
        "Graph-model metric exceeds 1"
    )

if not graph15[COUNT_COLUMNS15].ge(0).all().all():
    raise RuntimeError(
        "Negative confusion count"
    )

if not np.allclose(
    graph15[COUNT_COLUMNS15].to_numpy(),
    np.round(graph15[COUNT_COLUMNS15].to_numpy())
):
    raise RuntimeError(
        "Noninteger confusion count"
    )

EXPECTED_POSITIVES15 = {
    "H5": 1125,
    "H10": 2250,
    "H15": 3375
}

for _, row in graph15.iterrows():

    tp = int(row["TP"])
    fp = int(row["FP"])
    fn = int(row["FN"])
    tn = int(row["TN"])

    total = tp + fp + fn + tn

    if total != 18839:
        raise RuntimeError(
            f"{row['Model']}/{row['Horizon']}: "
            f"expected 18,839 windows, got {total}"
        )

    if tp + fn != EXPECTED_POSITIVES15[row["Horizon"]]:
        raise RuntimeError(
            "Positive-support mismatch"
        )

    precision = (
        tp / (tp + fp)
        if tp + fp else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn else 0.0
    )

    fpr = (
        fp / (fp + tn)
        if fp + tn else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    np.testing.assert_allclose(
        [
            precision,
            recall,
            f1,
            fpr
        ],
        [
            row["Precision@0.5"],
            row["Recall@0.5"],
            row["F1@0.5"],
            row["FPR@0.5"]
        ],
        atol=1e-7,
        rtol=1e-7
    )

print("\nE. GRAPH EVALUATION QUALITY CHECKS")
print("-" * 100)

print("Graph architectures:", len(GRAPH_MODELS15))
print("Horizon/model combinations:", len(graph15))
print("Validation windows per combination:", 18839)
print("Class-support verification: PASS")
print("Precision/recall/F1/FPR verification: PASS")

# ======================================================================
# F. INDEPENDENT ORIGINAL/OPTIMIZED CROSS-CHECK
# ======================================================================

required_original15 = {
    "Model",
    "Horizon",
    "AP",
    "AUROC",
    "Brier",
    "TP",
    "FP",
    "FN",
    "TN"
}

if not required_original15.issubset(
    original_comparison15.columns
):
    raise RuntimeError(
        "Original/optimized report schema mismatch"
    )

original_name_map15 = {
    "Original": "Original Graph LSTM",
    "Optimized C02": "Optimized Graph LSTM"
}

original_comparison15["Model"] = (
    original_comparison15["Model"]
    .map(original_name_map15)
)

if original_comparison15["Model"].isna().any():
    raise RuntimeError(
        "Unknown original/optimized model label"
    )

if len(original_comparison15) != 6:
    raise RuntimeError(
        "Expected six original/optimized comparison rows"
    )

for _, row in original_comparison15.iterrows():

    match = graph15.loc[
        (graph15["Model"] == row["Model"])
        & (graph15["Horizon"] == row["Horizon"])
    ]

    if len(match) != 1:
        raise RuntimeError(
            "Missing model/horizon in full report"
        )

    for metric in [
        "AP",
        "AUROC",
        "Brier",
        "TP",
        "FP",
        "FN",
        "TN"
    ]:

        np.testing.assert_allclose(
            float(row[metric]),
            float(match.iloc[0][metric]),
            rtol=1e-7,
            atol=1e-9
        )

print("\nF. ORIGINAL VS OPTIMIZED VERIFICATION")
print("-" * 100)

print("Independent comparison rows:", len(original_comparison15))
print("Agreement with full GAT benchmark: PASS")

# ======================================================================
# G. VERIFIED NON-GRAPH REFERENCE MODELS
# ======================================================================

required_baseline15 = {
    "Model",
    "Horizon",
    "Average_Precision",
    "Available",
    "Positive_Windows"
}

if not required_baseline15.issubset(
    baseline15.columns
):
    raise RuntimeError(
        "Seven-model reference table schema mismatch"
    )

REFERENCE_MODELS15 = [
    "Stacked LSTM",
    "CR-LSTM"
]

baseline_ref15 = baseline15.loc[
    baseline15["Model"].isin(REFERENCE_MODELS15)
].copy()

if len(baseline_ref15) != 6:
    raise RuntimeError(
        "Expected six baseline AP records"
    )

baseline_ref15["Average_Precision"] = pd.to_numeric(
    baseline_ref15["Average_Precision"],
    errors="raise"
)

if not baseline_ref15["Available"].astype(
    str
).str.lower().eq("true").all():
    raise RuntimeError(
        "Unavailable baseline AP result"
    )

for _, row in baseline_ref15.iterrows():

    if int(row["Positive_Windows"]) != (
        EXPECTED_POSITIVES15[row["Horizon"]]
    ):
        raise RuntimeError(
            "Reference baseline positive-support mismatch"
        )

GRAPH_AP15 = (
    graph15
    .pivot(
        index="Model",
        columns="Horizon",
        values="AP"
    )
    .loc[GRAPH_MODELS15, HORIZONS15]
)

BASELINE_AP15 = (
    baseline_ref15
    .pivot(
        index="Model",
        columns="Horizon",
        values="Average_Precision"
    )
    .loc[REFERENCE_MODELS15, HORIZONS15]
)

STACKED_AP15 = BASELINE_AP15.loc["Stacked LSTM"]

DELTA_AP15 = GRAPH_AP15.subtract(
    STACKED_AP15,
    axis="columns"
)

print("\nG. HORIZON-WISE AVERAGE PRECISION")
print("-" * 100)

print("\nGRAPH MODELS")
print(GRAPH_AP15.round(6).to_string())

print("\nREFERENCE BASELINES")
print(BASELINE_AP15.round(6).to_string())

print("\nGRAPH AP MINUS STACKED LSTM AP")
print(DELTA_AP15.round(6).to_string())

# ======================================================================
# H. FIGURE STYLING
# ======================================================================

COLORS15 = {
    "Original Graph LSTM": "#3F7DA8",
    "Optimized Graph LSTM": "#204780",
    "GCN-LSTM": "#748E4C",
    "GAT-LSTM": "#C5617B",
    "Stacked LSTM": "#BA7D3F",
    "CR-LSTM": "#268B84"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig15, axes15 = plt.subplots(
    2, 2,
    figsize=(17, 12.5),
    gridspec_kw={
        "wspace": 0.35,
        "hspace": 0.43
    }
)

fig15.patch.set_facecolor("white")

fig15.suptitle(
    "Graph-aware versus sequence-only attack forecasting",
    x=0.085,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig15.text(
    0.085,
    0.950,
    "Four graph architectures compared with "
    "two sequence-model baselines  |  "
    "18,839 aligned validation windows",
    fontsize=10.8,
    color="#586575"
)

def style_axis15(ax):

    ax.set_facecolor("#FAFBFC")

    ax.grid(
        linestyle=":",
        linewidth=0.7,
        color="#C7CFD9",
        alpha=0.6
    )

    ax.set_axisbelow(True)

    ax.tick_params(
        length=3,
        width=0.7,
        pad=6
    )

# ======================================================================
# PANEL A — HORIZON-WISE AP
# ======================================================================

ax = axes15[0, 0]
style_axis15(ax)

x15 = np.arange(3)

for model in GRAPH_MODELS15:

    ax.plot(
        x15,
        GRAPH_AP15.loc[model].to_numpy(),
        marker="o",
        markersize=5.5,
        linewidth=1.9,
        color=COLORS15[model],
        label=model
    )

for model in REFERENCE_MODELS15:

    ax.plot(
        x15,
        BASELINE_AP15.loc[model].to_numpy(),
        marker="s",
        markersize=5.5,
        linewidth=2.1,
        linestyle="--",
        color=COLORS15[model],
        label=model
    )

ax.set_xticks(x15)
ax.set_xticklabels(HORIZONS15)

ax.set_ylim(0.68, 0.97)

ax.set_ylabel("Average precision (AP)")
ax.set_xlabel("Prediction horizon")

ax.set_title(
    "A   Forecast discrimination across architectures",
    loc="left",
    fontweight="bold",
    pad=14
)

ax.legend(
    frameon=False,
    fontsize=8.5,
    loc="upper left",
    ncol=2
)

# ======================================================================
# PANEL B — AP GAP VERSUS STACKED LSTM
# ======================================================================

ax = axes15[0, 1]

difference15 = DELTA_AP15.to_numpy() * 100

limit15 = max(
    float(np.max(np.abs(difference15))),
    0.5
)

image15 = ax.imshow(
    difference15,
    cmap="RdBu",
    norm=TwoSlopeNorm(
        vmin=-limit15 * 1.08,
        vcenter=0,
        vmax=limit15 * 1.08
    ),
    aspect="auto",
    interpolation="nearest"
)

ax.set_xticks(np.arange(3))
ax.set_xticklabels(HORIZONS15)

ax.set_yticks(np.arange(len(GRAPH_MODELS15)))
ax.set_yticklabels(GRAPH_MODELS15)

for i in range(len(GRAPH_MODELS15)):

    for j in range(len(HORIZONS15)):

        value = float(difference15[i, j])

        ax.text(
            j,
            i,
            f"{value:+.2f} pp",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="medium",
            color=(
                "white"
                if abs(value) > limit15 * 0.60
                else "#263545"
            )
        )

ax.set_title(
    "B   Average precision relative to Stacked LSTM",
    loc="left",
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Prediction horizon")

# ======================================================================
# PANEL C — H10 RECALL VERSUS FALSE-POSITIVE RATE
# ======================================================================

ax = axes15[1, 0]
style_axis15(ax)

h10_15 = (
    graph15.loc[
        graph15["Horizon"] == "H10"
    ]
    .set_index("Model")
    .loc[GRAPH_MODELS15]
)

for model in GRAPH_MODELS15:

    row = h10_15.loc[model]

    fpr = float(row["FPR@0.5"]) * 100
    recall = float(row["Recall@0.5"]) * 100

    ax.scatter(
        fpr,
        recall,
        s=125,
        color=COLORS15[model],
        edgecolor="white",
        linewidth=1.2,
        zorder=4
    )

    ax.annotate(
        model,
        xy=(fpr, recall),
        xytext=(6, 7),
        textcoords="offset points",
        fontsize=8.2,
        color="#27384B"
    )

min_fpr15 = (
    h10_15["FPR@0.5"].min() * 100
)

max_fpr15 = (
    h10_15["FPR@0.5"].max() * 100
)

min_recall15 = (
    h10_15["Recall@0.5"].min() * 100
)

max_recall15 = (
    h10_15["Recall@0.5"].max() * 100
)

ax.set_xlim(
    max(0, min_fpr15 - 0.7),
    max_fpr15 + 1.4
)

ax.set_ylim(
    max(0, min_recall15 - 5),
    min(100, max_recall15 + 7)
)

ax.set_xlabel(
    "False-positive rate at threshold 0.5 (%)"
)

ax.set_ylabel(
    "Recall at threshold 0.5 (%)"
)

ax.set_title(
    "C   H10 operational trade-off",
    loc="left",
    fontweight="bold",
    pad=14
)

# ======================================================================
# PANEL D — ORIGINAL VS OPTIMIZED GRAPH LSTM
# ======================================================================

ax = axes15[1, 1]
style_axis15(ax)

original15 = (
    graph15.loc[
        graph15["Model"] == "Original Graph LSTM"
    ]
    .set_index("Horizon")
    .loc[HORIZONS15]
)

optimized15 = (
    graph15.loc[
        graph15["Model"] == "Optimized Graph LSTM"
    ]
    .set_index("Horizon")
    .loc[HORIZONS15]
)

TRADEOFFS15 = {
    "AP difference": (
        optimized15["AP"]
        - original15["AP"]
    ).to_numpy() * 100,

    "Recall difference": (
        optimized15["Recall@0.5"]
        - original15["Recall@0.5"]
    ).to_numpy() * 100,

    "FPR difference": (
        optimized15["FPR@0.5"]
        - original15["FPR@0.5"]
    ).to_numpy() * 100
}

TRADEOFF_COLORS15 = {
    "AP difference": "#2B8387",
    "Recall difference": "#BC674F",
    "FPR difference": "#5266A2"
}

ax.axhline(
    0,
    color="#778493",
    linestyle="--",
    linewidth=1.1
)

for name, values in TRADEOFFS15.items():

    ax.plot(
        x15,
        values,
        marker="o",
        linewidth=2,
        markersize=6,
        color=TRADEOFF_COLORS15[name],
        label=name
    )

ax.set_xticks(x15)
ax.set_xticklabels(HORIZONS15)

ax.set_xlabel("Prediction horizon")

ax.set_ylabel(
    "Optimized minus original (percentage points)"
)

ax.set_title(
    "D   Graph LSTM optimization trade-offs",
    loc="left",
    fontweight="bold",
    pad=14
)

ax.legend(
    frameon=False,
    loc="lower left",
    fontsize=8.8
)

# ======================================================================
# I. FIGURE CAPTION AND LAYOUT
# ======================================================================

fig15.subplots_adjust(
    left=0.115,
    right=0.955,
    top=0.875,
    bottom=0.115
)

fig15.text(
    0.085,
    0.057,
    "Synthetic NAF v2 development validation. "
    "AP differences are relative to the "
    "Stacked LSTM baseline verified in Cell 13. "
    "Higher AP and recall are preferable.",
    fontsize=9,
    color="#576574"
)

fig15.text(
    0.085,
    0.039,
    "Lower false-positive rate is preferable. "
    "Fixed-threshold measurements use 0.5; "
    "model differences are descriptive and do not "
    "imply statistical significance.",
    fontsize=9,
    color="#576574"
)

# ======================================================================
# J. SAVE FIGURE
# ======================================================================

RUN15 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM15 = (
    f"F11_Graph_Architecture_Comparison_{RUN15}"
)

FIGURE_FILES15 = []

for extension in ["pdf", "svg", "png"]:

    destination = (
        FIGURES15
        / f"{STEM15}.{extension}"
    )

    if destination.exists():
        raise FileExistsError(destination)

    fig15.savefig(
        destination,
        dpi=450 if extension == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES15.append(destination)

plt.show()
plt.close(fig15)

# ======================================================================
# K. SAVE RESEARCH TABLE
# ======================================================================

TABLE_PATH15 = (
    TABLES15
    / f"T14_Graph_Architecture_Benchmark_{RUN15}.csv"
)

graph15.sort_values(
    ["Model", "Horizon"]
).to_csv(
    TABLE_PATH15,
    index=False,
    mode="x",
    float_format="%.8f"
)

# ======================================================================
# L. SAVE JSON METRICS
# ======================================================================

METRICS_PATH15 = (
    METRICS15
    / f"graph_architecture_metrics_{RUN15}.json"
)

metrics_data15 = {
    "project": "NetGuard",
    "benchmark": "Synthetic_NAF_v2",
    "split": "development_validation",
    "graph_models": GRAPH_MODELS15,
    "reference_models": REFERENCE_MODELS15,
    "horizons": HORIZONS15,
    "windows_per_horizon": 18839,
    "graph_ap": {
        model: {
            horizon: float(
                GRAPH_AP15.loc[model, horizon]
            )
            for horizon in HORIZONS15
        }
        for model in GRAPH_MODELS15
    },
    "reference_ap": {
        model: {
            horizon: float(
                BASELINE_AP15.loc[model, horizon]
            )
            for horizon in HORIZONS15
        }
        for model in REFERENCE_MODELS15
    },
    "ap_difference_vs_stacked_lstm": {
        model: {
            horizon: float(
                DELTA_AP15.loc[model, horizon]
            )
            for horizon in HORIZONS15
        }
        for model in GRAPH_MODELS15
    },
    "report_validation": {
        "gcn_report_rows": 9,
        "gat_report_rows": 12,
        "shared_rows_verified": 9,
        "independent_graph_lstm_comparison_verified": True,
        "positive_support_verified": True,
        "precision_recall_f1_fpr_recomputed": True
    },
    "models_retrained": 0,
    "official_test_accessed": False
}

with METRICS_PATH15.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        metrics_data15,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# M. SHA256 PROVENANCE MANIFEST
# ======================================================================

def sha256_file15(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

GENERATED15 = (
    FIGURE_FILES15
    + [
        TABLE_PATH15,
        METRICS_PATH15
    ]
)

MANIFEST_PATH15 = (
    MANIFESTS15
    / f"graph_architecture_manifest_{RUN15}.json"
)

manifest_data15 = {
    "project": "NetGuard",
    "figure": STEM15,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "evaluation": "Synthetic_NAF_v2 development validation",
    "reconciliation": {
        "gcn_report_records": 9,
        "gat_report_records": 12,
        "nine_shared_records_verified": True,
        "complete_graph_benchmark_source": str(
            GAT_PATH15.relative_to(PROJECT15)
        )
    },
    "input_files": [
        {
            "path": str(
                path.relative_to(PROJECT15)
            ),
            "sha256": sha256_file15(path)
        }
        for path in SOURCE_FILES15
    ],
    "generated_files": [
        {
            "path": str(
                path.relative_to(PROJECT15)
            ),
            "sha256": sha256_file15(path)
        }
        for path in GENERATED15
    ],
    "scientific_limitations": [
        "Results are from saved synthetic validation data.",
        "All models use the same nominal validation support.",
        "No new paired uncertainty intervals were computed.",
        "Fixed-threshold comparison uses threshold 0.5.",
        "AP differences are descriptive.",
        "No live-network generalization claim is implied.",
        "No official TEST data accessed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with MANIFEST_PATH15.open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest_data15,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# N. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 100)
print("CELL 15 COMPLETE — CORRECTED")
print("=" * 100)

print("Graph models evaluated:", len(GRAPH_MODELS15))
print("Reference baselines:", len(REFERENCE_MODELS15))
print("Horizon/model combinations:", len(graph15))
print("Validation windows per horizon:", 18839)
print("Figure panels:", 4)

print("\nBEST GRAPH MODEL BY HORIZON")
print("-" * 100)

for horizon in HORIZONS15:

    best_model = GRAPH_AP15[horizon].idxmax()

    best_ap = float(
        GRAPH_AP15.loc[
            best_model,
            horizon
        ]
    )

    print(
        f"{horizon:<5} "
        f"{best_model:<25} "
        f"AP={best_ap:.6f}"
    )

print("\nSAVED RESEARCH OUTPUTS")
print("-" * 100)

for path in GENERATED15 + [MANIFEST_PATH15]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        "PASS:",
        path.relative_to(OUTPUT15)
    )

print("\nREPORT VALIDATION")
print("-" * 100)

print("GCN/GAT shared records: PASS")
print("Original/optimized comparison: PASS")
print("Validation counts: PASS")
print("Precision/recall/F1/FPR recomputation: PASS")

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Forecasting checkpoints modified: NO")
print("Original reports modified: NO")
print("Previous research outputs overwritten: NO")

print("=" * 100)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 16 — CURRENT ATTACK TEN-CLASS PERFORMANCE
#
# FIGURE F12:
# A. V1 ten-class row-normalized confusion matrix
# B. V1 per-class precision, recall and F1
# C. Four-model overall validation comparison
# D. Largest directed attack-class confusions
#
# Uses saved JSON metrics only.
# No UNSW training-data loading.
# No model training.
# No official TEST access.
# No existing files overwritten.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle

print("=" * 96)
print("CELL 16 — CURRENT ATTACK TEN-CLASS PERFORMANCE")
print("=" * 96)

# ----------------------------------------------------------------------
# A. EXACT SAVED EXPERIMENT PATHS
# ----------------------------------------------------------------------

PROJECT16 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DETECTION16 = PROJECT16 / "Current_Attack_Detection"
OUTPUT16 = PROJECT16 / "Research_Outputs"

FIG16 = OUTPUT16 / "01_Figures" / "Current_Detection"
TAB16 = OUTPUT16 / "02_Tables"
MET16 = OUTPUT16 / "03_Metrics"
MAN16 = OUTPUT16 / "05_Manifests"

for folder in [
    DETECTION16, FIG16, TAB16, MET16, MAN16
]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

PATHS16 = {
    "XGBoost V1": (
        DETECTION16 / "artifacts"
        / "xgboost_20261027_023158_885060"
        / "validation_metrics.json"
    ),
    "Weighted XGBoost V2": (
        DETECTION16 / "experiments"
        / "weighted_xgb_v2_20261027_035816_500107"
        / "metadata.json"
    ),
    "Conflict-aware XGBoost": (
        DETECTION16 / "experiments"
        / "conflict_aware_xgb_20261027_042501_508802"
        / "metadata.json"
    ),
    "CatBoost": (
        DETECTION16 / "experiments"
        / "catboost_20261027_034931_720016"
        / "metadata.json"
    )
}

for name, path in PATHS16.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

# ----------------------------------------------------------------------
# B. LOAD SAVED JSON FILES
# ----------------------------------------------------------------------

SAVED16 = {}

for model, path in PATHS16.items():

    with path.open("r", encoding="utf-8") as f:
        SAVED16[model] = json.load(f)

v1_16 = SAVED16["XGBoost V1"]

CLASSES16 = v1_16["class_order"]

if len(CLASSES16) != 10:
    raise RuntimeError("Expected exactly 10 classes.")

if len(set(CLASSES16)) != 10:
    raise RuntimeError("Duplicate class names.")

for model, key in [
    ("Weighted XGBoost V2", "classes"),
    ("Conflict-aware XGBoost", "classes"),
    ("CatBoost", "class_names")
]:

    if SAVED16[model][key] != CLASSES16:
        raise RuntimeError(
            f"{model}: inconsistent class order"
        )

for model, metadata in SAVED16.items():

    if metadata.get("official_test_used") is not False:
        raise RuntimeError(
            f"{model}: TEST-use status not verified"
        )

CM16 = np.asarray(
    v1_16["confusion_matrix"]
)

if CM16.shape != (10, 10):
    raise RuntimeError(
        "Expected a 10x10 confusion matrix."
    )

if not np.issubdtype(CM16.dtype, np.number):
    raise RuntimeError("Non-numeric confusion matrix.")

if not np.isfinite(CM16).all():
    raise RuntimeError("Nonfinite confusion counts.")

if (CM16 < 0).any():
    raise RuntimeError("Negative confusion counts.")

if not np.all(CM16 == np.round(CM16)):
    raise RuntimeError("Noninteger confusion counts.")

CM16 = CM16.astype(np.int64)

SUPPORT16 = CM16.sum(axis=1)
PREDICTED_COUNT16 = CM16.sum(axis=0)
TRUE_POSITIVES16 = np.diag(CM16)

TOTAL16 = int(CM16.sum())

if TOTAL16 != 35167:
    raise RuntimeError(
        f"Expected 35,167 validation rows; got {TOTAL16}"
    )

if TOTAL16 != int(v1_16["validation_rows"]):
    raise RuntimeError("Saved validation row mismatch.")

EXPECTED_SUPPORT16 = {
    "Analysis": 423,
    "Backdoor": 343,
    "DoS": 2552,
    "Exploits": 6700,
    "Fuzzers": 3747,
    "Generic": 8040,
    "Normal": 11191,
    "Reconnaissance": 1920,
    "Shellcode": 229,
    "Worms": 22
}

for name, observed in zip(CLASSES16, SUPPORT16):

    if int(observed) != EXPECTED_SUPPORT16[name]:
        raise RuntimeError(
            f"Unexpected support for {name}: {observed}"
        )

# ----------------------------------------------------------------------
# C. RECOMPUTE ORIGINAL CLASSIFICATION METRICS
# ----------------------------------------------------------------------

def divide16(a, b):

    return np.divide(
        a,
        b,
        out=np.zeros_like(
            np.asarray(a, dtype=float)
        ),
        where=np.asarray(b) != 0
    )

PRECISION16 = divide16(
    TRUE_POSITIVES16,
    PREDICTED_COUNT16
)

RECALL16 = divide16(
    TRUE_POSITIVES16,
    SUPPORT16
)

F1_16 = divide16(
    2 * PRECISION16 * RECALL16,
    PRECISION16 + RECALL16
)

ACCURACY16 = (
    TRUE_POSITIVES16.sum() / TOTAL16
)

BALANCED16 = float(
    np.mean(RECALL16)
)

MACRO_F1_16 = float(
    np.mean(F1_16)
)

WEIGHTED_F1_16 = float(
    np.average(
        F1_16,
        weights=SUPPORT16
    )
)

CHECKS16 = {
    "accuracy": ACCURACY16,
    "balanced_accuracy": BALANCED16,
    "macro_f1": MACRO_F1_16,
    "weighted_f1": WEIGHTED_F1_16
}

for key, calculated in CHECKS16.items():

    saved = float(v1_16[key])

    np.testing.assert_allclose(
        calculated,
        saved,
        atol=1e-10,
        rtol=1e-10,
        err_msg=f"V1 {key} mismatch"
    )

print("\nA. ORIGINAL XGBOOST VALIDATION")
print("-" * 96)

print("Validation records:", TOTAL16)
print("Attack classes:", len(CLASSES16))
print(f"Accuracy:          {ACCURACY16:.6f}")
print(f"Balanced accuracy: {BALANCED16:.6f}")
print(f"Macro F1:          {MACRO_F1_16:.6f}")
print(f"Weighted F1:       {WEIGHTED_F1_16:.6f}")
print("Confusion-matrix metrics verified: PASS")

# ----------------------------------------------------------------------
# D. COLLECT ALL FOUR EXPERIMENT SUMMARIES
# ----------------------------------------------------------------------

MODEL_ORDER16 = [
    "XGBoost V1",
    "Weighted XGBoost V2",
    "Conflict-aware XGBoost",
    "CatBoost"
]

def get_metric16(model, metric):

    data = SAVED16[model]

    if model == "XGBoost V1":

        key_map = {
            "Accuracy": "accuracy",
            "Macro F1": "macro_f1",
            "Balanced accuracy": "balanced_accuracy",
            "Weighted F1": "weighted_f1"
        }

        return float(data[key_map[metric]])

    container = (
        "metrics"
        if model == "CatBoost"
        else "validation_metrics"
    )

    if container not in data:
        raise RuntimeError(
            f"{model}: missing {container}"
        )

    if metric not in data[container]:
        raise RuntimeError(
            f"{model}: missing {metric}"
        )

    return float(data[container][metric])

SCORES16 = pd.DataFrame([
    {
        "Model": model,
        "Accuracy": get_metric16(model, "Accuracy"),
        "Macro_F1": get_metric16(model, "Macro F1"),
        "Balanced_Accuracy": get_metric16(
            model, "Balanced accuracy"
        ),
        "Weighted_F1": get_metric16(
            model, "Weighted F1"
        )
    }
    for model in MODEL_ORDER16
])

if not SCORES16.drop(
    columns="Model"
).apply(
    lambda s: s.between(0, 1).all()
).all():
    raise RuntimeError("Model score outside [0,1].")

print("\nB. SAVED MODEL COMPARISON")
print("-" * 96)
print(SCORES16.round(5).to_string(index=False))

# ----------------------------------------------------------------------
# E. V1 PER-CLASS TABLES
# ----------------------------------------------------------------------

CLASS_TABLE16 = pd.DataFrame({
    "Class": CLASSES16,
    "Support": SUPPORT16,
    "Predicted_Count": PREDICTED_COUNT16,
    "TP": TRUE_POSITIVES16,
    "Precision": PRECISION16,
    "Recall": RECALL16,
    "F1": F1_16
})

NORMALIZED16 = divide16(
    CM16.astype(float),
    SUPPORT16[:, None]
)

CONFUSION_ROWS16 = []

for i, true_label in enumerate(CLASSES16):

    for j, predicted_label in enumerate(CLASSES16):

        CONFUSION_ROWS16.append({
            "True_Class": true_label,
            "Predicted_Class": predicted_label,
            "Count": int(CM16[i, j]),
            "True_Class_Support": int(SUPPORT16[i]),
            "Fraction_Of_True_Class": float(
                NORMALIZED16[i, j]
            )
        })

CONFUSION_TABLE16 = pd.DataFrame(
    CONFUSION_ROWS16
)

ERRORS16 = (
    CONFUSION_TABLE16.loc[
        lambda d: (
            (d["True_Class"] != d["Predicted_Class"])
            & (d["Count"] > 0)
        )
    ]
    .sort_values(
        "Count",
        ascending=False
    )
    .reset_index(drop=True)
)

# ----------------------------------------------------------------------
# F. PUBLICATION-QUALITY FOUR-PANEL FIGURE
# ----------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig16, axes16 = plt.subplots(
    2, 2,
    figsize=(19, 15),
    gridspec_kw={
        "width_ratios": [1.12, 1],
        "height_ratios": [1.18, 1],
        "wspace": 0.38,
        "hspace": 0.38
    }
)

fig16.suptitle(
    "Current network attack classification: performance and failure modes",
    x=0.085,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig16.text(
    0.085,
    0.953,
    "UNSW-NB15 development validation | "
    "35,167 held-out development-fold records | "
    "10 attack/normal classes",
    fontsize=11,
    color="#586576"
)

# ----------------------------------------------------------------------
# PANEL A — TEN-CLASS CONFUSION MATRIX
# ----------------------------------------------------------------------

ax = axes16[0, 0]

image16 = ax.imshow(
    NORMALIZED16,
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    interpolation="nearest",
    aspect="equal"
)

ax.set_xticks(range(10))
ax.set_yticks(range(10))

ax.set_xticklabels(
    CLASSES16,
    rotation=40,
    ha="right",
    fontsize=8.4
)

ax.set_yticklabels(
    [
        f"{name} (n={int(support):,})"
        for name, support in zip(
            CLASSES16, SUPPORT16
        )
    ],
    fontsize=8.4
)

for i in range(10):

    for j in range(10):

        share = NORMALIZED16[i, j]
        count = int(CM16[i, j])

        if count == 0:
            label = "·"

        elif share >= 0.015 or i == j:
            label = f"{share*100:.1f}%\n({count:,})"

        else:
            label = f"{count:,}"

        ax.text(
            j, i,
            label,
            ha="center",
            va="center",
            fontsize=7.5,
            color=(
                "white" if share > 0.56
                else "#203247"
            ),
            fontweight=(
                "bold" if i == j else "normal"
            )
        )

    ax.add_patch(
        Rectangle(
            (i-0.48, i-0.48),
            0.96,
            0.96,
            fill=False,
            edgecolor="#D39049",
            linewidth=1.2
        )
    )

ax.set_xlabel("Predicted current attack class")
ax.set_ylabel("Actual current attack class")

ax.set_title(
    "A   Original XGBoost: ten-class confusion",
    loc="left",
    fontweight="bold",
    pad=15
)

fig16.colorbar(
    image16,
    ax=ax,
    fraction=0.046,
    pad=0.025,
    label="Fraction of true-class records"
)

# ----------------------------------------------------------------------
# PANEL B — PER-CLASS PRECISION, RECALL, F1
# ----------------------------------------------------------------------

ax = axes16[0, 1]

class_sorted16 = CLASS_TABLE16.sort_values(
    "Recall",
    ascending=True
)

yy = np.arange(len(class_sorted16))

ax.barh(
    yy,
    class_sorted16["Recall"],
    height=0.58,
    color="#438F9A",
    alpha=0.78,
    label="Recall"
)

ax.scatter(
    class_sorted16["Precision"],
    yy + 0.13,
    marker="D",
    s=38,
    color="#A66A41",
    label="Precision",
    zorder=4
)

ax.scatter(
    class_sorted16["F1"],
    yy - 0.13,
    marker="o",
    s=44,
    color="#7F59A3",
    label="F1",
    zorder=4
)

ax.axvline(
    0.9,
    color="#B65256",
    linestyle="--",
    linewidth=1,
    alpha=0.75
)

ax.set_yticks(yy)

ax.set_yticklabels(
    [
        f"{row.Class} (n={int(row.Support):,})"
        for row in class_sorted16.itertuples()
    ],
    fontsize=9
)

ax.set_xlim(0, 1.04)
ax.set_xlabel("Classification metric")
ax.set_title(
    "B   Performance for each attack class",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.legend(
    frameon=False,
    loc="lower right",
    fontsize=9
)

ax.grid(axis="x", linestyle=":", alpha=0.35)
ax.set_axisbelow(True)

# ----------------------------------------------------------------------
# PANEL C — FOUR-MODEL OVERALL SCORE COMPARISON
# ----------------------------------------------------------------------

ax = axes16[1, 0]

metrics_display16 = [
    ("Accuracy", "Accuracy", "#487DA6"),
    ("Macro_F1", "Macro F1", "#9255A3"),
    ("Balanced_Accuracy", "Balanced accuracy", "#3C9787")
]

y_base = np.arange(len(SCORES16))
bar_height = 0.22

for j, (key, name, color) in enumerate(
    metrics_display16
):

    positions = y_base + (j-1)*bar_height

    bars = ax.barh(
        positions,
        SCORES16[key],
        height=bar_height*0.92,
        color=color,
        label=name,
        alpha=0.90
    )

    for bar, value in zip(bars, SCORES16[key]):

        ax.text(
            value + 0.013,
            bar.get_y() + bar.get_height()/2,
            f"{value:.3f}",
            va="center",
            fontsize=8.2,
            color="#28394C"
        )

ax.set_yticks(y_base)
ax.set_yticklabels(
    SCORES16["Model"],
    fontsize=9
)
ax.invert_yaxis()

ax.set_xlim(0, 1.06)
ax.set_xlabel("Development-validation score")

ax.set_title(
    "C   Overall comparison across four classifiers",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.grid(axis="x", linestyle=":", alpha=0.35)
ax.set_axisbelow(True)

ax.legend(
    frameon=False,
    loc="lower right",
    fontsize=8.8
)

# ----------------------------------------------------------------------
# PANEL D — MAIN MISCLASSIFICATION PATHWAYS
# ----------------------------------------------------------------------

ax = axes16[1, 1]

top_errors16 = (
    ERRORS16.head(9)
    .sort_values("Count", ascending=True)
)

labels16 = [
    f"{r.True_Class} → {r.Predicted_Class}"
    for r in top_errors16.itertuples()
]

bars = ax.barh(
    labels16,
    top_errors16["Count"],
    color=[
        "#B35F56"
        if "DoS" in true_name and pred_name == "Exploits"
        else "#5B82A3"
        for true_name, pred_name in zip(
            top_errors16["True_Class"],
            top_errors16["Predicted_Class"]
        )
    ],
    height=0.67
)

max_error16 = max(
    int(top_errors16["Count"].max()), 1
)

ax.set_xlim(0, max_error16*1.33)

for bar, row in zip(
    bars,
    top_errors16.itertuples()
):

    ax.text(
        bar.get_width() + max_error16*0.018,
        bar.get_y() + bar.get_height()/2,
        (
            f"{int(row.Count):,}  "
            f"({100*row.Fraction_Of_True_Class:.1f}%)"
        ),
        ha="left",
        va="center",
        fontsize=8.7,
        color="#26374A"
    )

ax.set_xlabel("Misclassified validation records")

ax.set_title(
    "D   Most frequent class-confusion pathways",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.grid(
    axis="x",
    linestyle=":",
    alpha=0.35
)

ax.set_axisbelow(True)

# ----------------------------------------------------------------------
# G. FINAL LAYOUT AND SCIENTIFIC CAPTION
# ----------------------------------------------------------------------

for ax in axes16.flat:
    ax.set_facecolor("#FAFBFC")

fig16.subplots_adjust(
    left=0.145,
    right=0.965,
    top=0.88,
    bottom=0.13
)

fig16.text(
    0.085,
    0.060,
    "Original XGBoost provides the confusion matrix and "
    "class-wise measurements; the other model comparisons "
    "use saved experimental validation summaries.",
    fontsize=9,
    color="#576575"
)

fig16.text(
    0.085,
    0.042,
    "Development validation was repeatedly used for "
    "model selection and diagnostics. These results are "
    "not an independent TEST-set estimate. "
    "No model was retrained for this figure.",
    fontsize=9,
    color="#576575"
)

# ----------------------------------------------------------------------
# H. SAVE NEW FIGURE AND RESEARCH TABLES
# ----------------------------------------------------------------------

STAMP16 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM16 = (
    f"F12_Current_Attack_Multiclass_Analysis_{STAMP16}"
)

FIGURE_FILES16 = []

for ext in ["pdf", "svg", "png"]:

    path = FIG16 / f"{STEM16}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig16.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES16.append(path)

plt.show()
plt.close(fig16)

TABLE_PATHS16 = {
    "T15_Current_Attack_Model_Comparison":
        SCORES16,

    "T16_Current_Attack_Class_Performance":
        CLASS_TABLE16,

    "T17_Current_Attack_Confusion_Details":
        CONFUSION_TABLE16
}

SAVED_TABLES16 = []

for name, frame in TABLE_PATHS16.items():

    path = TAB16 / f"{name}_{STAMP16}.csv"

    frame.to_csv(
        path,
        index=False,
        mode="x",
        float_format="%.8f"
    )

    SAVED_TABLES16.append(path)

# ----------------------------------------------------------------------
# I. MACHINE-READABLE METRICS
# ----------------------------------------------------------------------

METRIC_PATH16 = (
    MET16
    / f"current_attack_multiclass_metrics_{STAMP16}.json"
)

metrics_data16 = {
    "project": "NetGuard",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "split": "development_validation",
    "validation_rows": TOTAL16,
    "classes": CLASSES16,
    "baseline_model": "XGBoost V1",
    "baseline_metrics_recomputed": {
        "accuracy": float(ACCURACY16),
        "balanced_accuracy": float(BALANCED16),
        "macro_f1": float(MACRO_F1_16),
        "weighted_f1": float(WEIGHTED_F1_16)
    },
    "model_comparison": SCORES16.to_dict(
        orient="records"
    ),
    "independent_test_evaluated": False,
    "models_retrained": 0
}

with METRIC_PATH16.open(
    "x",
    encoding="utf-8"
) as f:

    json.dump(
        metrics_data16,
        f,
        indent=2,
        allow_nan=False
    )

# ----------------------------------------------------------------------
# J. MANIFEST WITH INPUT AND OUTPUT HASHES
# ----------------------------------------------------------------------

def hash16(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

GENERATED16 = (
    FIGURE_FILES16
    + SAVED_TABLES16
    + [METRIC_PATH16]
)

MANIFEST_PATH16 = (
    MAN16
    / f"current_attack_multiclass_manifest_{STAMP16}.json"
)

manifest16 = {
    "project": "NetGuard",
    "figure": STEM16,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "figure_panels": 4,
    "model_sources": [
        {
            "model": model,
            "path": str(path.relative_to(PROJECT16)),
            "sha256": hash16(path)
        }
        for model, path in PATHS16.items()
    ],
    "generated_files": [
        {
            "path": str(path.relative_to(PROJECT16)),
            "sha256": hash16(path)
        }
        for path in GENERATED16
    ],
    "limitations": [
        "Detailed per-class confusion is available for V1 only.",
        "Other model scores come from saved experiment summaries.",
        "Repeated development-validation comparisons are exploratory.",
        "Accuracy alone does not describe rare-class performance.",
        "No independent TEST evaluation was conducted.",
        "No models were retrained."
    ],
    "official_test_accessed": False,
    "models_retrained": 0
}

with MANIFEST_PATH16.open(
    "x",
    encoding="utf-8"
) as f:

    json.dump(
        manifest16,
        f,
        indent=2,
        allow_nan=False
    )

# ----------------------------------------------------------------------
# K. FINAL VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "=" * 96)
print("CELL 16 COMPLETE")
print("=" * 96)

print("Current attack classifiers compared:", len(SCORES16))
print("Attack classes:", len(CLASSES16))
print("Validation records:", TOTAL16)
print("Figure panels:", 4)
print("Saved V1 confusion verification: PASS")

print("\nMODEL QUALITY COMPARISON")
print("-" * 96)

print(
    SCORES16
    .round(4)
    .to_string(index=False)
)

print("\nMAIN ERROR PATHWAYS")
print("-" * 96)

print(
    ERRORS16.head(8)[
        [
            "True_Class",
            "Predicted_Class",
            "Count",
            "Fraction_Of_True_Class"
        ]
    ].round(4).to_string(index=False)
)

print("\nSAVED OUTPUTS")
print("-" * 96)

for path in GENERATED16 + [MANIFEST_PATH16]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        "PASS:",
        path.relative_to(OUTPUT16)
    )

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Forecasting checkpoints modified: NO")
print("Current detection checkpoints modified: NO")
print("Previous research outputs overwritten: NO")

print("=" * 96)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 17 — DATASET AMBIGUITY AND DoS / EXPLOITS ANALYSIS
#
# FIGURE F13:
# A. Conflicting-label prevalence by attack class
# B. Error decomposition by class and ambiguity status
# C. DoS / Exploits trade-off: XGBoost V1 vs weighted V2
# D. Validation accuracy and duplicate-label error floor
#
# SOURCES:
#   Saved V1 confusion matrix (machine verified)
#   Saved weighted V2 model metadata (machine verified)
#   Exact printed diagnostics from Cells 19-22 (transcribed)
#
# No retraining.
# No official TEST access.
# No previous outputs overwritten.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 96)
print("CELL 17 — LABEL AMBIGUITY AND DoS / EXPLOITS DIAGNOSTICS")
print("=" * 96)

# ----------------------------------------------------------------------
# A. PATHS AND SOURCE FILES
# ----------------------------------------------------------------------

PROJECT17 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DETECTION17 = PROJECT17 / "Current_Attack_Detection"
OUTPUT17 = PROJECT17 / "Research_Outputs"

FIG17 = OUTPUT17 / "01_Figures" / "Current_Detection"
TAB17 = OUTPUT17 / "02_Tables"
MET17 = OUTPUT17 / "03_Metrics"
MAN17 = OUTPUT17 / "05_Manifests"

for folder in [FIG17, TAB17, MET17, MAN17]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)

V1_PATH17 = (
    DETECTION17 / "artifacts"
    / "xgboost_20261027_023158_885060"
    / "validation_metrics.json"
)

V2_PATH17 = (
    DETECTION17 / "experiments"
    / "weighted_xgb_v2_20261027_035816_500107"
    / "metadata.json"
)

for path in [V1_PATH17, V2_PATH17]:
    if not path.is_file():
        raise FileNotFoundError(path)

with V1_PATH17.open("r", encoding="utf-8") as f:
    v1_17 = json.load(f)

with V2_PATH17.open("r", encoding="utf-8") as f:
    v2_17 = json.load(f)

CLASSES17 = v1_17["class_order"]

if CLASSES17 != v2_17["classes"]:
    raise RuntimeError("V1 and V2 class orders differ")

if (
    v1_17.get("official_test_used") is not False
    or v2_17.get("official_test_used") is not False
):
    raise RuntimeError("TEST-data status not verified")

CM17 = np.asarray(
    v1_17["confusion_matrix"],
    dtype=np.int64
)

if CM17.shape != (10, 10) or (CM17 < 0).any():
    raise RuntimeError("Invalid V1 confusion matrix")

SUPPORT17 = CM17.sum(axis=1)
ERRORS17 = SUPPORT17 - np.diag(CM17)

N17 = int(CM17.sum())

if N17 != 35167:
    raise RuntimeError("Validation row-count mismatch")

print("\nA. SOURCE VERIFICATION")
print("-" * 96)
print("Development validation records:", N17)
print("Classes:", len(CLASSES17))
print("Saved V1 accuracy:", v1_17["accuracy"])
print(
    "Saved V2 macro F1:",
    v2_17["validation_metrics"]["Macro F1"]
)
print("Both source metadata files verified: PASS")

# ----------------------------------------------------------------------
# B. TRANSCRIBED AMBIGUITY AUDIT
#
# These are EXACT counts from previous notebook output,
# not newly recomputed feature-group statistics.
#
# Cell 20: class-specific ambiguous/error breakdown
# Cell 19: validation conflict floor
# Cell 22: weighted model per-class recalls
#
# Every available arithmetic identity is verified below.
# ----------------------------------------------------------------------

TRANSCRIBED17 = {
    "Analysis": {
        "ambiguous": 289,
        "errors_ambiguous": 289,
        "errors_unambiguous": 81
    },
    "Backdoor": {
        "ambiguous": 267,
        "errors_ambiguous": 256,
        "errors_unambiguous": 36
    },
    "DoS": {
        "ambiguous": 2023,
        "errors_ambiguous": 2018,
        "errors_unambiguous": 380
    },
    "Exploits": {
        "ambiguous": 2795,
        "errors_ambiguous": 47,
        "errors_unambiguous": 290
    },
    "Fuzzers": {
        "ambiguous": 449,
        "errors_ambiguous": 334,
        "errors_unambiguous": 635
    },
    "Generic": {
        "ambiguous": 67,
        "errors_ambiguous": 65,
        "errors_unambiguous": 84
    },
    "Normal": {
        "ambiguous": 93,
        "errors_ambiguous": 69,
        "errors_unambiguous": 818
    },
    "Reconnaissance": {
        "ambiguous": 399,
        "errors_ambiguous": 383,
        "errors_unambiguous": 177
    },
    "Shellcode": {
        "ambiguous": 3,
        "errors_ambiguous": 0,
        "errors_unambiguous": 50
    },
    "Worms": {
        "ambiguous": 2,
        "errors_ambiguous": 2,
        "errors_unambiguous": 9
    }
}

# Rounded recall values from Cell 22.
# They are not reconstructed per-row predictions.
WEIGHTED_RECALL17 = {
    "DoS": 0.6391,
    "Exploits": 0.6646
}

CONFLICT_ERROR_FLOOR17 = 3366

rows17 = []

for idx, label in enumerate(CLASSES17):

    audit = TRANSCRIBED17[label]

    support = int(SUPPORT17[idx])
    ambiguous = int(audit["ambiguous"])
    nonambiguous = support - ambiguous

    error_ambiguous = int(audit["errors_ambiguous"])
    error_nonambiguous = int(
        audit["errors_unambiguous"]
    )

    if not 0 <= ambiguous <= support:
        raise RuntimeError(f"{label}: ambiguous support invalid")

    if error_ambiguous > ambiguous:
        raise RuntimeError(f"{label}: ambiguous errors invalid")

    if error_nonambiguous > nonambiguous:
        raise RuntimeError(f"{label}: clear errors invalid")

    if error_ambiguous + error_nonambiguous != ERRORS17[idx]:
        raise RuntimeError(
            f"{label}: transcribed errors disagree "
            "with saved V1 confusion matrix"
        )

    rows17.append({
        "Class": label,
        "Support": support,
        "Ambiguous_Records": ambiguous,
        "Unambiguous_Records": nonambiguous,
        "Ambiguous_Fraction": ambiguous / support,
        "Errors_Ambiguous": error_ambiguous,
        "Errors_Unambiguous": error_nonambiguous,
        "Total_Errors": int(ERRORS17[idx]),
        "V1_Recall": (
            float(CM17[idx, idx]) / support
        )
    })

AUDIT17 = pd.DataFrame(rows17)

AMBIGUOUS17 = int(AUDIT17["Ambiguous_Records"].sum())
CLEAR17 = int(AUDIT17["Unambiguous_Records"].sum())

AMBIGUOUS_ERRORS17 = int(
    AUDIT17["Errors_Ambiguous"].sum()
)

CLEAR_ERRORS17 = int(
    AUDIT17["Errors_Unambiguous"].sum()
)

TOTAL_ERRORS17 = int(
    AUDIT17["Total_Errors"].sum()
)

# Cross-check exact earlier results.
assert AMBIGUOUS17 == 6387
assert CLEAR17 == 28780
assert AMBIGUOUS_ERRORS17 == 3463
assert CLEAR_ERRORS17 == 2560
assert TOTAL_ERRORS17 == 6023
assert TOTAL_ERRORS17 == N17 - np.trace(CM17)

assert (
    CONFLICT_ERROR_FLOOR17
    <= AMBIGUOUS_ERRORS17
)

ACCURACY_CEILING17 = (
    1 - CONFLICT_ERROR_FLOOR17 / N17
)

V1_ACCURACY17 = (
    1 - TOTAL_ERRORS17 / N17
)

np.testing.assert_allclose(
    V1_ACCURACY17,
    float(v1_17["accuracy"]),
    atol=1e-12
)

print("\nB. AMBIGUITY VALIDATION")
print("-" * 96)

print("Ambiguous validation records:", AMBIGUOUS17)
print("Unambiguous validation records:", CLEAR17)
print("Errors in ambiguous records:", AMBIGUOUS_ERRORS17)
print("Errors in unambiguous records:", CLEAR_ERRORS17)
print("Total baseline errors:", TOTAL_ERRORS17)

print(
    "Identical-feature conflicting-label error floor:",
    CONFLICT_ERROR_FLOOR17
)

print(
    "Optimistic validation accuracy ceiling:",
    f"{ACCURACY_CEILING17:.4%}"
)

print("Arithmetic consistency: PASS")

# ----------------------------------------------------------------------
# C. DoS / EXPLOITS RECALL COMPARISON
# ----------------------------------------------------------------------

dos_index17 = CLASSES17.index("DoS")
exploit_index17 = CLASSES17.index("Exploits")

DOS_RECALL_V1_17 = (
    CM17[dos_index17, dos_index17]
    / SUPPORT17[dos_index17]
)

EXPLOIT_RECALL_V1_17 = (
    CM17[exploit_index17, exploit_index17]
    / SUPPORT17[exploit_index17]
)

# Check the earlier rounded Cell 22 values against the
# saved baseline matrix.
assert abs(DOS_RECALL_V1_17 - 0.0603) < 0.0001
assert abs(EXPLOIT_RECALL_V1_17 - 0.9497) < 0.0001

TRADEOFF17 = pd.DataFrame([
    {
        "Class": "DoS",
        "V1_Recall": DOS_RECALL_V1_17,
        "V2_Recall": WEIGHTED_RECALL17["DoS"]
    },
    {
        "Class": "Exploits",
        "V1_Recall": EXPLOIT_RECALL_V1_17,
        "V2_Recall": WEIGHTED_RECALL17["Exploits"]
    }
])

TRADEOFF17["Recall_Change"] = (
    TRADEOFF17["V2_Recall"]
    - TRADEOFF17["V1_Recall"]
)

print("\nC. DoS / EXPLOITS RECALL TRADE-OFF")
print("-" * 96)

print(
    TRADEOFF17.round(5).to_string(index=False)
)

# ----------------------------------------------------------------------
# D. PUBLICATION FIGURE
# ----------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig17, axes17 = plt.subplots(
    2, 2,
    figsize=(17.5, 13.2),
    gridspec_kw={
        "wspace": 0.32,
        "hspace": 0.40
    }
)

fig17.suptitle(
    "Ambiguous network-flow labels and attack-class trade-offs",
    x=0.09,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig17.text(
    0.09,
    0.950,
    "UNSW-NB15 development validation | "
    "Identical feature vectors with conflicting labels | "
    "DoS versus Exploits",
    fontsize=10.5,
    color="#586576"
)

COLOR_CLEAR17 = "#5B94A5"
COLOR_AMBIG17 = "#CF8263"
COLOR_V1_17 = "#426F9D"
COLOR_V2_17 = "#338D81"

def style17(ax):
    ax.set_facecolor("#FAFBFC")
    ax.grid(
        axis="x",
        linestyle=":",
        linewidth=0.7,
        alpha=0.40
    )
    ax.set_axisbelow(True)

# ----------------------------------------------------------------------
# PANEL A — CLASS-SPECIFIC AMBIGUITY
# ----------------------------------------------------------------------

ax = axes17[0, 0]
style17(ax)

order17 = (
    AUDIT17
    .sort_values("Ambiguous_Fraction", ascending=True)
)

yy17 = np.arange(len(order17))

ax.barh(
    yy17,
    order17["Ambiguous_Fraction"] * 100,
    color=COLOR_AMBIG17,
    height=0.67,
    label="Ambiguous feature signature"
)

ax.set_yticks(yy17)

ax.set_yticklabels([
    f"{r.Class} (n={int(r.Support):,})"
    for r in order17.itertuples()
])

ax.set_xlim(0, 92)
ax.set_xlabel(
    "Records sharing features with conflicting labels (%)"
)

ax.set_title(
    "A   Label ambiguity by attack class",
    loc="left",
    fontweight="bold",
    pad=15
)

for y, r in enumerate(order17.itertuples()):

    ax.text(
        r.Ambiguous_Fraction*100 + 1.3,
        y,
        f"{r.Ambiguous_Fraction*100:.1f}%",
        va="center",
        fontsize=9
    )

# ----------------------------------------------------------------------
# PANEL B — ERRORS WITH AND WITHOUT FEATURE CONFLICTS
# ----------------------------------------------------------------------

ax = axes17[0, 1]
style17(ax)

error_order17 = (
    AUDIT17
    .sort_values("Total_Errors", ascending=True)
)

y17 = np.arange(len(error_order17))

ax.barh(
    y17,
    error_order17["Errors_Ambiguous"],
    color=COLOR_AMBIG17,
    height=0.68,
    label="Errors in ambiguous records"
)

ax.barh(
    y17,
    error_order17["Errors_Unambiguous"],
    left=error_order17["Errors_Ambiguous"],
    color=COLOR_CLEAR17,
    height=0.68,
    label="Errors in unambiguous records"
)

ax.set_yticks(y17)
ax.set_yticklabels(
    error_order17["Class"]
)

ax.set_xlabel("Misclassified validation records")

ax.set_title(
    "B   Where the original classifier makes errors",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.legend(
    loc="lower right",
    frameon=False,
    fontsize=8.5
)

# ----------------------------------------------------------------------
# PANEL C — DoS / EXPLOITS WEIGHTING TRADE-OFF
# ----------------------------------------------------------------------

ax = axes17[1, 0]
style17(ax)

x17 = np.arange(2)
width17 = 0.34

b1 = ax.bar(
    x17-width17/2,
    TRADEOFF17["V1_Recall"]*100,
    width17,
    color=COLOR_V1_17,
    label="Original XGBoost V1"
)

b2 = ax.bar(
    x17+width17/2,
    TRADEOFF17["V2_Recall"]*100,
    width17,
    color=COLOR_V2_17,
    label="Class-weighted XGBoost V2"
)

ax.set_xticks(x17)
ax.set_xticklabels(
    ["DoS", "Exploits"],
    fontsize=11
)

ax.set_ylim(0, 108)

ax.set_ylabel("Class recall (%)")

ax.set_title(
    "C   Class weighting shifts DoS/Exploits decisions",
    loc="left",
    fontweight="bold",
    pad=15
)

for bars in [b1, b2]:

    for bar in bars:

        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height()+1.8,
            f"{bar.get_height():.1f}%",
            ha="center",
            fontsize=9,
            fontweight="medium"
        )

ax.legend(
    loc="upper center",
    frameon=False,
    fontsize=8.5
)

# ----------------------------------------------------------------------
# PANEL D — VALIDATION ACCURACY ERROR DECOMPOSITION
# ----------------------------------------------------------------------

ax = axes17[1, 1]

correct17 = N17 - TOTAL_ERRORS17
extra_errors17 = (
    TOTAL_ERRORS17 - CONFLICT_ERROR_FLOOR17
)

# Stack the recorded correct predictions, extra observed
# errors, and feature-conflict error floor as percentages.

segments17 = [
    ("Correct V1 predictions", correct17, "#418C82"),
    ("Errors above conflict floor", extra_errors17, "#E1A764"),
    ("Conflict error floor", CONFLICT_ERROR_FLOOR17, "#BB6262")
]

left17 = 0

for name, count, color in segments17:

    percentage = 100*count/N17

    ax.barh(
        0,
        percentage,
        left=left17,
        height=0.42,
        color=color,
        label=name,
        edgecolor="white"
    )

    ax.text(
        left17 + percentage/2,
        0,
        f"{percentage:.1f}%",
        ha="center",
        va="center",
        fontsize=9.5,
        color="white" if percentage > 12 else "#213042",
        fontweight="bold"
    )

    left17 += percentage

ax.axvline(
    ACCURACY_CEILING17*100,
    color="#A0474F",
    linestyle="--",
    linewidth=1.3
)

ax.text(
    99,
    0.39,
    f"Optimistic deterministic ceiling: "
    f"{ACCURACY_CEILING17:.2%}",
    ha="right",
    fontsize=9.3,
    color="#9C454F"
)

ax.text(
    0,
    -0.42,
    f"Original XGBoost accuracy: {V1_ACCURACY17:.2%}",
    fontsize=10.3,
    fontweight="medium"
)

ax.set_xlim(0, 100)
ax.set_ylim(-0.75, 0.8)
ax.set_yticks([])
ax.set_xlabel("Share of development-validation records (%)")

ax.set_title(
    "D   Observed errors and optimistic accuracy ceiling",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.13),
    ncol=2,
    frameon=False,
    fontsize=8.2
)

# ----------------------------------------------------------------------
# E. SCIENTIFIC LIMITATIONS
# ----------------------------------------------------------------------

fig17.subplots_adjust(
    left=0.145,
    right=0.965,
    top=0.875,
    bottom=0.13
)

fig17.text(
    0.09,
    0.060,
    "Conflict counts are transcribed from previously executed "
    "Cells 19–21 and checked against the saved V1 confusion matrix. "
    "V2 recalls are rounded Cell 22 diagnostics.",
    fontsize=9,
    color="#576575"
)

fig17.text(
    0.09,
    0.040,
    "The 90.43% ceiling is an optimistic in-sample bound for "
    "deterministic classification of the observed feature vectors, "
    "not an estimate of achievable test accuracy.",
    fontsize=9,
    color="#576575"
)

# ----------------------------------------------------------------------
# F. SAVE FIGURE AND TABLES
# ----------------------------------------------------------------------

STAMP17 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM17 = (
    f"F13_Label_Ambiguity_And_DoS_Tradeoff_{STAMP17}"
)

FIGURE_FILES17 = []

for ext in ["pdf", "svg", "png"]:

    path = FIG17 / f"{STEM17}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig17.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES17.append(path)

plt.show()
plt.close(fig17)

TABLES17 = {
    "T18_Label_Ambiguity_By_Class": AUDIT17,
    "T19_DoS_Exploits_Recall_Tradeoff": TRADEOFF17
}

TABLE_FILES17 = []

for name, data in TABLES17.items():

    path = TAB17 / f"{name}_{STAMP17}.csv"

    data.to_csv(
        path,
        index=False,
        mode="x",
        float_format="%.8f"
    )

    TABLE_FILES17.append(path)

# ----------------------------------------------------------------------
# G. SAVE METRICS AND PROVENANCE
# ----------------------------------------------------------------------

METRICS_PATH17 = (
    MET17 / f"label_ambiguity_metrics_{STAMP17}.json"
)

METRICS_DATA17 = {
    "project": "NetGuard",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "evaluation": "development_validation",
    "validation_rows": N17,
    "total_errors_v1": TOTAL_ERRORS17,
    "ambiguous_records": AMBIGUOUS17,
    "unambiguous_records": CLEAR17,
    "errors_ambiguous": AMBIGUOUS_ERRORS17,
    "errors_unambiguous": CLEAR_ERRORS17,
    "conflicting_label_error_floor": CONFLICT_ERROR_FLOOR17,
    "optimistic_accuracy_ceiling": float(
        ACCURACY_CEILING17
    ),
    "v1_accuracy": float(V1_ACCURACY17),
    "audit_counts_source": (
        "Previously executed Cells 19-21; "
        "transcribed and arithmetically verified"
    ),
    "weighted_recalls_source": (
        "Previously executed Cell 22; rounded values"
    ),
    "official_test_accessed": False,
    "models_retrained": 0
}

with METRICS_PATH17.open("x", encoding="utf-8") as f:

    json.dump(
        METRICS_DATA17,
        f,
        indent=2,
        allow_nan=False
    )

def hash17(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024*1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

CREATED17 = (
    FIGURE_FILES17
    + TABLE_FILES17
    + [METRICS_PATH17]
)

MANIFEST_PATH17 = (
    MAN17 / f"label_ambiguity_manifest_{STAMP17}.json"
)

MANIFEST_DATA17 = {
    "project": "NetGuard",
    "figure": STEM17,
    "input_files": [
        {
            "path": str(p.relative_to(PROJECT17)),
            "sha256": hash17(p)
        }
        for p in [V1_PATH17, V2_PATH17]
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT17)),
            "sha256": hash17(p)
        }
        for p in CREATED17
    ],
    "supplementary_transcribed_inputs": {
        "class_ambiguity": TRANSCRIBED17,
        "weighted_class_recalls": WEIGHTED_RECALL17,
        "conflicting_label_error_floor": CONFLICT_ERROR_FLOOR17
    },
    "limitations": [
        "Ambiguity counts were transcribed from earlier notebook output.",
        "Weighted recall values are rounded, not re-evaluated.",
        "The ceiling is an in-sample deterministic bound.",
        "The bound does not establish generalization accuracy.",
        "Development validation was repeatedly examined.",
        "No independent test evaluation was performed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with MANIFEST_PATH17.open("x", encoding="utf-8") as f:

    json.dump(
        MANIFEST_DATA17,
        f,
        indent=2,
        allow_nan=False
    )

# ----------------------------------------------------------------------
# H. FINAL VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "=" * 96)
print("CELL 17 COMPLETE")
print("=" * 96)

print("Attack classes analyzed:", len(CLASSES17))
print("Validation records:", N17)
print("Ambiguous records:", AMBIGUOUS17)
print("Conflict error floor:", CONFLICT_ERROR_FLOOR17)
print(
    "Optimistic deterministic accuracy ceiling:",
    f"{ACCURACY_CEILING17:.4%}"
)

print("\nDoS / EXPLOITS RECALL CHANGE")
print(
    TRADEOFF17.round(5).to_string(index=False)
)

print("\nSAVED RESEARCH OUTPUTS")
print("-" * 96)

for path in CREATED17 + [MANIFEST_PATH17]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print("PASS:", path.relative_to(OUTPUT17))

print("\nAmbiguity arithmetic checks: PASS")
print("Saved confusion consistency: PASS")
print("New models trained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Previous research outputs overwritten: NO")
print("=" * 96)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 18 — CURRENT ATTACK BINARY DETECTION RELIABILITY
#
# F14:
#   A. Full binary precision-recall curve
#   B. ROC curve with low-FPR inset
#   C. Probability reliability + bin support
#   D. Decision-threshold trade-offs
#
# Exact reproduction of Current_Attack_Detection Cells 5–7:
#   - Canonical TRAIN-only UNSW dataset
#   - StratifiedGroupKFold, 5 folds, random_state=42
#   - Best fold by distribution + size error
#   - Saved V1 model and fitted preprocessor
#
# BEFORE SAVING:
#   The complete 10-class confusion matrix MUST exactly
#   match the previously saved V1 validation report.
#
# No model fitting or retraining.
# No official TEST data.
# Existing checkpoints and research outputs are untouched.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib
import xgboost as xgb

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    brier_score_loss,
    confusion_matrix,
    f1_score
)
from matplotlib.lines import Line2D

print("=" * 100)
print("CELL 18 — CURRENT ATTACK BINARY DETECTION RELIABILITY")
print("=" * 100)

# ======================================================================
# A. PROJECT AND SAVED V1 MODEL PATHS
# ======================================================================

PROJECT18 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DETECTION18 = PROJECT18 / "Current_Attack_Detection"

MODEL_DIR18 = (
    DETECTION18
    / "artifacts"
    / "xgboost_20261027_023158_885060"
)

MODEL_FILE18 = MODEL_DIR18 / "current_attack_xgboost.ubj"
PREPROCESSOR_FILE18 = MODEL_DIR18 / "preprocessor.joblib"
SCHEMA_FILE18 = MODEL_DIR18 / "model_schema.json"
V1_METRICS_FILE18 = MODEL_DIR18 / "validation_metrics.json"

OUTPUT18 = PROJECT18 / "Research_Outputs"

FIGURES18 = OUTPUT18 / "01_Figures" / "Current_Detection"
TABLES18 = OUTPUT18 / "02_Tables"
METRICS18 = OUTPUT18 / "03_Metrics"
REPORTS18 = OUTPUT18 / "04_Reports"
MANIFESTS18 = OUTPUT18 / "05_Manifests"

required_files18 = [
    MODEL_FILE18,
    PREPROCESSOR_FILE18,
    SCHEMA_FILE18,
    V1_METRICS_FILE18
]

for path in required_files18:

    if not path.is_file():
        raise FileNotFoundError(path)

for folder in [
    FIGURES18,
    TABLES18,
    METRICS18,
    REPORTS18,
    MANIFESTS18
]:

    if not folder.is_dir():
        raise FileNotFoundError(folder)

with SCHEMA_FILE18.open("r", encoding="utf-8") as f:
    schema18 = json.load(f)

with V1_METRICS_FILE18.open("r", encoding="utf-8") as f:
    saved_metrics18 = json.load(f)

FEATURES18 = schema18["feature_columns"]
CATEGORICAL18 = schema18["categorical_columns"]
NUMERICAL18 = schema18["numerical_columns"]
CLASSES18 = schema18["classes"]

if CLASSES18 != saved_metrics18["class_order"]:
    raise RuntimeError("Saved class orders disagree")

if len(CLASSES18) != 10 or len(FEATURES18) != 42:
    raise RuntimeError("Unexpected saved model schema")

if saved_metrics18.get("official_test_used") is not False:
    raise RuntimeError("Official TEST status not verified")

NORMAL_IDX18 = CLASSES18.index("Normal")

print("\nA. SAVED CLASSIFIER")
print("-" * 100)

print("Model:", MODEL_FILE18.name)
print("Features:", len(FEATURES18))
print("Classes:", len(CLASSES18))
print("Saved development-validation rows:",
      saved_metrics18["validation_rows"])

# ======================================================================
# B. VERIFY AND RESTORE THE CANONICAL TRAIN-ONLY DATASET
# ======================================================================

TRAIN_PATH18 = Path(
    "/content/UNSW_NB15_training-set.csv"
)

TEMP_PATH18 = Path(
    "/content/UNSW_NB15_training-set.csv.partial"
)

TRAIN_URL18 = (
    "https://huggingface.co/datasets/"
    "Mouwiya/UNSW-NB15/resolve/"
    "8ad6b19373f143086a66f52b0d6035a6959d83a9/"
    "UNSW_NB15_training-set.csv"
)

EXPECTED_SHA18 = (
    "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
)

def sha256_18(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

if not TRAIN_PATH18.is_file():

    print("\nRestoring pinned TRAIN-only CSV to temporary Colab storage.")

    try:

        with requests.get(
            TRAIN_URL18,
            stream=True,
            timeout=(30, 180)
        ) as response:

            response.raise_for_status()

            with TEMP_PATH18.open("wb") as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)

        if sha256_18(TEMP_PATH18) != EXPECTED_SHA18:
            raise RuntimeError(
                "TRAIN-only download checksum mismatch"
            )

        TEMP_PATH18.replace(TRAIN_PATH18)

    finally:

        if TEMP_PATH18.exists():
            TEMP_PATH18.unlink()

if sha256_18(TRAIN_PATH18) != EXPECTED_SHA18:
    raise RuntimeError(
        "Canonical TRAIN-only CSV checksum mismatch"
    )

train18 = pd.read_csv(
    TRAIN_PATH18,
    low_memory=False
)

if train18.shape != (175341, 45):
    raise RuntimeError(
        f"Unexpected TRAIN shape: {train18.shape}"
    )

expected_counts18 = {
    "Normal": 56000,
    "Generic": 40000,
    "Exploits": 33393,
    "Fuzzers": 18184,
    "DoS": 12264,
    "Reconnaissance": 10491,
    "Analysis": 2000,
    "Backdoor": 1746,
    "Shellcode": 1133,
    "Worms": 130
}

y_all18 = (
    train18["attack_cat"]
    .astype(str)
    .str.strip()
)

if y_all18.value_counts().to_dict() != expected_counts18:
    raise RuntimeError(
        "Canonical TRAIN class distribution mismatch"
    )

feature_cols18 = [
    c for c in train18.columns
    if c not in {"id", "attack_cat", "label"}
]

if feature_cols18 != FEATURES18:
    raise RuntimeError(
        "TRAIN feature order differs from saved V1 schema"
    )

X_all18 = train18[FEATURES18].copy()

print("\nB. CANONICAL TRAIN DATA")
print("-" * 100)

print("Rows:", len(train18))
print("Features:", len(FEATURES18))
print("Pinned TRAIN checksum: PASS")
print("Class distribution: PASS")
print("Official TEST loaded: NO")

# ======================================================================
# C. RECREATE THE EXACT DEVELOPMENT-VALIDATION SPLIT
# ======================================================================

groups18 = pd.util.hash_pandas_object(
    X_all18,
    index=False
)

splitter18 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

overall_proportions18 = (
    y_all18.value_counts(normalize=True)
)

candidates18 = []

for fold_id, (tr_idx, va_idx) in enumerate(
    splitter18.split(
        X_all18,
        y_all18,
        groups18
    )
):

    proportions = (
        y_all18.iloc[va_idx]
        .value_counts(normalize=True)
        .reindex(
            overall_proportions18.index,
            fill_value=0
        )
    )

    distribution_error = (
        proportions - overall_proportions18
    ).abs().sum()

    size_error = abs(
        len(va_idx) / len(train18) - 0.20
    )

    candidates18.append((
        distribution_error + size_error,
        fold_id,
        tr_idx,
        va_idx
    ))

_, selected_fold18, train_idx18, val_idx18 = min(
    candidates18,
    key=lambda entry: entry[0]
)

if selected_fold18 != 4:
    raise RuntimeError(
        "Reconstructed fold differs from original Cell 6"
    )

if len(train_idx18) != 140174:
    raise RuntimeError("Development TRAIN count mismatch")

if len(val_idx18) != 35167:
    raise RuntimeError("Development VALIDATION count mismatch")

if set(groups18.iloc[train_idx18]).intersection(
    set(groups18.iloc[val_idx18])
):
    raise RuntimeError("Feature-group overlap detected")

X_val18 = X_all18.iloc[val_idx18].copy()

y_labels18 = (
    y_all18.iloc[val_idx18]
    .to_numpy()
)

y_val18 = np.array(
    [CLASSES18.index(label) for label in y_labels18],
    dtype=int
)

expected_validation_support18 = {
    "Analysis": 423,
    "Backdoor": 343,
    "DoS": 2552,
    "Exploits": 6700,
    "Fuzzers": 3747,
    "Generic": 8040,
    "Normal": 11191,
    "Reconnaissance": 1920,
    "Shellcode": 229,
    "Worms": 22
}

observed_support18 = pd.Series(
    y_labels18
).value_counts().to_dict()

if observed_support18 != expected_validation_support18:
    raise RuntimeError(
        "Original development-validation support mismatch"
    )

print("\nC. DEVELOPMENT-VALIDATION RECONSTRUCTION")
print("-" * 100)

print("Selected fold:", selected_fold18)
print("Development TRAIN rows:", len(train_idx18))
print("Development VALIDATION rows:", len(val_idx18))
print("Feature-group overlap: ZERO")
print("Original class support: PASS")

# ======================================================================
# D. LOAD SAVED MODEL AND INFER — NO FITTING
# ======================================================================

classifier18 = xgb.XGBClassifier()
classifier18.load_model(str(MODEL_FILE18))

preprocessor18 = joblib.load(
    PREPROCESSOR_FILE18
)

X_input18 = X_val18[FEATURES18].copy()

for col in CATEGORICAL18:

    X_input18[col] = (
        X_input18[col]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

for col in NUMERICAL18:

    X_input18[col] = pd.to_numeric(
        X_input18[col],
        errors="coerce"
    )

    X_input18[col] = X_input18[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

X_processed18 = preprocessor18.transform(
    X_input18
)

probabilities18 = classifier18.predict_proba(
    X_processed18
)

if probabilities18.shape != (35167, 10):
    raise RuntimeError(
        f"Unexpected prediction shape: {probabilities18.shape}"
    )

if not np.isfinite(probabilities18).all():
    raise RuntimeError("Nonfinite model probabilities")

if not np.allclose(
    probabilities18.sum(axis=1),
    1,
    atol=1e-5
):
    raise RuntimeError(
        "Predicted class probabilities do not sum to 1"
    )

predicted_classes18 = np.argmax(
    probabilities18,
    axis=1
)

recomputed_cm18 = confusion_matrix(
    y_val18,
    predicted_classes18,
    labels=np.arange(10)
)

original_cm18 = np.asarray(
    saved_metrics18["confusion_matrix"],
    dtype=int
)

# THIS IS THE CRITICAL REPRODUCIBILITY CHECK.
if not np.array_equal(
    recomputed_cm18,
    original_cm18
):
    raise RuntimeError(
        "Reconstructed model predictions DO NOT match "
        "the original saved V1 confusion matrix. "
        "No figure will be saved."
    )

print("\nD. SAVED MODEL RELOAD VERIFICATION")
print("-" * 100)

print("Preprocessed shape:", X_processed18.shape)
print("Inference rows:", len(probabilities18))
print("Exact original confusion-matrix match: PASS")
print("Models retrained: ZERO")

# ======================================================================
# E. BINARY ATTACK VS NORMAL EVALUATION
# ======================================================================

binary_truth18 = (
    y_val18 != NORMAL_IDX18
).astype(int)

attack_scores18 = (
    1.0 - probabilities18[:, NORMAL_IDX18]
)

AP18 = float(
    average_precision_score(
        binary_truth18,
        attack_scores18
    )
)

AUC18 = float(
    roc_auc_score(
        binary_truth18,
        attack_scores18
    )
)

BRIER18 = float(
    brier_score_loss(
        binary_truth18,
        attack_scores18
    )
)

PREVALENCE18 = float(
    binary_truth18.mean()
)

precision18, recall18, _ = precision_recall_curve(
    binary_truth18,
    attack_scores18
)

fpr18, tpr18, roc_thresholds18 = roc_curve(
    binary_truth18,
    attack_scores18
)

threshold_rows18 = []

for threshold in [0.50, 0.70, 0.80, 0.90, 0.95]:

    predicted = (
        attack_scores18 >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        binary_truth18,
        predicted,
        labels=[0, 1]
    ).ravel()

    threshold_rows18.append({
        "Threshold": threshold,
        "Precision": (
            tp / (tp + fp)
            if tp + fp else 0
        ),
        "Recall": (
            tp / (tp + fn)
            if tp + fn else 0
        ),
        "FPR": (
            fp / (fp + tn)
            if fp + tn else 0
        ),
        "F1": float(
            f1_score(binary_truth18, predicted)
        ),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn)
    })

THRESHOLD_TABLE18 = pd.DataFrame(
    threshold_rows18
)

# Independent binary confusion from original Cell 7.
original_binary_cm18 = np.array([
    [10218, 973],
    [621, 23355]
])

observed_binary_cm18 = np.array([
    [
        THRESHOLD_TABLE18.iloc[0]["TN"],
        THRESHOLD_TABLE18.iloc[0]["FP"]
    ],
    [
        THRESHOLD_TABLE18.iloc[0]["FN"],
        THRESHOLD_TABLE18.iloc[0]["TP"]
    ]
], dtype=int)

if not np.array_equal(
    original_binary_cm18,
    observed_binary_cm18
):
    raise RuntimeError(
        "Original binary validation confusion mismatch"
    )

# Check original Cell 10's independently printed results.
if not (
    abs(AP18 - 0.9967) < 0.00015
    and abs(AUC18 - 0.9929) < 0.00015
    and abs(BRIER18 - 0.0309) < 0.00015
):
    raise RuntimeError(
        "Binary evaluation disagrees with original "
        "Cell 10 results. No outputs saved."
    )

print("\nE. VERIFIED BINARY PERFORMANCE")
print("-" * 100)

print(f"Attack prevalence: {PREVALENCE18:.5f}")
print(f"Average precision: {AP18:.6f}")
print(f"ROC-AUC:           {AUC18:.6f}")
print(f"Brier score:       {BRIER18:.6f}")
print("Original binary confusion: PASS")

print("\nTHRESHOLD DIAGNOSTICS")
print(
    THRESHOLD_TABLE18.round(5).to_string(index=False)
)

# ======================================================================
# F. CALIBRATION: TEN FIXED SCORE BINS
# ======================================================================

calibration_rows18 = []
ece18 = 0.0

edges18 = np.linspace(0, 1, 11)

for i in range(10):

    lower = float(edges18[i])
    upper = float(edges18[i+1])

    if i == 9:
        mask = (
            (attack_scores18 >= lower)
            & (attack_scores18 <= upper)
        )
    else:
        mask = (
            (attack_scores18 >= lower)
            & (attack_scores18 < upper)
        )

    count = int(mask.sum())

    if count:

        mean_pred = float(
            attack_scores18[mask].mean()
        )

        observed_rate = float(
            binary_truth18[mask].mean()
        )

        ece18 += (
            count / len(binary_truth18)
        ) * abs(mean_pred - observed_rate)

    else:
        mean_pred = np.nan
        observed_rate = np.nan

    calibration_rows18.append({
        "Bin": i,
        "Lower": lower,
        "Upper": upper,
        "Count": count,
        "Mean_Predicted": mean_pred,
        "Observed_Attack_Fraction": observed_rate
    })

CALIBRATION18 = pd.DataFrame(
    calibration_rows18
)

assert CALIBRATION18["Count"].sum() == 35167

if abs(ece18 - 0.0055) > 0.00015:
    raise RuntimeError(
        "ECE differs from the original Cell 10 audit"
    )

print("\nF. PROBABILITY CALIBRATION")
print("-" * 100)

print("Ten-bin ECE:", round(ece18, 6))
print("Calibration-bin rows:", len(CALIBRATION18))
print("Sum of bin support: PASS")

# ======================================================================
# G. SCIENTIFIC FIGURE DESIGN
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig18 = plt.figure(
    figsize=(16.5, 12.5),
    facecolor="white"
)

outer18 = fig18.add_gridspec(
    2, 2,
    left=0.09,
    right=0.96,
    top=0.865,
    bottom=0.13,
    wspace=0.30,
    hspace=0.40
)

ax_pr18 = fig18.add_subplot(outer18[0, 0])
ax_roc18 = fig18.add_subplot(outer18[0, 1])

# Nested calibration plot + matching-width histogram.
cal_grid18 = outer18[1, 0].subgridspec(
    2, 1,
    height_ratios=[3.2, 1],
    hspace=0.12
)

ax_cal18 = fig18.add_subplot(cal_grid18[0])
ax_hist18 = fig18.add_subplot(
    cal_grid18[1],
    sharex=ax_cal18
)

ax_threshold18 = fig18.add_subplot(
    outer18[1, 1]
)

fig18.suptitle(
    "Binary current-attack detection: discrimination and reliability",
    x=0.09,
    y=0.985,
    ha="left",
    fontsize=19,
    fontweight="bold"
)

fig18.text(
    0.09,
    0.95,
    "Saved XGBoost V1 | "
    "UNSW-NB15 development validation | "
    "35,167 records | Attack vs Normal",
    fontsize=10.7,
    color="#586576"
)

def style18(ax):

    ax.set_facecolor("#FAFBFC")

    ax.grid(
        linestyle=":",
        linewidth=0.7,
        color="#C6D0DB",
        alpha=0.55
    )

    ax.set_axisbelow(True)
    ax.tick_params(length=3, pad=5)

# ======================================================================
# PANEL A — PRECISION–RECALL
# ======================================================================

style18(ax_pr18)

ax_pr18.plot(
    recall18,
    precision18,
    color="#227A91",
    linewidth=2.3,
    label=f"XGBoost V1 (AP = {AP18:.4f})"
)

ax_pr18.axhline(
    PREVALENCE18,
    linestyle="--",
    linewidth=1.4,
    color="#A97143",
    label=f"Attack prevalence = {PREVALENCE18:.3f}"
)

ax_pr18.set_xlim(0, 1)
ax_pr18.set_ylim(0, 1.02)

ax_pr18.set_xlabel("Attack recall")
ax_pr18.set_ylabel("Attack precision")

ax_pr18.set_title(
    "A   Precision–recall discrimination",
    loc="left",
    fontweight="bold",
    pad=14
)

ax_pr18.legend(
    loc="lower left",
    frameon=False,
    fontsize=9
)

# ======================================================================
# PANEL B — ROC WITH LOW-FPR INSET
# ======================================================================

style18(ax_roc18)

ax_roc18.plot(
    fpr18,
    tpr18,
    color="#79549D",
    linewidth=2.3,
    label=f"ROC-AUC = {AUC18:.4f}"
)

ax_roc18.plot(
    [0, 1],
    [0, 1],
    color="#83909E",
    linewidth=1.2,
    linestyle="--",
    label="Random ranking"
)

ax_roc18.set_xlim(0, 1)
ax_roc18.set_ylim(0, 1.02)

ax_roc18.set_xlabel("False-positive rate")
ax_roc18.set_ylabel("True-positive rate")

ax_roc18.set_title(
    "B   Receiver operating characteristic",
    loc="left",
    fontweight="bold",
    pad=14
)

ax_roc18.legend(
    loc="lower right",
    frameon=False,
    fontsize=9
)

# Enlarged low-false-positive region.
inset18 = ax_roc18.inset_axes([
    0.38, 0.16, 0.49, 0.48
])

inset18.plot(
    fpr18,
    tpr18,
    color="#79549D",
    linewidth=1.8
)

inset18.set_xlim(0, 0.12)
inset18.set_ylim(0.75, 1.005)

inset18.grid(
    linestyle=":",
    alpha=0.35
)

inset18.set_title(
    "Low-FPR region",
    fontsize=8,
    pad=5
)

inset18.tick_params(
    labelsize=7,
    length=2
)

# ======================================================================
# PANEL C — PROBABILITY RELIABILITY + BIN SUPPORT
# ======================================================================

style18(ax_cal18)

ax_cal18.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.3,
    color="#8894A1",
    label="Perfect calibration"
)

valid_bins18 = CALIBRATION18[
    CALIBRATION18["Count"] > 0
]

ax_cal18.plot(
    valid_bins18["Mean_Predicted"],
    valid_bins18["Observed_Attack_Fraction"],
    color="#28877C",
    marker="o",
    linewidth=2,
    markersize=5.5,
    label="Observed attack fraction"
)

ax_cal18.set_xlim(0, 1)
ax_cal18.set_ylim(0, 1.02)

ax_cal18.set_ylabel("Observed attack fraction")
ax_cal18.set_title(
    "C   Reliability of attack probabilities",
    loc="left",
    fontweight="bold",
    pad=14
)

ax_cal18.text(
    0.03,
    0.97,
    f"ECE = {ece18:.4f}\n"
    f"Brier = {BRIER18:.4f}",
    transform=ax_cal18.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    bbox={
        "facecolor": "white",
        "edgecolor": "#D5DDE6",
        "boxstyle": "round,pad=0.4"
    }
)

ax_cal18.tick_params(
    axis="x",
    labelbottom=False
)

centers18 = (
    CALIBRATION18["Lower"].to_numpy()
    + CALIBRATION18["Upper"].to_numpy()
) / 2

ax_hist18.bar(
    centers18,
    CALIBRATION18["Count"],
    width=0.085,
    color="#78A8A3",
    edgecolor="white"
)

ax_hist18.set_yscale("log")
ax_hist18.set_ylim(1, 50000)

ax_hist18.set_yticks([1, 100, 10000])
ax_hist18.set_yticklabels(["1", "100", "10k"])

ax_hist18.set_xticks(
    np.linspace(0, 1, 6)
)

ax_hist18.set_xlabel(
    "Predicted attack probability"
)

ax_hist18.set_ylabel(
    "Bin count\n(log)",
    fontsize=8.5
)

ax_hist18.grid(
    axis="y",
    linestyle=":",
    alpha=0.3
)

ax_hist18.set_facecolor("#FAFBFC")

# ======================================================================
# PANEL D — OPERATING THRESHOLD TRADE-OFFS
# ======================================================================

style18(ax_threshold18)

t18 = THRESHOLD_TABLE18["Threshold"].to_numpy()

ax_threshold18.plot(
    t18,
    THRESHOLD_TABLE18["Precision"],
    color="#277B93",
    linewidth=2,
    marker="o",
    label="Precision"
)

ax_threshold18.plot(
    t18,
    THRESHOLD_TABLE18["Recall"],
    color="#B47A3F",
    linewidth=2,
    marker="s",
    label="Recall"
)

ax_threshold18.plot(
    t18,
    THRESHOLD_TABLE18["F1"],
    color="#448D70",
    linewidth=2,
    marker="D",
    label="F1"
)

ax_threshold18.set_xlim(0.48, 0.97)
ax_threshold18.set_ylim(0.75, 1.02)

ax_threshold18.set_xticks(t18)

ax_threshold18.set_xlabel(
    "Attack decision threshold"
)

ax_threshold18.set_ylabel(
    "Precision / recall / F1"
)

ax_threshold18.set_title(
    "D   Threshold selection and false alarms",
    loc="left",
    fontweight="bold",
    pad=14
)

ax_fpr18 = ax_threshold18.twinx()

ax_fpr18.plot(
    t18,
    THRESHOLD_TABLE18["FPR"],
    color="#A55378",
    linestyle="--",
    linewidth=2,
    marker="^",
    label="False-positive rate"
)

ax_fpr18.set_ylabel(
    "False-positive rate",
    color="#A55378"
)

ax_fpr18.tick_params(
    axis="y",
    colors="#A55378"
)

ax_fpr18.set_ylim(0, 0.12)

handles1, labels1 = ax_threshold18.get_legend_handles_labels()
handles2, labels2 = ax_fpr18.get_legend_handles_labels()

ax_threshold18.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="lower left",
    fontsize=8.4,
    frameon=False
)

# ======================================================================
# H. SCIENTIFIC CAPTIONS
# ======================================================================

fig18.text(
    0.09,
    0.064,
    "UNSW-NB15 canonical TRAIN partition, recreated grouped "
    "development-validation split. Binary score = 1 − P(Normal). "
    "ROC and PR curves are recomputed from saved-model inference.",
    fontsize=9,
    color="#596575"
)

fig18.text(
    0.09,
    0.043,
    "Calibration uses ten fixed-width probability bins. "
    "The validation fold was repeatedly examined during "
    "development; these results are not independent test "
    "performance or live-network estimates.",
    fontsize=9,
    color="#596575"
)

# ======================================================================
# I. EXPORT ONLY AFTER ALL CHECKS PASSED
# ======================================================================

STAMP18 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM18 = (
    f"F14_Binary_Attack_Detection_Reliability_{STAMP18}"
)

FIGURE_FILES18 = []

for ext in ["pdf", "svg", "png"]:

    path = FIGURES18 / f"{STEM18}.{ext}"

    if path.exists():
        raise FileExistsError(path)

    fig18.savefig(
        path,
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

    FIGURE_FILES18.append(path)

plt.show()
plt.close(fig18)

# ======================================================================
# J. EXPORT VALIDATED RESEARCH TABLES
# ======================================================================

PR_TABLE18 = pd.DataFrame({
    "Recall": recall18,
    "Precision": precision18
})

ROC_TABLE18 = pd.DataFrame({
    "False_Positive_Rate": fpr18,
    "True_Positive_Rate": tpr18,
    "Threshold": roc_thresholds18
})

TABLE_DATA18 = {
    "T20_Binary_Threshold_Metrics":
        THRESHOLD_TABLE18,

    "T21_Binary_Calibration_Bins":
        CALIBRATION18,

    "T22_Binary_Precision_Recall_Curve":
        PR_TABLE18,

    "T23_Binary_ROC_Curve":
        ROC_TABLE18
}

TABLE_FILES18 = []

for prefix, table in TABLE_DATA18.items():

    path = TABLES18 / f"{prefix}_{STAMP18}.csv"

    table.to_csv(
        path,
        index=False,
        mode="x",
        float_format="%.10g"
    )

    TABLE_FILES18.append(path)

# ======================================================================
# K. SAVE METRICS
# ======================================================================

METRICS_PATH18 = (
    METRICS18
    / f"binary_attack_detection_metrics_{STAMP18}.json"
)

metrics_content18 = {
    "project": "NetGuard",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "split": "recreated_development_validation",
    "model": "XGBoost V1",
    "validation_rows": len(binary_truth18),
    "positive_attack_rows": int(binary_truth18.sum()),
    "normal_rows": int(
        len(binary_truth18) - binary_truth18.sum()
    ),
    "attack_prevalence": PREVALENCE18,
    "average_precision": AP18,
    "roc_auc": AUC18,
    "brier_score": BRIER18,
    "ece_10_bins": float(ece18),
    "original_confusion_matrix_match": True,
    "original_binary_confusion_match": True,
    "thresholds": THRESHOLD_TABLE18.to_dict("records"),
    "models_retrained": 0,
    "official_test_accessed": False
}

with METRICS_PATH18.open("x", encoding="utf-8") as f:
    json.dump(
        metrics_content18,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# L. SHORT SCIENTIFIC REPORT
# ======================================================================

REPORT_PATH18 = (
    REPORTS18
    / f"binary_attack_detection_report_{STAMP18}.txt"
)

report18 = [
    "NETGUARD — BINARY CURRENT ATTACK DETECTION",
    "=" * 80,
    "",
    "Dataset: UNSW-NB15 canonical TRAIN only",
    "Split: Reconstructed grouped development validation",
    f"Validation records: {len(binary_truth18):,}",
    f"Attack prevalence: {PREVALENCE18:.6f}",
    f"Average precision: {AP18:.6f}",
    f"ROC-AUC: {AUC18:.6f}",
    f"Brier score: {BRIER18:.6f}",
    f"Ten-bin ECE: {ece18:.6f}",
    "",
    "OPERATING THRESHOLDS",
    "-" * 80,
    THRESHOLD_TABLE18.round(6).to_string(index=False),
    "",
    "LIMITATIONS",
    "-" * 80,
    "The development-validation fold was examined repeatedly.",
    "This is not an independent final TEST-set estimate.",
    "High average precision should be interpreted with attack prevalence.",
    "Binary detection quality does not imply ten-class quality.",
    "Probabilities and operational thresholds need external validation.",
    "",
    "Saved model reused: YES",
    "Models retrained: 0",
    "Official TEST accessed: NO"
]

with REPORT_PATH18.open(
    "x", encoding="utf-8"
) as f:

    f.write("\n".join(report18))

# ======================================================================
# M. SHA256 PROVENANCE MANIFEST
# ======================================================================

GENERATED18 = (
    FIGURE_FILES18
    + TABLE_FILES18
    + [METRICS_PATH18, REPORT_PATH18]
)

MANIFEST_PATH18 = (
    MANIFESTS18
    / f"binary_detection_manifest_{STAMP18}.json"
)

manifest18 = {
    "project": "NetGuard",
    "figure": STEM18,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": "UNSW-NB15 canonical TRAIN",
    "training_csv_sha256": sha256_18(TRAIN_PATH18),
    "split_method": (
        "Original Cell 6 StratifiedGroupKFold "
        "with random_state=42 and score-based fold selection"
    ),
    "selected_fold": int(selected_fold18),
    "exact_confusion_matrix_match": True,
    "inputs": [
        {
            "path": str(path.relative_to(PROJECT18)),
            "sha256": sha256_18(path)
        }
        for path in required_files18
    ],
    "outputs": [
        {
            "path": str(path.relative_to(PROJECT18)),
            "sha256": sha256_18(path)
        }
        for path in GENERATED18
    ],
    "limitations": [
        "Repeated development-validation usage.",
        "Not independent TEST performance.",
        "Binary attack detection is not ten-class accuracy.",
        "Class probabilities require external validation.",
        "No live network evaluation.",
        "No model retraining."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with MANIFEST_PATH18.open("x", encoding="utf-8") as f:
    json.dump(
        manifest18,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# N. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 100)
print("CELL 18 COMPLETE — FINAL ML FIGURE")
print("=" * 100)

print("Development-validation records:", len(binary_truth18))
print("Saved classifier reloaded: YES")
print("Exact multiclass confusion match: PASS")
print("Exact binary confusion match: PASS")
print("Figure panels: 4")

print("\nPERFORMANCE")
print("-" * 100)

print(f"Attack AP:     {AP18:.6f}")
print(f"ROC-AUC:       {AUC18:.6f}")
print(f"Brier score:   {BRIER18:.6f}")
print(f"Ten-bin ECE:   {ece18:.6f}")

print("\nSAVED RESEARCH OUTPUTS")
print("-" * 100)

for path in GENERATED18 + [MANIFEST_PATH18]:

    assert path.is_file()
    assert path.stat().st_size > 0

    print("PASS:", path.relative_to(OUTPUT18))

print("\nModels retrained: 0")
print("Official TEST accessed: NO")
print("Previous checkpoints modified: NO")
print("Previous research outputs overwritten: NO")

print("=" * 100)


In [ ]:

# =====================================================================
# NETGUARD — RESEARCH OUTPUTS
# CELL 19 — FINAL ML FIGURE INVENTORY & PAPER SHORTLIST
#
# Audits F01–F14 across all research figure folders.
# Prefers corrected F04 and revised F06.
# Verifies PDF / SVG / PNG exports.
# Checks available output-manifest SHA256 hashes.
# Creates:
#   02_Tables/T24_Research_Figure_Catalog_*.csv
#   04_Reports/Research_Outputs_Review_*.txt
#   05_Manifests/research_review_*.json
#
# Does NOT delete or modify existing outputs.
# Does NOT load TEST data or train models.
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import json
import re

import pandas as pd

ROOT19 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)
OUT19 = ROOT19 / "Research_Outputs"

FIG_ROOT19 = OUT19 / "01_Figures"
TABLE_ROOT19 = OUT19 / "02_Tables"
MET_ROOT19 = OUT19 / "03_Metrics"
REPORT_ROOT19 = OUT19 / "04_Reports"
MAN_ROOT19 = OUT19 / "05_Manifests"

for p in [
    FIG_ROOT19, TABLE_ROOT19, MET_ROOT19,
    REPORT_ROOT19, MAN_ROOT19
]:
    if not p.is_dir():
        raise FileNotFoundError(p)

print("=" * 94)
print("CELL 19 — NETGUARD RESEARCH OUTPUTS FINAL AUDIT")
print("=" * 94)

# -----------------------------------------------------------
# A. EXPECTED FIGURE TYPES AND INITIAL PAPER RECOMMENDATIONS
# -----------------------------------------------------------

EXPECTED19 = {
    "F01": ("Seven-model temporal atlas", "Supplementary"),
    "F02": ("Cross-family forecasting atlas", "Supplementary"),
    "F03": ("Paired statistical comparison", "Main-paper candidate"),
    "F04": ("Source-specific family confusion", "Main-paper candidate"),
    "F05": ("Source-specific family recall", "Supplementary"),
    "F06": ("Forecast probability calibration", "Supplementary"),
    "F07": ("Operational thresholds and false alerts", "Main-paper candidate"),
    "F08": ("Traffic versus forecasting trajectories", "Supplementary"),
    "F09": ("Seven-model precision–recall benchmark", "Main-paper candidate"),
    "F10": ("Conditional ETA accuracy", "Main-paper candidate"),
    "F11": ("Graph architecture comparison", "Supplementary"),
    "F12": ("Ten-class current attack performance", "Main-paper candidate"),
    "F13": ("Dataset ambiguity and DoS trade-offs", "Needs layout revision"),
    "F14": ("Binary current-attack reliability", "Main-paper candidate")
}

# -----------------------------------------------------------
# B. FIND ALL ACTUAL FIGURE EXPORTS
# -----------------------------------------------------------

pattern19 = re.compile(
    r"^(F\d{2})_(.+)\.(pdf|svg|png)$",
    flags=re.IGNORECASE
)

versions19 = defaultdict(
    lambda: {
        "formats": {},
        "folder": None
    }
)

for path in FIG_ROOT19.rglob("*"):

    if not path.is_file():
        continue

    match = pattern19.match(path.name)

    if not match:
        continue

    figure_id = match.group(1).upper()
    extension = match.group(3).lower()

    key = (figure_id, path.stem)

    versions19[key]["formats"][extension] = path
    versions19[key]["folder"] = str(
        path.parent.relative_to(FIG_ROOT19)
    )

def preferred_version19(fig_id, stem):

    # Prefer scientifically corrected/revised versions.
    if fig_id == "F04":
        return 10 if "CORRECTED" in stem.upper() else 0

    if fig_id == "F06":
        return 10 if "V2" in stem.upper() else 0

    return 0

catalog_rows19 = []
selected_paths19 = []

print("\nA. FIGURE INVENTORY")
print("-" * 94)

for figure_id, (description, recommendation) in EXPECTED19.items():

    candidates = [
        (stem, data)
        for (fid, stem), data in versions19.items()
        if fid == figure_id
    ]

    if not candidates:

        catalog_rows19.append({
            "Figure_ID": figure_id,
            "Description": description,
            "Recommendation": recommendation,
            "Preferred_Version": "",
            "Available_Versions": 0,
            "PDF": False,
            "SVG": False,
            "PNG": False,
            "Status": "MISSING",
            "Needs_Review": True,
            "Figure_Folder": ""
        })

        print(f"MISSING | {figure_id} | {description}")
        continue

    candidates.sort(
        key=lambda item: (
            preferred_version19(figure_id, item[0]),
            item[0]
        ),
        reverse=True
    )

    stem, data = candidates[0]

    availability = {
        ext: (
            ext in data["formats"]
            and data["formats"][ext].is_file()
            and data["formats"][ext].stat().st_size > 0
        )
        for ext in ["pdf", "svg", "png"]
    }

    for path in data["formats"].values():
        selected_paths19.append(path)

    complete = all(availability.values())

    needs_review = (
        recommendation == "Needs layout revision"
        or not complete
    )

    status = (
        "COMPLETE"
        if complete else "INCOMPLETE"
    )

    catalog_rows19.append({
        "Figure_ID": figure_id,
        "Description": description,
        "Recommendation": recommendation,
        "Preferred_Version": stem,
        "Available_Versions": len(candidates),
        "PDF": availability["pdf"],
        "SVG": availability["svg"],
        "PNG": availability["png"],
        "Status": status,
        "Needs_Review": needs_review,
        "Figure_Folder": data["folder"]
    })

    print(
        f"{status:<11} | {figure_id} | "
        f"versions={len(candidates):<2} | "
        f"{stem}"
    )

CATALOG19 = pd.DataFrame(catalog_rows19)

# -----------------------------------------------------------
# C. IDENTIFY ADDITIONAL / HISTORICAL VERSIONS
# -----------------------------------------------------------

historical19 = []

selected_stems19 = set(
    CATALOG19["Preferred_Version"]
)

for (figure_id, stem), data in versions19.items():

    if stem not in selected_stems19:

        historical19.append({
            "figure_id": figure_id,
            "stem": stem,
            "formats": sorted(data["formats"]),
            "folder": data["folder"]
        })

print("\nB. HISTORICAL FIGURE VERSIONS")
print("-" * 94)

print("Nonpreferred versions:", len(historical19))

for record in sorted(
    historical19,
    key=lambda r: (r["figure_id"], r["stem"])
):

    print(
        f"{record['figure_id']} | "
        f"{record['stem']} | "
        f"{','.join(record['formats'])}"
    )

print(
    "\nNothing in this list is deleted automatically."
)

# -----------------------------------------------------------
# D. CHECK SOURCE MANIFEST OUTPUT INTEGRITY
#
# Only check files described as generated outputs.
# Never hash model checkpoints or raw datasets here.
# -----------------------------------------------------------

def sha256_19(path):

    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()

hash_cache19 = {}

def resolve_output19(raw):

    p = Path(str(raw))

    if not p.is_absolute():

        if p.parts and p.parts[0] == "Research_Outputs":
            p = ROOT19 / p
        else:
            p = OUT19 / p

    p = p.resolve()

    # Never follow output-manifest entries outside
    # the research-output directory.
    if not p.is_relative_to(OUT19.resolve()):
        return None

    return p

manifest_checks19 = []

print("\nC. OUTPUT MANIFEST INTEGRITY")
print("-" * 94)

for manifest_path in sorted(MAN_ROOT19.glob("*.json")):

    try:

        with manifest_path.open(
            "r", encoding="utf-8"
        ) as f:
            payload = json.load(f)

    except Exception as exc:

        manifest_checks19.append({
            "manifest": manifest_path.name,
            "status": "INVALID_JSON",
            "detail": str(exc)
        })

        continue

    # Previous research cells used either generated_files
    # or outputs to record output artifact hashes.
    entries = (
        payload.get("generated_files")
        or payload.get("outputs")
        or []
    )

    if not isinstance(entries, list):
        continue

    for item in entries:

        if not isinstance(item, dict):
            continue

        raw_path = item.get("path")
        expected_hash = item.get("sha256")

        if not raw_path:
            continue

        path = resolve_output19(raw_path)

        if path is None:

            status = "OUTSIDE_RESEARCH_OUTPUTS"

        elif not path.is_file():

            status = "MISSING_HISTORICAL_REFERENCE"

        elif not expected_hash:

            status = "FILE_EXISTS_NO_HASH"

        else:

            if path not in hash_cache19:
                hash_cache19[path] = sha256_19(path)

            status = (
                "HASH_PASS"
                if hash_cache19[path] == expected_hash
                else "HASH_MISMATCH"
            )

        manifest_checks19.append({
            "manifest": manifest_path.name,
            "status": status,
            "detail": str(raw_path)
        })

MANIFEST_CHECK19 = pd.DataFrame(
    manifest_checks19,
    columns=["manifest", "status", "detail"]
)

if len(MANIFEST_CHECK19):

    print(
        MANIFEST_CHECK19["status"]
        .value_counts()
        .to_string()
    )

else:

    print("No output hash entries found.")

# Earlier cleanup may have removed obsolete versions.
# Missing references in historical manifests are reported,
# not silently recreated or treated as model corruption.

# -----------------------------------------------------------
# E. RESEARCH OUTPUT COUNTS
# -----------------------------------------------------------

def count_files19(folder):

    return sum(
        1 for p in folder.rglob("*")
        if p.is_file()
    )

counts19 = {
    "figure_files": count_files19(FIG_ROOT19),
    "table_files": count_files19(TABLE_ROOT19),
    "metric_files": count_files19(MET_ROOT19),
    "report_files": count_files19(REPORT_ROOT19),
    "manifest_files": count_files19(MAN_ROOT19),
    "planned_figure_types": len(EXPECTED19),
    "complete_preferred_figures": int(
        CATALOG19["Status"].eq("COMPLETE").sum()
    ),
    "historical_figure_versions": len(historical19),
    "main_paper_candidates": int(
        CATALOG19["Recommendation"]
        .eq("Main-paper candidate")
        .sum()
    )
}

print("\nD. RESEARCH OUTPUT COUNTS")
print("-" * 94)

for key, value in counts19.items():
    print(f"{key:<34}: {value}")

# -----------------------------------------------------------
# F. WRITE FINAL INVENTORY + REVIEW REPORT
# -----------------------------------------------------------

STAMP19 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

TABLE_PATH19 = (
    TABLE_ROOT19
    / f"T24_Research_Figure_Catalog_{STAMP19}.csv"
)

REPORT_PATH19 = (
    REPORT_ROOT19
    / f"Research_Outputs_Review_{STAMP19}.txt"
)

MANIFEST_PATH19 = (
    MAN_ROOT19
    / f"research_review_{STAMP19}.json"
)

CATALOG19.to_csv(
    TABLE_PATH19,
    index=False,
    mode="x"
)

main_candidates19 = CATALOG19.loc[
    CATALOG19["Recommendation"] == "Main-paper candidate",
    ["Figure_ID", "Description", "Status"]
]

review_lines19 = [
    "NETGUARD — RESEARCH OUTPUT REVIEW",
    "=" * 90,
    "",
    "RESEARCH OUTPUT COUNTS",
    "-" * 90,
    *[
        f"{key}: {value}"
        for key, value in counts19.items()
    ],
    "",
    "FIGURE CATALOG",
    "-" * 90,
    CATALOG19.to_string(index=False),
    "",
    "PROVISIONAL MAIN-PAPER SHORTLIST",
    "-" * 90,
    main_candidates19.to_string(index=False),
    "",
    "MANIFEST INTEGRITY",
    "-" * 90,
    (
        MANIFEST_CHECK19["status"]
        .value_counts()
        .to_string()
        if len(MANIFEST_CHECK19)
        else "No hash entries checked"
    ),
    "",
    "KNOWN REVIEW ACTIONS",
    "-" * 90,
    "1. Polish F13 panel C labels and panel D legend.",
    "2. Select only the strongest figures for the main paper.",
    "3. Keep F01/F02 and other illustrative atlases as supplements.",
    "4. Keep historical figure versions until cleanup is approved.",
    "5. Consolidate final benchmark tables without altering originals.",
    "6. Preserve development-validation limitations in captions.",
    "",
    "No official TEST data accessed.",
    "No training performed.",
    "No existing research file deleted or overwritten."
]

with REPORT_PATH19.open(
    "x", encoding="utf-8"
) as f:

    f.write("\n".join(review_lines19))

def sanitize19(obj):

    if isinstance(obj, dict):
        return {
            str(k): sanitize19(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [sanitize19(v) for v in obj]

    if isinstance(obj, np.generic):
        return obj.item()

    return obj

# pandas/numpy imports are already available from above.
import numpy as np

summary19 = {
    "project": "NetGuard",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "counts": counts19,
    "figure_catalog": CATALOG19.to_dict("records"),
    "historical_versions": historical19,
    "manifest_checks": MANIFEST_CHECK19.to_dict("records"),
    "generated_outputs": [
        {
            "path": str(p.relative_to(ROOT19)),
            "sha256": sha256_19(p)
        }
        for p in [TABLE_PATH19, REPORT_PATH19]
    ],
    "deletions_performed": 0,
    "models_retrained": 0,
    "official_test_accessed": False
}

with MANIFEST_PATH19.open(
    "x", encoding="utf-8"
) as f:

    json.dump(
        sanitize19(summary19),
        f,
        indent=2,
        allow_nan=False
    )

# -----------------------------------------------------------
# G. FINAL SUMMARY
# -----------------------------------------------------------

print("\n" + "=" * 94)
print("CELL 19 COMPLETE")
print("=" * 94)

print("\nFIGURE REVIEW")

print(
    CATALOG19[
        [
            "Figure_ID",
            "Status",
            "Recommendation",
            "Available_Versions"
        ]
    ].to_string(index=False)
)

print("\nSAVED AUDIT OUTPUTS")

for p in [
    TABLE_PATH19,
    REPORT_PATH19,
    MANIFEST_PATH19
]:

    assert p.is_file()
    assert p.stat().st_size > 0

    print("PASS:", p.relative_to(OUT19))

print("\nOriginal figures deleted: 0")
print("Original figures overwritten: 0")
print("Models trained: 0")
print("Official TEST accessed: NO")
print("=" * 94)
